Main Code.py

In [ ]:
from enum import Enum
import pickle
import os
from datetime import datetime, date


class UserRole(Enum):
    """
    UserRole defines the three actors in the system.

    Purpose:
    - STUDENT users book student-access facilities.
    - STAFF users book staff-access facilities.
    - ADMIN users manage users, bookings, and facility availability.

    Logical reason:
    Using Enum prevents spelling mistakes like "student", "Student", or "STUDNT".
    The system only accepts the exact role values defined here.
    """
    STUDENT = "STUDENT"
    STAFF = "STAFF"
    ADMIN = "ADMIN"


class AccessType(Enum):
    """
    AccessType defines the user's permission level.

    Purpose:
    - STANDARD users have limited booking access.
    - PREMIUM users have more booking access.

    Logical reason:
    The booking rules depend on whether the user is STANDARD or PREMIUM,
    so this enum keeps the permission levels consistent across the whole system.
    """
    STANDARD = "STANDARD"
    PREMIUM = "PREMIUM"


class UserStatus(Enum):
    """
    UserStatus tracks whether a user is allowed to use the system.

    Purpose:
    - ACTIVE users can book normally.
    - FROZEN users are blocked from booking.

    Logical reason:
    This supports the punishment rule where users can be frozen after violations.
    """
    ACTIVE = "ACTIVE"
    FROZEN = "FROZEN"


class BookingStatus(Enum):
    """
    BookingStatus tracks the current state of a booking.

    Purpose:
    - CONFIRMED means the booking is active.
    - CANCELLED means the user cancelled it.
    - NO_SHOW means the user missed it.

    Logical reason:
    Booking status is needed for reports, penalties, and freeing booked slots.
    """
    PENDING = "PENDING"
    CONFIRMED = "CONFIRMED"
    CANCELLED = "CANCELLED"
    NO_SHOW = "NO_SHOW"


class UsageType(Enum):
    """
    UsageType explains how a facility is being used.

    Purpose:
    - INTERNAL use can be free for EventHall.
    - EXTERNAL use may require payment.

    Logical reason:
    EventHall cost depends on usage type, so this enum helps calculate fees correctly.
    """
    INTERNAL = "INTERNAL"
    EXTERNAL = "EXTERNAL"


class PaymentStatus(Enum):
    """
    PaymentStatus tracks the payment result.

    Purpose:
    - NOT_REQUIRED is used when the cost is 0.
    - PAID is used after successful payment.
    - REFUNDED is used when money is returned.

    Logical reason:
    The system needs payment status for receipts and proof of payment.
    """
    NOT_REQUIRED = "NOT_REQUIRED"
    PENDING = "PENDING"
    PAID = "PAID"
    FAILED = "FAILED"
    REFUNDED = "REFUNDED"


class PaymentPurpose(Enum):
    """
    PaymentPurpose explains why the payment exists.

    Purpose:
    - BOOKING_FEE is for facility bookings.
    - ACCESS_UPGRADE is for changing STANDARD to PREMIUM.

    Logical reason:
    Separating payment purposes makes reports and receipts clearer.
    """
    BOOKING_FEE = "BOOKING_FEE"
    ACCESS_UPGRADE = "ACCESS_UPGRADE"


class User:
    """
    User is the parent class for Student, Staff, and Administrator.

    Purpose:
    - Stores shared user information such as ID, name, email, password, role, access type, and status.
    - Provides shared actions such as login, profile update, support contact, and access requests.

    Logical reason:
    Student, Staff, and Administrator share common data. Using inheritance avoids repeating the same attributes and methods in every child class.
    """

    def __init__(self, user_id, university_id, name, email, password, role):
        # Validation protects the system from creating incomplete or invalid user objects.
        if not user_id:
            raise ValueError("User ID cannot be empty.")
        if not university_id:
            raise ValueError("University/work ID cannot be empty.")
        if not name:
            raise ValueError("Name cannot be empty.")
        if "@" not in email:
            raise ValueError("Invalid email format.")
        if len(password) < 4:
            raise ValueError("Password must be at least 4 characters.")

        # Private attributes match UML encapsulation.
        self.__user_id = user_id
        self.__university_id = university_id
        self.__name = name
        self.__email = email
        self.__password = password
        self.__role = role

        # Every new user starts with STANDARD access and ACTIVE status.
        # This matches the business rule that upgrades happen later.
        self.__access_type = AccessType.STANDARD
        self.__status = UserStatus.ACTIVE

        # Penalty credits start at 3.
        # Each violation decreases this by 1, and 0 causes freezing.
        self.__penalty_credits = 3

    def get_user_id(self):
        # Returns the user ID so the system can identify and search for a user.
        return self.__user_id

    def set_user_id(self, user_id):
        # Updates user ID only after checking it is not empty.
        if not user_id:
            raise ValueError("User ID cannot be empty.")
        self.__user_id = user_id

    def get_university_id(self):
        # Returns university or work ID for profile and ID generation tracking.
        return self.__university_id

    def set_university_id(self, university_id):
        # Updates university/work ID and prevents empty IDs.
        if not university_id:
            raise ValueError("University/work ID cannot be empty.")
        self.__university_id = university_id

    def get_name(self):
        # Returns the user's name for profile display, receipts, and booking summaries.
        return self.__name

    def set_name(self, name):
        # Updates the name after validation.
        if not name:
            raise ValueError("Name cannot be empty.")
        self.__name = name

    def get_email(self):
        # Returns email for support messages and user profile display.
        return self.__email

    def set_email(self, email):
        # Updates email and checks that it looks like an email address.
        if "@" not in email:
            raise ValueError("Invalid email format.")
        self.__email = email

    def get_password(self):
        # Returns password for login checking.
        return self.__password

    def set_password(self, password):
        # Updates password while keeping a minimum length rule.
        if len(password) < 4:
            raise ValueError("Password must be at least 4 characters.")
        self.__password = password

    def get_role(self):
        # Returns the role because access rules depend on Student, Staff, or Admin.
        return self.__role

    def set_role(self, role):
        # Updates the role only if it is a valid UserRole enum.
        if not isinstance(role, UserRole):
            raise TypeError("role must be UserRole.")
        self.__role = role

    def get_access_type(self):
        # Returns STANDARD or PREMIUM because booking permissions depend on it.
        return self.__access_type

    def set_access_type(self, access_type):
        # Updates access type after checking it is a valid AccessType enum.
        if not isinstance(access_type, AccessType):
            raise TypeError("accessType must be AccessType.")
        self.__access_type = access_type

    def get_status(self):
        # Returns ACTIVE or FROZEN. Frozen users cannot book.
        return self.__status

    def set_status(self, status):
        # Updates account status and prevents invalid status values.
        if not isinstance(status, UserStatus):
            raise TypeError("status must be UserStatus.")
        self.__status = status

    def get_penalty_credits(self):
        # Returns remaining penalty credits for admin monitoring and user display.
        return self.__penalty_credits

    def set_penalty_credits(self, penalty_credits):
        # Updates penalty credits but prevents negative values.
        if penalty_credits < 0:
            raise ValueError("Penalty credits cannot be negative.")
        self.__penalty_credits = penalty_credits

    def decrease_penalty_credit(self):
        # Decreases penalty credit after cancellation or no-show.
        # This supports the rule that repeated violations can freeze the user.
        if self.__penalty_credits > 0:
            self.__penalty_credits -= 1

    def reset_penalty_credits(self):
        # Restores penalty credits and makes the user active again.
        # This is used when admin unfreezes the account.
        self.__penalty_credits = 3
        self.__status = UserStatus.ACTIVE

    def login(self, user_id, password):
        # Checks if the entered login information matches this user.
        # This function exists so BookingSystem can validate users without directly accessing private password data.
        return self.__user_id == user_id and self.__password == password

    def update_profile(self, name, email):
        # Updates user profile using setters so validation still happens.
        self.set_name(name)
        self.set_email(email)

    def request_access_upgrade(self):
        # Represents a user asking to upgrade.
        # The actual payment and access update happens inside BookingSystem.
        return "Upgrade request submitted."

    def request_access_downgrade(self):
        # Allows user to downgrade back to STANDARD.
        # Downgrade does not require payment in the business rules.
        self.__access_type = AccessType.STANDARD

    def contact_support(self, subject, message):
        # Lets user send a support issue.
        # The subject and message are required so support receives meaningful details.
        if not subject or not message:
            raise ValueError("Subject and message are required.")
        return f"Support request submitted: {subject} - {message}"

# RELATIONSHIP EVIDENCE: Student inherits from User.
# This shows inheritance because Student reuses User attributes and methods.
class Student(User):
    """
    Student is a child class of User.

    Purpose:
    - Represents student users.
    - Stores student-specific bookings.
    - Supports the business rules for student facility access.

    Logical reason:
    Students have different permissions from Staff and Admin, so they need their own child class while still sharing User data.
    """

    def __init__(self, user_id, university_id, name, email, password):
        # super() calls the User constructor and sets role to STUDENT.
        super().__init__(user_id, university_id, name, email, password, UserRole.STUDENT)
        self.__student_bookings = {}

    def get_student_bookings(self):
        # Returns student booking records for profile display or testing.
        return self.__student_bookings

    def set_student_bookings(self, student_bookings):
        # Updates student bookings and checks the value is a dictionary.
        if not isinstance(student_bookings, dict):
            raise TypeError("studentBookings must be dictionary.")
        self.__student_bookings = student_bookings

    def view_student_bookings(self):
        # Shows all bookings connected to this student.
        return self.__student_bookings

    def book_study_room(self):
        # This represents the student's ability to book StudyRoom.
        # Actual booking validation is handled in BookingSystem and AccessPolicy.
        return "Student can book StudyRoom."

    def book_sports_court(self):
        # This represents the rule that only PREMIUM students can book SportsCourt.
        # AccessPolicy enforces the real permission check.
        return "Premium Student can book SportsCourt."

    def book_event_hall(self):
        # This represents the rule that only PREMIUM students can book EventHall.
        # BookingSystem checks this before creating a booking.
        return "Premium Student can book EventHall."

# RELATIONSHIP EVIDENCE: Staff inherits from User.
# This shows inheritance because Staff is a specialized type of User.
class Staff(User):
    """
    Staff is a child class of User.

    Purpose:
    - Represents staff users.
    - Stores staff-specific bookings.
    - Supports staff booking permissions.

    Logical reason:
    Staff rules are different from student rules. Staff can book EventHall, and PREMIUM staff can book SportsCourt, but staff can never book StudyRoom.
    """

    def __init__(self, user_id, university_id, name, email, password):
        # super() calls the User constructor and sets role to STAFF.
        super().__init__(user_id, university_id, name, email, password, UserRole.STAFF)
        self.__staff_bookings = {}

    def get_staff_bookings(self):
        # Returns staff booking records.
        return self.__staff_bookings

    def set_staff_bookings(self, staff_bookings):
        # Updates staff bookings and ensures dictionary structure.
        if not isinstance(staff_bookings, dict):
            raise TypeError("staffBookings must be dictionary.")
        self.__staff_bookings = staff_bookings

    def view_staff_bookings(self):
        # Displays staff bookings for profile or report use.
        return self.__staff_bookings

    def book_event_hall(self):
        # Represents staff permission for EventHall.
        # Standard and Premium staff can both book EventHall.
        return "Staff can book EventHall."

    def book_sports_court(self):
        # Represents the rule that only PREMIUM staff can book SportsCourt.
        return "Premium Staff can book SportsCourt."

# RELATIONSHIP EVIDENCE: Administrator inherits from User.
# This shows inheritance because Admin is also a User but has extra management functions.
class Administrator(User):
    """
    Administrator is a child class of User.

    Purpose:
    - Manages users.
    - Freezes or unfreezes accounts.
    - Upgrades or downgrades access.
    - Makes announcements.
    - Opens or closes facility slots.

    Logical reason:
    Admin has management permissions that normal users should not have.
    """

    def __init__(self, user_id, university_id, name, email, password):
        # super() calls the User constructor and sets role to ADMIN.
        super().__init__(user_id, university_id, name, email, password, UserRole.ADMIN)
        self.__admin_bookings_view = {}

    def get_admin_bookings_view(self):
        # Returns admin view of bookings.
        return self.__admin_bookings_view

    def set_admin_bookings_view(self, admin_bookings_view):
        # Updates admin booking view and ensures dictionary structure.
        if not isinstance(admin_bookings_view, dict):
            raise TypeError("adminBookingsView must be dictionary.")
        self.__admin_bookings_view = admin_bookings_view

    def freeze_user(self, user):
        # Freezes a user after violations.
        # Frozen users cannot book facilities.
        user.set_status(UserStatus.FROZEN)

    def unfreeze_user(self, user):
        # Unfreezes a user by resetting penalty credits and status.
        user.reset_penalty_credits()

    def upgrade_user_access(self, user):
        # Admin can help users upgrade when needed.
        user.set_access_type(AccessType.PREMIUM)

    def downgrade_user_access(self, user):
        # Admin can downgrade users back to STANDARD.
        user.set_access_type(AccessType.STANDARD)

    def make_announcement(self, announcements, message):
        # Adds announcement with timestamp.
        # Used for maintenance messages and system updates.
        if not message:
            raise ValueError("Announcement cannot be empty.")
        stamp = datetime.now().strftime("%Y-%m-%d %H:%M")
        announcements.append(f"{stamp}: {message}")

    def close_facility_slot(self, facility, selected_date, slot):
        # Admin closes a facility slot so users cannot book it.
        # Used for maintenance or unavailable times.
        facility.close_slot(selected_date, slot)

    def open_facility_slot(self, facility, selected_date, slot):
        # Admin reopens a previously closed slot.
        facility.open_slot(selected_date, slot)


class Facility:
    """
    Facility is the parent class for StudyRoom, SportsCourt, and EventHall.

    Purpose:
    - Stores shared facility information.
    - Manages available, booked, and closed time slots.
    - Calculates basic facility cost.

    Logical reason:
    All facility types share common data like location, capacity, slots, and price, so a parent class avoids repeated code.
    """

    def __init__(self, facility_id, facility_name, facility_type, location, capacity, price_per_hour, rules):
        self.__facility_id = facility_id
        self.__facility_name = facility_name
        self.__facility_type = facility_type
        self.__location = location
        self.__capacity = capacity
        self.__is_available = True
        self.__price_per_hour = price_per_hour
        self.__rules = rules

        # Fixed slots are used so the GUI can show clear booking options.
        self.__time_slots = [
            "09:00-10:00",
            "10:00-11:00",
            "11:00-12:00",
            "12:00-13:00",
            "14:00-15:00",
            "15:00-16:00"
        ]

        # booked_slots_by_date stores confirmed bookings per date.
        # closed_slots_by_date stores admin closed slots per date.
        self.__booked_slots_by_date = {}
        self.__closed_slots_by_date = {}

    def get_facility_id(self):
        # Returns facility ID so the catalog can find this facility.
        return self.__facility_id

    def set_facility_id(self, facility_id):
        # Updates facility ID.
        self.__facility_id = facility_id

    def get_facility_name(self):
        # Returns facility name for GUI and booking summaries.
        return self.__facility_name

    def set_facility_name(self, facility_name):
        # Updates displayed facility name.
        self.__facility_name = facility_name

    def get_facility_type(self):
        # Returns facility type because access rules depend on it.
        return self.__facility_type

    def set_facility_type(self, facility_type):
        # Updates facility type.
        self.__facility_type = facility_type

    def get_location(self):
        # Returns facility location for user display.
        return self.__location

    def set_location(self, location):
        # Updates facility location.
        self.__location = location

    def get_capacity(self):
        # Returns maximum number of people allowed.
        return self.__capacity

    def set_capacity(self, capacity):
        # Capacity must be positive because a facility cannot hold 0 or negative people.
        if capacity <= 0:
            raise ValueError("Capacity must be positive.")
        self.__capacity = capacity

    def get_is_available(self):
        # Returns general facility availability.
        return self.__is_available

    def set_is_available(self, is_available):
        # Opens or closes the whole facility.
        self.__is_available = is_available

    def get_price_per_hour(self):
        # Returns the basic hourly price.
        return self.__price_per_hour

    def set_price_per_hour(self, price_per_hour):
        # Price cannot be negative because cost must be valid.
        if price_per_hour < 0:
            raise ValueError("Price cannot be negative.")
        self.__price_per_hour = price_per_hour

    def get_rules(self):
        # Returns facility rules to show users before booking.
        return self.__rules

    def set_rules(self, rules):
        # Updates facility rules.
        self.__rules = rules

    def get_time_slots(self):
        # Returns all possible time slots.
        return self.__time_slots

    def set_time_slots(self, time_slots):
        # Updates available slot list if admin changes schedule.
        self.__time_slots = time_slots

    def get_booked_slots_by_date(self):
        # Returns booked slots grouped by date.
        return self.__booked_slots_by_date

    def set_booked_slots_by_date(self, booked_slots_by_date):
        # Used when repairing loaded pickle data.
        self.__booked_slots_by_date = booked_slots_by_date

    def get_closed_slots_by_date(self):
        # Returns admin-closed slots grouped by date.
        return self.__closed_slots_by_date

    def set_closed_slots_by_date(self, closed_slots_by_date):
        # Used when repairing old pickle data.
        self.__closed_slots_by_date = closed_slots_by_date

    def get_closed_slots(self, booking_date):
        # Makes sure every date has a closed-slot list.
        # This prevents key errors when checking closed slots.
        if booking_date not in self.__closed_slots_by_date:
            self.__closed_slots_by_date[booking_date] = []
        return self.__closed_slots_by_date[booking_date]

    def close_slot(self, booking_date, slot):
        # Closes one specific slot for maintenance or admin reasons.
        # Users cannot book slots in this closed list.
        if slot not in self.__time_slots:
            raise ValueError("Invalid time slot.")
        if booking_date not in self.__closed_slots_by_date:
            self.__closed_slots_by_date[booking_date] = []
        if slot not in self.__closed_slots_by_date[booking_date]:
            self.__closed_slots_by_date[booking_date].append(slot)

    def open_slot(self, booking_date, slot):
        # Reopens a closed slot so users can book it again.
        if booking_date in self.__closed_slots_by_date:
            if slot in self.__closed_slots_by_date[booking_date]:
                self.__closed_slots_by_date[booking_date].remove(slot)

    def get_available_slots(self, booking_date):
        # Returns slots that are not booked and not closed.
        # This is needed so users only see valid booking choices.
        if booking_date not in self.__booked_slots_by_date:
            self.__booked_slots_by_date[booking_date] = {}
        if booking_date not in self.__closed_slots_by_date:
            self.__closed_slots_by_date[booking_date] = []

        booked = self.__booked_slots_by_date[booking_date]
        closed = self.__closed_slots_by_date[booking_date]

        return [
            slot for slot in self.__time_slots
            if slot not in booked and slot not in closed
        ]

    def book_slot(self, booking_date, slot, booking_id):
        # Marks a slot as booked for a specific date.
        # This prevents another user from taking the same slot.
        if slot not in self.__time_slots:
            raise ValueError("Invalid time slot.")
        if booking_date not in self.__booked_slots_by_date:
            self.__booked_slots_by_date[booking_date] = {}
        if slot in self.__booked_slots_by_date[booking_date]:
            raise ValueError("This slot is already booked.")
        if slot in self.get_closed_slots(booking_date):
            raise ValueError("This slot is closed by admin.")
        self.__booked_slots_by_date[booking_date][slot] = booking_id

    def release_slot(self, booking_date, slot):
        # Frees a slot after cancellation or no-show.
        # This allows another user to book it later.
        if booking_date in self.__booked_slots_by_date:
            if slot in self.__booked_slots_by_date[booking_date]:
                del self.__booked_slots_by_date[booking_date][slot]

    def calculate_cost(self, usage_type):
        # Returns the normal price per hour.
        # Child classes can override this for special pricing.
        return self.__price_per_hour

    def display_details(self):
        # Returns a simple facility summary for GUI lists.
        return f"{self.__facility_name} | {self.__location} | Capacity: {self.__capacity}"

# RELATIONSHIP EVIDENCE: StudyRoom inherits from Facility.
# This shows inheritance because StudyRoom uses common facility details from Facility.
class StudyRoom(Facility):
    """
    StudyRoom represents a student-only facility.

    Purpose:
    - Allows students to book study spaces.
    - Has no booking fee.

    Logical reason:
    Study rooms are restricted to students only, so this child class separates it from SportsCourt and EventHall.
    """

    def __init__(self, facility_id, room_code, location, capacity):
        super().__init__(
            facility_id,
            f"Study Room {room_code}",
            "STUDY_ROOM",
            location,
            capacity,
            0,
            "Rules: Students only. Keep the room clean. Arrive on time."
        )
        self.__room_code = room_code

    def get_room_code(self):
        # Returns room code such as A or B.
        return self.__room_code

    def set_room_code(self, room_code):
        # Updates room code.
        self.__room_code = room_code

# RELATIONSHIP EVIDENCE: SportsCourt inherits from Facility.
# This shows inheritance because SportsCourt is a specialized type of Facility.
class SportsCourt(Facility):
    """
    SportsCourt represents sports facilities.

    Purpose:
    - Allows PREMIUM students and PREMIUM staff to book courts.
    - Stores sport type and hourly price.

    Logical reason:
    SportsCourt has different access rules from StudyRoom and EventHall.
    """

    def __init__(self, facility_id, sport_type, location, capacity, price_per_hour):
        super().__init__(
            facility_id,
            f"{sport_type} Court",
            "SPORTS_COURT",
            location,
            capacity,
            price_per_hour,
            "Rules: Sports shoes required. Respect the booked time."
        )
        self.__sport_type = sport_type

    def get_sport_type(self):
        # Returns sport type such as Basketball or Tennis.
        return self.__sport_type

    def set_sport_type(self, sport_type):
        # Updates sport type.
        self.__sport_type = sport_type

# RELATIONSHIP EVIDENCE: EventHall inherits from Facility.
# This shows inheritance because EventHall shares facility data but overrides cost logic.
class EventHall(Facility):
    """
    EventHall represents large event spaces.

    Purpose:
    - Allows staff to book halls.
    - Allows PREMIUM students to book halls.
    - Charges external use but keeps internal use free.

    Logical reason:
    EventHall has special payment logic, so it overrides calculate_cost().
    """

    def __init__(self, facility_id, location, capacity, external_rate):
        super().__init__(
            facility_id,
            "Event Hall",
            "EVENT_HALL",
            location,
            capacity,
            external_rate,
            "Rules: Internal use is free. External use requires payment."
        )
        self.__internal_use_free = True
        self.__external_rate = external_rate

    def get_internal_use_free(self):
        # Returns whether internal use is free.
        return self.__internal_use_free

    def set_internal_use_free(self, internal_use_free):
        # Updates internal use free rule.
        self.__internal_use_free = internal_use_free

    def get_external_rate(self):
        # Returns price charged for external event hall use.
        return self.__external_rate

    def set_external_rate(self, external_rate):
        # Updates external rate and parent price.
        self.__external_rate = external_rate
        self.set_price_per_hour(external_rate)

    def calculate_cost(self, usage_type):
        # Internal use is free based on business rules.
        if usage_type == UsageType.INTERNAL.value:
            return 0

        # External use requires payment.
        return self.__external_rate


class TimeSlot:
    """
    TimeSlot represents a simple time period.

    Purpose:
    - Stores start and end time.
    - Tracks if the slot is booked.

    Logical reason:
    This supports the UML TimeSlot class, but the current system mainly uses string slots like "10:00-11:00" inside Facility.
    """

    def __init__(self, slot_id, start_time, end_time):
        self.__slot_id = slot_id
        self.__start_time = start_time
        self.__end_time = end_time
        self.__is_booked = False

    def get_slot_id(self):
        # Returns slot ID.
        return self.__slot_id

    def set_slot_id(self, slot_id):
        # Updates slot ID.
        self.__slot_id = slot_id

    def get_start_time(self):
        # Returns slot start time.
        return self.__start_time

    def set_start_time(self, start_time):
        # Updates slot start time.
        self.__start_time = start_time

    def get_end_time(self):
        # Returns slot end time.
        return self.__end_time

    def set_end_time(self, end_time):
        # Updates slot end time.
        self.__end_time = end_time

    def get_is_booked(self):
        # Returns whether this TimeSlot object is booked.
        return self.__is_booked

    def set_is_booked(self, is_booked):
        # Updates booking state for this TimeSlot.
        self.__is_booked = is_booked


class Booking:
    """
    Booking connects User, Facility, and selected slot.

    Purpose:
    - Stores reservation details.
    - Tracks booking cost and status.
    - Generates booking summaries.

    Logical reason:
    A booking is the main relationship object that proves a user reserved a facility at a specific time.
    """

    def __init__(self, booking_id, user, facility, slot, booking_date, usage_type, total_cost):
        self.__booking_id = booking_id
        # RELATIONSHIP EVIDENCE: Booking is associated with User.
        # A Booking stores one User object, which proves each booking belongs to one user.
        self.__user = user
        # RELATIONSHIP EVIDENCE: Booking is associated with Facility.
        # A Booking stores one Facility object, which proves each booking reserves one facility.
        self.__facility = facility
        # RELATIONSHIP EVIDENCE: Booking is associated with a time slot.
        # The booking stores the selected slot string, so the reservation is tied to one time period.
        self.__slot = slot
        self.__booking_date = booking_date
        self.__usage_type = usage_type
        self.__total_cost = total_cost
        self.__status = BookingStatus.CONFIRMED

    def get_booking_id(self):
        # Returns booking ID for searching, modifying, or cancelling.
        return self.__booking_id

    def get_user(self):
        # Returns the user connected to this booking.
        return self.__user

    def get_facility(self):
        # Returns the facility connected to this booking.
        return self.__facility

    def get_slot(self):
        # Returns the booked time slot.
        return self.__slot

    def set_slot(self, slot):
        # Updates the booking slot when a booking is modified.
        self.__slot = slot

    def get_booking_date(self):
        # Returns booking date for reports and conflict checks.
        return self.__booking_date

    def set_booking_date(self, booking_date):
        # Updates booking date when booking is modified.
        self.__booking_date = booking_date

    def get_usage_type(self):
        # Returns internal or external usage type for cost rules.
        return self.__usage_type

    def get_total_cost(self):
        # Returns total cost for payment and receipt.
        return self.__total_cost

    def get_status(self):
        # Returns current booking status.
        return self.__status

    def set_status(self, status):
        # Updates booking status.
        self.__status = status

    def cancel_booking(self):
        # Marks booking as cancelled.
        # BookingSystem also releases the facility slot and applies penalty.
        self.__status = BookingStatus.CANCELLED

    def mark_no_show(self):
        # Marks booking as no-show.
        # BookingSystem uses this to apply penalty credit loss.
        self.__status = BookingStatus.NO_SHOW

    def generate_booking_summary(self):
        # Creates readable booking details for receipt, testing, and GUI display.
        return (
            f"Booking ID: {self.__booking_id}\n"
            f"User: {self.__user.get_name()}\n"
            f"Facility: {self.__facility.get_facility_name()}\n"
            f"Facility Type: {self.__facility.get_facility_type()}\n"
            f"Location: {self.__facility.get_location()}\n"
            f"Date: {self.__booking_date}\n"
            f"Time Slot: {self.__slot}\n"
            f"Usage Type: {self.__usage_type}\n"
            f"Cost: {self.__total_cost} AED\n"
            f"Status: {self.__status.value}\n"
            f"{self.__facility.get_rules()}"
        )


class Payment:
    """
    Payment represents payment for bookings or upgrades.

    Purpose:
    - Stores payment ID, amount, date, status, and purpose.
    - Processes payments.

    Logical reason:
    Some bookings are free and some require payment, so the system needs a payment object to track the financial result.
    """

    def __init__(self, payment_id, amount, purpose):
        self.__payment_id = payment_id
        self.__amount = amount
        self.__payment_date = str(date.today())
        self.__status = PaymentStatus.PENDING if amount > 0 else PaymentStatus.NOT_REQUIRED
        self.__purpose = purpose

    def get_payment_id(self):
        # Returns payment ID for receipt and payment records.
        return self.__payment_id

    def get_amount(self):
        # Returns payment amount.
        return self.__amount

    def get_payment_date(self):
        # Returns payment date.
        return self.__payment_date

    def get_status(self):
        # Returns payment status.
        return self.__status

    def set_status(self, status):
        # Updates payment status.
        self.__status = status

    def get_purpose(self):
        # Returns whether payment is for booking fee or upgrade.
        return self.__purpose

    def process_payment(self):
        # Processes payment.
        # If amount is 0, payment is not required.
        # If amount is more than 0, payment becomes PAID.
        if self.__amount == 0:
            self.__status = PaymentStatus.NOT_REQUIRED
        else:
            self.__status = PaymentStatus.PAID
        return True


class Receipt:
    """
    Receipt is proof of booking/payment.

    Purpose:
    - Combines booking and payment details.
    - Generates receipt text.

    Logical reason:
    After booking and payment, the user needs evidence that the transaction was completed.
    """

    def __init__(self, receipt_id, booking, payment):
        self.__receipt_id = receipt_id
        # RELATIONSHIP EVIDENCE: Receipt is associated with Booking.
        # Receipt stores the booking object so it can display booking details in the receipt.
        self.__booking = booking
        # RELATIONSHIP EVIDENCE: Receipt is connected to Payment.
        # Receipt stores the payment object because receipt information depends on payment result.
        self.__payment = payment
        self.__issue_date = str(date.today())

    def get_receipt_id(self):
        # Returns receipt ID for storage and search.
        return self.__receipt_id

    def generate_receipt(self):
        # Generates formatted receipt details for the user.
        return (
            "BOOKING RECEIPT\n"
            "----------------------\n"
            f"Receipt ID: {self.__receipt_id}\n"
            f"Issue Date: {self.__issue_date}\n"
            f"Payment Status: {self.__payment.get_status().value}\n"
            f"Payment Amount: {self.__payment.get_amount()} AED\n\n"
            f"{self.__booking.generate_booking_summary()}"
        )


class Notification:
    """
    Notification stores messages sent by the system.

    Purpose:
    - Stores booking confirmations.
    - Stores freeze warnings.
    - Stores maintenance or admin messages.

    Logical reason:
    Users need to be informed when important system actions happen.
    """

    def __init__(self, notification_id, message):
        self.__notification_id = notification_id
        self.__message = message
        self.__send_date = str(date.today())
        self.__is_read = False

    def get_notification_id(self):
        # Returns notification ID for storage.
        return self.__notification_id

    def get_message(self):
        # Returns message text for GUI display.
        return self.__message

    def get_send_date(self):
        # Returns notification date.
        return self.__send_date

    def get_is_read(self):
        # Returns whether user has read the notification.
        return self.__is_read

    def mark_as_read(self):
        # Marks notification as read after user views it.
        self.__is_read = True


class AccessPolicy:
    """
    AccessPolicy controls booking permission rules.

    Purpose:
    - Checks role.
    - Checks access type.
    - Checks facility type.

    Logical reason:
    Keeping permission rules in one class makes the system easier to update if business rules change.
    """
    # RELATIONSHIP EVIDENCE: AccessPolicy depends on User and Facility.
    # This method receives user and facility as parameters to check if booking is allowed.
    def can_book(self, user, facility):
        # Get user and facility details needed for permission checking.
        # RELATIONSHIP EVIDENCE: AccessPolicy depends on User.
        # It reads user role and access type to apply permission rules.
        role = user.get_role()
        access = user.get_access_type()
        # RELATIONSHIP EVIDENCE: AccessPolicy depends on Facility.
        # It reads facility type to decide whether the user can book it.
        facility_type = facility.get_facility_type()

        # Admin manages the system but does not make normal facility bookings.
        if role == UserRole.ADMIN:
            return False

        # StudyRoom is student-only.
        if facility_type == "STUDY_ROOM":
            return role == UserRole.STUDENT

        # SportsCourt is only for PREMIUM students and PREMIUM staff.
        if facility_type == "SPORTS_COURT":
            return access == AccessType.PREMIUM and role in [UserRole.STUDENT, UserRole.STAFF]

        # EventHall can be booked by all staff and PREMIUM students.
        if facility_type == "EVENT_HALL":
            if role == UserRole.STAFF:
                return True
            return role == UserRole.STUDENT and access == AccessType.PREMIUM

        # Unknown facility types are rejected.
        return False


class FacilityCatalog:
    """
    FacilityCatalog stores all facility objects.

    Purpose:
    - Adds facilities.
    - Removes facilities.
    - Finds facilities by ID.

    Logical reason:
    BookingSystem needs one organized place to manage all facilities.
    """

    def __init__(self):
        # RELATIONSHIP EVIDENCE: FacilityCatalog aggregates Facilities.
        # The catalog stores many Facility objects using facility ID as the key.
        self.__facilities = {}

    def get_facilities(self):
        # Returns the full facility dictionary.
        return self.__facilities

    def set_facilities(self, facilities):
        # Replaces facility dictionary.
        # Used when loading data from pickle.
        self.__facilities = facilities

    def add_facility(self, facility):
        # Adds facility using its facility ID as the key.
        # RELATIONSHIP EVIDENCE: FacilityCatalog aggregates Facilities.
        # Adding a facility object into the catalog shows the catalog manages many facilities.
        self.__facilities[facility.get_facility_id()] = facility

    def remove_facility(self, facility_id):
        # Removes facility if it exists.
        if facility_id in self.__facilities:
            del self.__facilities[facility_id]

    def get_facility(self, facility_id):
        # Returns one facility by ID.
        return self.__facilities.get(facility_id)

    def display_all_facilities(self):
        # Returns all facilities for GUI display.
        return self.__facilities


class BookingSystem:
    """
    BookingSystem is the main controller class.

    Purpose:
    - Stores all users, bookings, payments, receipts, notifications, and announcements.
    - Handles login, registration, booking, payment, penalties, upgrades, maintenance, and persistence.

    Logical reason:
    The system needs one central controller to connect all other classes together.
    """

    def __init__(self):
        self.__system_name = "Smart Campus Facility Booking System"
        self.__support_email = "support@campus.ae"
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Users.
        # Users are stored in a dictionary so the system can manage many user objects.
        self.__users = {}
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Bookings.
        # Bookings are stored in a dictionary as system records.
        self.__bookings = {}
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Payments.
        # Payments are stored separately so the system can keep financial records.
        self.__payments = {}
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Receipts.
        # Receipts are stored separately as confirmation records.
        self.__receipts = {}
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Notifications.
        # Notifications are stored so the system can keep user messages.
        self.__notifications = {}
        self.__announcements = []
        # RELATIONSHIP EVIDENCE: BookingSystem composes FacilityCatalog.
        # BookingSystem creates and owns the FacilityCatalog object.
        self.__facility_catalog = FacilityCatalog()
        # RELATIONSHIP EVIDENCE: BookingSystem composes AccessPolicy.
        # BookingSystem creates AccessPolicy to control booking permissions.
        self.__access_policy = AccessPolicy()
        self.__data_file_name = "smart_campus_data.dat"
        self.__session_file_name = "smart_campus_session.dat"
        self.__current_user_id = ""
        self.__remember_login = False
        self.__next_booking_number = 1

        # Load saved data so the system remembers users and bookings after closing.
        self.load_data_from_file(self.__data_file_name)

        # Repair old saved data if new attributes were added later.
        self.repair_loaded_data()

        # Create default facilities only if no facilities were loaded.
        if not self.__facility_catalog.get_facilities():
            self.create_default_facilities()

        # Load saved login session if remember login was used.
        self.load_session()

    def get_system_name(self):
        # Returns system name for GUI title or reports.
        return self.__system_name

    def get_support_email(self):
        # Returns support email for contact/support section.
        return self.__support_email

    def get_users(self):
        # Returns registered users.
        return self.__users

    def get_bookings(self):
        # Returns all bookings.
        return self.__bookings

    def get_payments(self):
        # Returns all payment records.
        return self.__payments

    def get_receipts(self):
        # Returns all receipts.
        return self.__receipts

    def get_notifications(self):
        # Returns all notifications.
        return self.__notifications

    def get_announcements(self):
        # Returns system announcements.
        return self.__announcements

    def get_facility_catalog(self):
        # Returns facility catalog object.
        return self.__facility_catalog

    def get_current_user_id(self):
        # Returns the logged-in user ID.
        return self.__current_user_id

    def get_current_user(self):
        # Returns the logged-in user object if one exists.
        if self.__current_user_id in self.__users:
            return self.__users[self.__current_user_id]
        return None

    def create_default_facilities(self):
        # Creates built-in facilities so the system has ready booking options.
        self.__facility_catalog.add_facility(StudyRoom("SR-A", "A", "Library - Floor 1", 6))
        self.__facility_catalog.add_facility(StudyRoom("SR-B", "B", "Library - Floor 2", 4))
        self.__facility_catalog.add_facility(SportsCourt("SC-B", "Basketball", "Sports Complex", 10, 20))
        self.__facility_catalog.add_facility(SportsCourt("SC-T", "Tennis", "Sports Complex", 4, 15))
        self.__facility_catalog.add_facility(EventHall("EH-1", "Main Building", 50, 100))
        self.__facility_catalog.add_facility(EventHall("EH-2", "Conference Building", 100, 150))

    def repair_loaded_data(self):
        # Adds missing attributes to old pickle data.
        # This prevents crashes when the code was updated after data was saved.
        for user in self.__users.values():
            if not hasattr(user, "_User__penalty_credits"):
                user.set_penalty_credits(3)

        for facility in self.__facility_catalog.get_facilities().values():
            if not hasattr(facility, "_Facility__booked_slots_by_date"):
                facility.set_booked_slots_by_date({})
            if not hasattr(facility, "_Facility__closed_slots_by_date"):
                facility.set_closed_slots_by_date({})

    def save_data_to_file(self, file_name):
        # Saves system data in a binary pickle file.
        # This is required for persistence.
        try:
            with open(file_name, "wb") as file:
                pickle.dump({
                    "users": self.__users,
                    "bookings": self.__bookings,
                    "payments": self.__payments,
                    "receipts": self.__receipts,
                    "notifications": self.__notifications,
                    "announcements": self.__announcements,
                    "facility_catalog": self.__facility_catalog,
                    "next_booking_number": self.__next_booking_number
                }, file)
            return True
        except (pickle.PickleError, OSError) as error:
            raise OSError(f"Saving failed: {error}")

    def load_data_from_file(self, file_name):
        # Loads system data from a binary pickle file.
        # If the file is missing or corrupted, the system resets safely.
        try:
            if os.path.exists(file_name):
                with open(file_name, "rb") as file:
                    data = pickle.load(file)

                self.__users = data.get("users", {})
                self.__bookings = data.get("bookings", {})
                self.__payments = data.get("payments", {})
                self.__receipts = data.get("receipts", {})
                self.__notifications = data.get("notifications", {})
                self.__announcements = data.get("announcements", [])
                self.__facility_catalog = data.get("facility_catalog", FacilityCatalog())
                self.__next_booking_number = data.get("next_booking_number", 1)

            return self
        except (EOFError, pickle.UnpicklingError, OSError, AttributeError):
            self.__users = {}
            self.__bookings = {}
            self.__payments = {}
            self.__receipts = {}
            self.__notifications = {}
            self.__announcements = []
            self.__facility_catalog = FacilityCatalog()
            self.__next_booking_number = 1
            return self

    def save_session(self, user_id):
        # Saves current login session using pickle.
        # This supports remember-login functionality.
        try:
            with open(self.__session_file_name, "wb") as file:
                pickle.dump({"current_user_id": user_id}, file)
            return True
        except OSError:
            return False

    def load_session(self):
        # Loads saved login session if it exists.
        try:
            if os.path.exists(self.__session_file_name):
                with open(self.__session_file_name, "rb") as file:
                    data = pickle.load(file)

                user_id = data.get("current_user_id", "")
                if user_id in self.__users:
                    self.__current_user_id = user_id
                    return self.__users[user_id]
            return None
        except (EOFError, pickle.UnpicklingError, OSError, AttributeError):
            return None

    def logout(self):
        # Clears current user and removes saved login session.
        self.__current_user_id = ""
        if os.path.exists(self.__session_file_name):
            os.remove(self.__session_file_name)

    def generate_user_id(self, role, university_id):
        # Creates user ID using role prefix.
        # This keeps user IDs consistent and easy to identify.
        if role == UserRole.STUDENT:
            return "STU" + university_id
        if role == UserRole.STAFF:
            return "STF" + university_id
        if role == UserRole.ADMIN:
            return "ADM" + university_id
        raise ValueError("Invalid role.")

    def register_user(self, role, university_id, name, email, password):
        # Registers a user based on role.
        # The function creates the correct child object automatically.
        user_id = self.generate_user_id(role, university_id)

        if user_id in self.__users:
            raise ValueError("This account already exists.")
        # RELATIONSHIP EVIDENCE: BookingSystem creates different User child objects.
        # This shows polymorphism because the system creates Student, Staff, or Administrator based on role.
        if role == UserRole.STUDENT:
            user = Student(user_id, university_id, name, email, password)
        elif role == UserRole.STAFF:
            user = Staff(user_id, university_id, name, email, password)
        elif role == UserRole.ADMIN:
            user = Administrator(user_id, university_id, name, email, password)
        else:
            raise ValueError("Invalid role.")
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates User objects.
        # The created user object is stored in the users dictionary.
        self.__users[user_id] = user
        self.save_data_to_file(self.__data_file_name)
        return user

    def validate_login(self, user_id, password, remember_login=False):
        # Validates login and stores current user if successful.
        if user_id not in self.__users:
            raise ValueError(
                "User ID not found.\n\n"
                "Example:\n"
                "Student 20240001 becomes STU20240001\n"
                "Staff 8899 becomes STF8899\n"
                "Admin 1000 becomes ADM1000"
            )

        user = self.__users[user_id]

        if not user.login(user_id, password):
            raise PermissionError("Incorrect password.")

        self.__current_user_id = user_id
        self.__remember_login = remember_login

        if remember_login:
            self.save_session(user_id)

        return user

    def validate_weekday_booking(self, selected_date):
        # Stops weekend bookings.
        # The system only allows Monday to Friday bookings.
        booking_day = datetime.strptime(selected_date, "%Y-%m-%d").date()
        if booking_day.weekday() >= 5:
            raise ValueError("Bookings are only allowed from Monday to Friday.")
        return True

    def get_allowed_facilities(self, user):
        # Returns only facilities that the user is allowed to book.
        # This helps the GUI show correct options.
        allowed = {}
        for facility_id, facility in self.__facility_catalog.get_facilities().items():
            if facility.get_is_available() and self.__access_policy.can_book(user, facility):
                allowed[facility_id] = facility
        return allowed

    def count_user_bookings_on_date(self, user, selected_date):
        # Counts confirmed bookings for one user on one date.
        # Used to enforce STANDARD daily booking credit.
        count = 0
        for booking in self.__bookings.values():
            if (
                booking.get_user().get_user_id() == user.get_user_id()
                and booking.get_booking_date() == selected_date
                and booking.get_status() == BookingStatus.CONFIRMED
            ):
                count += 1
        return count

    def count_user_facility_type_bookings_on_date(self, user, facility_type, selected_date):
        # Counts confirmed bookings for one user and one facility type on a date.
        # Used to enforce PREMIUM limit of 3 bookings per facility type per day.
        count = 0
        for booking in self.__bookings.values():
            if (
                booking.get_user().get_user_id() == user.get_user_id()
                and booking.get_facility().get_facility_type() == facility_type
                and booking.get_booking_date() == selected_date
                and booking.get_status() == BookingStatus.CONFIRMED
            ):
                count += 1
        return count

    def get_daily_credit_text(self, user, selected_date=None):
        # Creates readable credit text for GUI display.
        if selected_date is None:
            selected_date = str(date.today())

        if user.get_access_type() == AccessType.STANDARD:
            used = self.count_user_bookings_on_date(user, selected_date)
            return f"Booking Credit: {max(0, 1 - used)} / 1 | Penalty Credit: {user.get_penalty_credits()} / 3"

        allowed_types = set()
        for facility in self.get_allowed_facilities(user).values():
            allowed_types.add(facility.get_facility_type())

        credit_lines = []
        for facility_type in allowed_types:
            used = self.count_user_facility_type_bookings_on_date(user, facility_type, selected_date)
            credit_lines.append(f"{facility_type}: {max(0, 3 - used)} / 3")

        return " | ".join(credit_lines) + f" | Penalty Credit: {user.get_penalty_credits()} / 3"

    def check_booking_credit(self, user, facility, selected_date):
        # Enforces daily booking limits.
        # STANDARD gets 1 booking per day.
        # PREMIUM gets 3 bookings per facility type per day.
        if user.get_access_type() == AccessType.STANDARD:
            if self.count_user_bookings_on_date(user, selected_date) >= 1:
                raise PermissionError("STANDARD access allows only 1 booking per day.")

        if user.get_access_type() == AccessType.PREMIUM:
            used = self.count_user_facility_type_bookings_on_date(
                user,
                facility.get_facility_type(),
                selected_date
            )
            if used >= 3:
                raise PermissionError(
                    f"PREMIUM access allows only 3 bookings per {facility.get_facility_type()} per day."
                )

    def check_booking_conflict(self, user, slot, selected_date):
        # Checks if user already has a confirmed booking at the same date and time.
        # This prevents double-booking conflict.
        for booking in self.__bookings.values():
            if (
                booking.get_user().get_user_id() == user.get_user_id()
                and booking.get_booking_date() == selected_date
                and booking.get_slot() == slot
                and booking.get_status() == BookingStatus.CONFIRMED
            ):
                return True
        return False

    def auto_freeze_if_needed(self, user):
        # Automatically freezes users when penalty credits reach 0.
        # Also creates a notification explaining why the account was frozen.
        if user.get_penalty_credits() <= 0:
            user.set_status(UserStatus.FROZEN)
            notification_id = "N-FREEZE-" + user.get_user_id()
            self.__notifications[notification_id] = Notification(
                notification_id,
                "Your account has been automatically frozen because you reached 3 violations."
            )

    def record_cancellation_violation(self, user):
        # Records cancellation penalty.
        # Each violation decreases penalty credit by 1.
        user.decrease_penalty_credit()
        self.auto_freeze_if_needed(user)
        self.save_data_to_file(self.__data_file_name)

    def record_no_show_violation(self, user):
        # Records no-show penalty.
        # This supports the punishment rule in the assignment.
        user.decrease_penalty_credit()
        self.auto_freeze_if_needed(user)
        self.save_data_to_file(self.__data_file_name)

    def create_booking_after_payment(self, facility_id, slot, selected_date, usage_type):
        # Main booking creation function.
        # It checks login user, permission, date, slot availability, conflicts, credit limit, payment, receipt, and notification.
        user = self.get_current_user()
        facility = self.__facility_catalog.get_facility(facility_id)

        if user.get_status() == UserStatus.FROZEN:
            raise PermissionError("Your account is FROZEN. You cannot book.")

        if not facility.get_is_available():
            raise PermissionError("Facility is unavailable.")

        datetime.strptime(selected_date, "%Y-%m-%d")
        self.validate_weekday_booking(selected_date)

        if datetime.strptime(selected_date, "%Y-%m-%d").date() < date.today():
            raise ValueError("Booking date cannot be in the past.")
        # RELATIONSHIP EVIDENCE: BookingSystem uses AccessPolicy.
        # The system calls AccessPolicy to check if the user is allowed to book this facility.
        if not self.__access_policy.can_book(user, facility):
            raise PermissionError("You cannot book this facility.")

        if slot not in facility.get_available_slots(selected_date):
            raise ValueError("This slot is already booked or closed.")

        if self.check_booking_conflict(user, slot, selected_date):
            raise ValueError("You already have a booking at this time.")

        self.check_booking_credit(user, facility, selected_date)

        total_cost = facility.calculate_cost(usage_type)

        booking_id = "B" + str(self.__next_booking_number).zfill(4)
        self.__next_booking_number += 1
        # RELATIONSHIP EVIDENCE: BookingSystem creates a Booking linked to User and Facility.
        # The booking object receives user and facility objects, proving both associations.
        booking = Booking(booking_id, user, facility, slot, selected_date, usage_type, total_cost)
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Booking objects.
        # The booking is stored in the system booking dictionary.
        self.__bookings[booking_id] = booking
        # RELATIONSHIP EVIDENCE: Facility is associated with booked slots.
        # The facility records the selected slot using booking ID, proving the booking affects facility availability.
        facility.book_slot(selected_date, slot, booking_id)
        # RELATIONSHIP EVIDENCE: BookingSystem creates Payment for Booking.
        # The payment ID is based on the booking ID, showing payment is linked to the booking transac
        payment = Payment("P" + booking_id, total_cost, PaymentPurpose.BOOKING_FEE)
        payment.process_payment()
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Payment objects.
        # The payment object is stored as a system payment record.
        self.__payments[payment.get_payment_id()] = payment
        # RELATIONSHIP EVIDENCE: Receipt is connected to Booking and Payment.
        # The receipt is created using both booking and payment objects.
        receipt = Receipt("R" + booking_id, booking, payment)
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Receipt objects.
        # The receipt object is stored as a confirmation record.
        self.__receipts[receipt.get_receipt_id()] = receipt
        # RELATIONSHIP EVIDENCE: Booking can generate Notification.
        # A booking confirmation notification is created using the booking ID.
        notification = Notification("N" + booking_id, f"Booking {booking_id} confirmed.")
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Notification objects.
        # The notification is stored in the notification dictionary.
        self.__notifications[notification.get_notification_id()] = notification

        self.save_data_to_file(self.__data_file_name)
        return booking, receipt

    def cancel_booking(self, booking_id):
        # Cancels a booking, releases the slot, and applies a cancellation penalty.
        if booking_id not in self.__bookings:
            raise ValueError("Booking not found.")

        booking = self.__bookings[booking_id]

        if booking.get_user().get_user_id() != self.__current_user_id:
            raise PermissionError("You can only cancel your own booking.")

        booking.get_facility().release_slot(booking.get_booking_date(), booking.get_slot())
        booking.cancel_booking()
        self.record_cancellation_violation(booking.get_user())
        self.save_data_to_file(self.__data_file_name)

    def modify_booking(self, booking_id, new_slot, new_date):
        # Modifies a booking by changing the slot and date.
        # The old slot is released and the new slot is booked.
        if booking_id not in self.__bookings:
            raise ValueError("Booking not found.")

        booking = self.__bookings[booking_id]

        if booking.get_user().get_user_id() != self.__current_user_id:
            raise PermissionError("You can only modify your own booking.")

        self.validate_weekday_booking(new_date)

        facility = booking.get_facility()

        if new_slot not in facility.get_available_slots(new_date):
            raise ValueError("New slot is not available.")

        datetime.strptime(new_date, "%Y-%m-%d")

        facility.release_slot(booking.get_booking_date(), booking.get_slot())
        booking.set_slot(new_slot)
        booking.set_booking_date(new_date)
        facility.book_slot(new_date, new_slot, booking_id)

        self.save_data_to_file(self.__data_file_name)

    def mark_booking_no_show(self, booking_id):
        # Marks booking as no-show, releases the slot, and applies penalty.
        if booking_id not in self.__bookings:
            raise ValueError("Booking not found.")

        booking = self.__bookings[booking_id]
        user = booking.get_user()

        booking.get_facility().release_slot(booking.get_booking_date(), booking.get_slot())
        booking.mark_no_show()

        self.record_no_show_violation(user)
        self.save_data_to_file(self.__data_file_name)

    def process_upgrade_payment(self):
        # Processes STANDARD to PREMIUM upgrade.
        # User pays upgrade fee, then access changes to PREMIUM.
        user = self.get_current_user()

        if user.get_access_type() == AccessType.PREMIUM:
            raise ValueError("User already has PREMIUM access.")

        payment = Payment("P-UP-" + user.get_user_id(), 30, PaymentPurpose.ACCESS_UPGRADE)
        payment.process_payment()
        self.__payments[payment.get_payment_id()] = payment

        user.set_access_type(AccessType.PREMIUM)
        self.save_data_to_file(self.__data_file_name)

    def downgrade_current_user(self):
        # Downgrades the logged-in user to STANDARD.
        user = self.get_current_user()
        user.set_access_type(AccessType.STANDARD)
        self.save_data_to_file(self.__data_file_name)

    def freeze_user(self, user_id):
        # Freezes a selected user manually.
        # Admin can use this after serious violations.
        if user_id not in self.__users:
            raise ValueError("User not found.")
        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            admin.freeze_user(self.__users[user_id])
        else:
            self.__users[user_id].set_status(UserStatus.FROZEN)
        self.save_data_to_file(self.__data_file_name)

    def unfreeze_user(self, user_id):
        # Unfreezes selected user and resets penalty credits.
        if user_id not in self.__users:
            raise ValueError("User not found.")
        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            # RELATIONSHIP EVIDENCE: Admin manages User.
            # The admin object freezes another user object, showing an admin-user management association.
            admin.unfreeze_user(self.__users[user_id])
        else:
            self.__users[user_id].reset_penalty_credits()
        self.save_data_to_file(self.__data_file_name)

    def upgrade_user_by_admin(self, user_id):
        # Admin upgrades any user to PREMIUM.
        if user_id not in self.__users:
            raise ValueError("User not found.")
        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            # RELATIONSHIP EVIDENCE: Admin manages User access.
            # The admin object upgrades a selected user to Premium.
            admin.upgrade_user_access(self.__users[user_id])
        else:
            self.__users[user_id].set_access_type(AccessType.PREMIUM)
        self.save_data_to_file(self.__data_file_name)

    def downgrade_user_by_admin(self, user_id):
        # Admin downgrades any user to STANDARD.
        if user_id not in self.__users:
            raise ValueError("User not found.")
        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            admin.downgrade_user_access(self.__users[user_id])
        else:
            self.__users[user_id].set_access_type(AccessType.STANDARD)
        self.save_data_to_file(self.__data_file_name)

    def close_facility_slot(self, facility_id, selected_date, slot):
        # Closes one facility slot for maintenance.
        # Users cannot book this closed slot.
        # RELATIONSHIP EVIDENCE: BookingSystem uses FacilityCatalog.
        # The system asks the catalog to find the facility object by facility ID.
        facility = self.__facility_catalog.get_facility(facility_id)
        if facility is None:
            raise ValueError("Facility not found.")

        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            admin.close_facility_slot(facility, selected_date, slot)
        else:
            facility.close_slot(selected_date, slot)

        self.save_data_to_file(self.__data_file_name)

    def open_facility_slot(self, facility_id, selected_date, slot):
        # Opens a previously closed facility slot.
        facility = self.__facility_catalog.get_facility(facility_id)
        if facility is None:
            raise ValueError("Facility not found.")

        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            admin.open_facility_slot(facility, selected_date, slot)
        else:
            facility.open_slot(selected_date, slot)

        self.save_data_to_file(self.__data_file_name)

    def add_announcement(self, message):
        # Adds maintenance or system announcement.
        # This supports the admin announcement test case.
        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            admin.make_announcement(self.__announcements, message)
        else:
            if not message:
                raise ValueError("Announcement cannot be empty.")
            stamp = datetime.now().strftime("%Y-%m-%d %H:%M")
            self.__announcements.append(f"{stamp}: {message}")

        self.save_data_to_file(self.__data_file_name)

    def get_bookings_by_date(self, selected_date):
        # Returns all bookings on a selected date.
        # Admin uses this for monitoring daily booking activity.
        return {
            booking_id: booking
            for booking_id, booking in self.__bookings.items()
            if booking.get_booking_date() == selected_date
        }


GUI.py

GUI+MainCode

In [ ]:
"""
Smart Campus Facility Booking System

This file contains the full backend and Tkinter GUI for the booking system.

Purpose:
- Implements the UML classes.
- Demonstrates relationships between users, facilities, bookings, payments, receipts, notifications, and GUI screens.
- Uses pickle for binary data persistence.
- Provides a user-friendly interface for students, staff, and administrators.

Logical reason:
The assignment requires working code, GUI features, data storage, error handling, testing scenarios, and clear documentation.
The comments in this file explain why each class and function exists, not only what it does.
"""

from enum import Enum
import pickle
import os
import calendar
import tkinter as tk
from tkinter import ttk, messagebox
from datetime import datetime, date


class UserRole(Enum):
    """
    UserRole stores the allowed actor types in the system.

    Purpose:
    - STUDENT, STAFF, and ADMIN are the only valid roles.
    - This prevents spelling mistakes and keeps role checks consistent.

    Logical reason:
    Booking rules depend on the role, so using an Enum makes the code safer than using random strings.
    """
    STUDENT = "STUDENT"
    STAFF = "STAFF"
    ADMIN = "ADMIN"


class AccessType(Enum):
    """
    AccessType stores the access level of a user.

    Purpose:
    - STANDARD users have limited booking permissions.
    - PREMIUM users have extra booking permissions and more booking credits.

    Logical reason:
    The system needs a clear way to check if a user can book premium facilities.
    """
    STANDARD = "STANDARD"
    PREMIUM = "PREMIUM"


class UserStatus(Enum):
    """
    UserStatus stores whether the user account is active or frozen.

    Purpose:
    - ACTIVE users can use the system normally.
    - FROZEN users are blocked from making bookings.

    Logical reason:
    This supports the penalty rule where users can be frozen after repeated violations.
    """
    ACTIVE = "ACTIVE"
    FROZEN = "FROZEN"


class BookingStatus(Enum):
    """
    BookingStatus tracks the lifecycle of a booking.

    Purpose:
    - CONFIRMED means the booking is active.
    - CANCELLED means the booking was cancelled.
    - NO_SHOW means the user did not attend.

    Logical reason:
    The admin dashboard and penalty system need to know the booking state.
    """
    PENDING = "PENDING"
    CONFIRMED = "CONFIRMED"
    CANCELLED = "CANCELLED"
    NO_SHOW = "NO_SHOW"


class UsageType(Enum):
    """
    UsageType explains why the facility is being used.

    Purpose:
    - INTERNAL use can be free for EventHall.
    - EXTERNAL use may require payment.

    Logical reason:
    EventHall cost depends on whether the booking is internal or external.
    """
    INTERNAL = "INTERNAL"
    EXTERNAL = "EXTERNAL"


class PaymentStatus(Enum):
    """
    PaymentStatus tracks payment progress and result.

    Purpose:
    - NOT_REQUIRED is used for free bookings.
    - PAID is used after a successful payment.
    - REFUNDED is available for returned payments.

    Logical reason:
    Receipts and booking records need a clear payment state.
    """
    NOT_REQUIRED = "NOT_REQUIRED"
    PENDING = "PENDING"
    PAID = "PAID"
    FAILED = "FAILED"
    REFUNDED = "REFUNDED"


class PaymentPurpose(Enum):
    """
    PaymentPurpose explains why a payment was created.

    Purpose:
    - BOOKING_FEE is used for paid facility bookings.
    - ACCESS_UPGRADE is used for upgrading to premium.

    Logical reason:
    This helps separate booking payments from upgrade payments.
    """
    BOOKING_FEE = "BOOKING_FEE"
    ACCESS_UPGRADE = "ACCESS_UPGRADE"


class User:
    """
    User is the parent class for Student, Staff, and Administrator.

    Purpose:
    - Stores shared user information such as ID, name, email, password, role, access type, status, and penalty credits.
    - Provides shared actions like login, profile update, support contact, and access requests.

    Logical reason:
    Student, Staff, and Admin share common data, so inheritance avoids repeating the same code.
    """
    def __init__(self, user_id, university_id, name, email, password, role):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        if not user_id:
            raise ValueError("User ID cannot be empty.")
        if not university_id:
            raise ValueError("University/work ID cannot be empty.")
        if not name:
            raise ValueError("Name cannot be empty.")
        if "@" not in email:
            raise ValueError("Invalid email format.")
        if len(password) < 4:
            raise ValueError("Password must be at least 4 characters.")

        self.__user_id = user_id
        self.__university_id = university_id
        self.__name = name
        self.__email = email
        self.__password = password
        self.__role = role
        self.__access_type = AccessType.STANDARD
        self.__status = UserStatus.ACTIVE
        self.__penalty_credits = 3

    def get_user_id(self):
        """
        Purpose:
        - Returns the user ID so the system can identify the user.

        Logical reason:
        Many actions such as login, booking ownership, and admin updates depend on user ID.
        """
        return self.__user_id

    def set_user_id(self, user_id):
        """
        Purpose:
        - Updates the user ID after validation.

        Logical reason:
        Setters protect private attributes from invalid changes.
        """
        if not user_id:
            raise ValueError("User ID cannot be empty.")
        self.__user_id = user_id

    def get_university_id(self):
        """
        Purpose:
        - Returns university or work ID.

        Logical reason:
        This ID is used to create the system login ID.
        """
        return self.__university_id

    def set_university_id(self, university_id):
        """
        Purpose:
        - Updates university or work ID after validation.

        Logical reason:
        The system should not store empty identification data.
        """
        if not university_id:
            raise ValueError("University/work ID cannot be empty.")
        self.__university_id = university_id

    def get_name(self):
        """
        Purpose:
        - Returns the user's name for profiles, receipts, and dashboards.

        Logical reason:
        Readable names make booking records easier to understand.
        """
        return self.__name

    def set_name(self, name):
        """
        Purpose:
        - Updates the user's name after validation.

        Logical reason:
        Profile changes should still protect required data.
        """
        if not name:
            raise ValueError("Name cannot be empty.")
        self.__name = name

    def get_email(self):
        """
        Purpose:
        - Returns the user's email.

        Logical reason:
        Email is part of the profile and support contact information.
        """
        return self.__email

    def set_email(self, email):
        """
        Purpose:
        - Updates the email after checking its format.

        Logical reason:
        The system should not save invalid contact information.
        """
        if "@" not in email:
            raise ValueError("Invalid email format.")
        self.__email = email

    def get_password(self):
        """
        Purpose:
        - Returns the saved password for login checking.

        Logical reason:
        The login function needs to compare entered and stored passwords.
        """
        return self.__password

    def set_password(self, password):
        """
        Purpose:
        - Updates the password with a minimum length rule.

        Logical reason:
        This prevents very weak or empty passwords.
        """
        if len(password) < 4:
            raise ValueError("Password must be at least 4 characters.")
        self.__password = password

    def get_role(self):
        """
        Purpose:
        - Returns the user's role.

        Logical reason:
        Access rules depend on whether the user is Student, Staff, or Admin.
        """
        return self.__role

    def set_role(self, role):
        """
        Purpose:
        - Updates the role only if it is a valid UserRole enum.

        Logical reason:
        This prevents invalid role values.
        """
        if not isinstance(role, UserRole):
            raise TypeError("role must be UserRole.")
        self.__role = role

    def get_access_type(self):
        """
        Purpose:
        - Returns STANDARD or PREMIUM access.

        Logical reason:
        Facility permissions and booking credits depend on access type.
        """
        return self.__access_type

    def set_access_type(self, access_type):
        """
        Purpose:
        - Updates access level after validation.

        Logical reason:
        Upgrade and downgrade actions must change this value safely.
        """
        if not isinstance(access_type, AccessType):
            raise TypeError("accessType must be AccessType.")
        self.__access_type = access_type

    def get_status(self):
        """
        Purpose:
        - Returns current object status.

        Logical reason:
        The system needs status for users, bookings, or payments.
        """
        return self.__status

    def set_status(self, status):
        """
        Purpose:
        - Updates status safely.

        Logical reason:
        Object state changes during booking, payment, cancellation, and freezing.
        """
        if not isinstance(status, UserStatus):
            raise TypeError("status must be UserStatus.")
        self.__status = status

    def get_penalty_credits(self):
        """
        Purpose:
        - Returns remaining penalty credits.

        Logical reason:
        The system uses this to decide if the user should be frozen.
        """
        return self.__penalty_credits

    def set_penalty_credits(self, penalty_credits):
        """
        Purpose:
        - Updates penalty credits after checking the value is not negative.

        Logical reason:
        Penalty credits should stay within valid values.
        """
        if penalty_credits < 0:
            raise ValueError("Penalty credits cannot be negative.")
        self.__penalty_credits = penalty_credits

    def decrease_penalty_credit(self):
        """
        Purpose:
        - Reduces penalty credits after a violation.

        Logical reason:
        Repeated cancellations or no-shows should have consequences.
        """
        if self.__penalty_credits > 0:
            self.__penalty_credits -= 1

    def reset_penalty_credits(self):
        """
        Purpose:
        - Restores penalty credits and reactivates the user.

        Logical reason:
        Admin needs a way to restore frozen users.
        """
        self.__penalty_credits = 3
        self.__status = UserStatus.ACTIVE

    def login(self, user_id, password):
        """
        Purpose:
        - Checks user ID and password.

        Logical reason:
        Only valid users should enter the system.
        """
        return self.__user_id == user_id and self.__password == password

    def update_profile(self, name, email):
        """
        Purpose:
        - Updates name and email through validation setters.

        Logical reason:
        Users should be able to edit their profile safely.
        """
        self.set_name(name)
        self.set_email(email)

    def request_access_upgrade(self):
        """
        Purpose:
        - Records that a user wants premium access.

        Logical reason:
        Upgrade request is a user action before payment/admin approval.
        """
        return "Upgrade request submitted."

    def request_access_downgrade(self):
        """
        Purpose:
        - Changes the user's access back to STANDARD.

        Logical reason:
        Downgrades do not require payment in the business rules.
        """
        self.__access_type = AccessType.STANDARD

    def contact_support(self, subject, message):
        """
        Purpose:
        - Lets a user submit a support message.

        Logical reason:
        Users need a way to report issues such as upgrade or booking problems.
        """
        if not subject or not message:
            raise ValueError("Subject and message are required.")
        return f"Support request submitted: {subject} - {message}"


class Student(User):
    """
    Student represents a student actor in the system.

    Purpose:
    - Stores student-specific booking records.
    - Uses the shared User behavior from the parent class.

    Logical reason:
    Students follow different booking rules from staff, so they are modeled as their own class.
    """
    def __init__(self, user_id, university_id, name, email, password):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        super().__init__(user_id, university_id, name, email, password, UserRole.STUDENT)
        self.__student_bookings = {}

    def get_student_bookings(self):
        """
        Purpose:
        - Returns student-specific booking records.

        Logical reason:
        Student booking history may be displayed in profile or tests.
        """
        return self.__student_bookings

    def set_student_bookings(self, student_bookings):
        """
        Purpose:
        - Updates student booking dictionary.

        Logical reason:
        The value must stay a dictionary for organized storage.
        """
        if not isinstance(student_bookings, dict):
            raise TypeError("studentBookings must be dictionary.")
        self.__student_bookings = student_bookings

    def view_student_bookings(self):
        """
        Purpose:
        - Displays all bookings for the student.

        Logical reason:
        Students need to review their own reservations.
        """
        return self.__student_bookings


class Staff(User):
    """
    Staff represents a staff actor in the system.

    Purpose:
    - Stores staff-specific booking records.
    - Uses the shared User behavior from the parent class.

    Logical reason:
    Staff can book different facilities from students, so they need a separate child class.
    """
    def __init__(self, user_id, university_id, name, email, password):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        super().__init__(user_id, university_id, name, email, password, UserRole.STAFF)
        self.__staff_bookings = {}

    def get_staff_bookings(self):
        """
        Purpose:
        - Returns staff-specific booking records.

        Logical reason:
        Staff booking history may be displayed in profile or tests.
        """
        return self.__staff_bookings

    def set_staff_bookings(self, staff_bookings):
        """
        Purpose:
        - Updates staff booking dictionary.

        Logical reason:
        The value must stay a dictionary for organized storage.
        """
        if not isinstance(staff_bookings, dict):
            raise TypeError("staffBookings must be dictionary.")
        self.__staff_bookings = staff_bookings

    def view_staff_bookings(self):
        """
        Purpose:
        - Displays all bookings for the staff member.

        Logical reason:
        Staff need to review their own reservations.
        """
        return self.__staff_bookings


class Administrator(User):
    """
    Administrator represents the admin actor.

    Purpose:
    - Freezes and restores users.
    - Supports admin management actions in the GUI.

    Logical reason:
    Admin users have management permissions that normal students and staff should not have.
    """
    def __init__(self, user_id, university_id, name, email, password):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        super().__init__(user_id, university_id, name, email, password, UserRole.ADMIN)
        self.__admin_bookings_view = {}

    def get_admin_bookings_view(self):
        """
        Purpose:
        - Returns the admin booking view.

        Logical reason:
        Admin needs to monitor system activity.
        """
        return self.__admin_bookings_view

    def set_admin_bookings_view(self, admin_bookings_view):
        """
        Purpose:
        - Updates admin booking view dictionary.

        Logical reason:
        Admin dashboard data should stay organized.
        """
        if not isinstance(admin_bookings_view, dict):
            raise TypeError("adminBookingsView must be dictionary.")
        self.__admin_bookings_view = admin_bookings_view

    def freeze_user(self, user):
        """
        Purpose:
        - Freezes selected user.

        Logical reason:
        Admin can punish users who break rules.
        """
        user.set_status(UserStatus.FROZEN)

    def restore_user(self, user):
        """
        Purpose:
        - Restores user penalty credits and active status.

        Logical reason:
        Admin needs a way to give booking access back.
        """
        user.reset_penalty_credits()


class Facility:
    """
    Facility is the parent class for StudyRoom, SportsCourt, and EventHall.

    Purpose:
    - Stores facility details like ID, name, type, location, capacity, price, rules, time slots, booked slots, and closed slots.
    - Controls booking availability and slot management.

    Logical reason:
    All facility types share common data, so a parent class avoids repeated code.
    """
    def __init__(self, facility_id, facility_name, facility_type, location, capacity, price_per_hour, rules):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        self.__facility_id = facility_id
        self.__facility_name = facility_name
        self.__facility_type = facility_type
        self.__location = location
        self.__capacity = capacity
        self.__is_available = True
        self.__price_per_hour = price_per_hour
        self.__rules = rules
        self.__time_slots = [
            "09:00-10:00",
            "10:00-11:00",
            "11:00-12:00",
            "12:00-13:00",
            "14:00-15:00",
            "15:00-16:00"
        ]
        self.__booked_slots_by_date = {}
        self.__closed_slots_by_date = {}

    def get_facility_id(self):
        """
        Purpose:
        - Returns facility ID.

        Logical reason:
        The catalog uses this ID to find and store facilities.
        """
        return self.__facility_id

    def set_facility_id(self, facility_id):
        """
        Purpose:
        - Updates facility ID.

        Logical reason:
        Facility identity may need correction in admin management.
        """
        self.__facility_id = facility_id

    def get_facility_name(self):
        """
        Purpose:
        - Returns facility name.

        Logical reason:
        Users need readable facility names in the GUI.
        """
        return self.__facility_name

    def set_facility_name(self, facility_name):
        """
        Purpose:
        - Updates facility name.

        Logical reason:
        Admin may need to rename facilities.
        """
        self.__facility_name = facility_name

    def get_facility_type(self):
        """
        Purpose:
        - Returns facility type.

        Logical reason:
        Access rules depend on the facility type.
        """
        return self.__facility_type

    def set_facility_type(self, facility_type):
        """
        Purpose:
        - Updates facility type.

        Logical reason:
        Admin may need to correct facility classification.
        """
        self.__facility_type = facility_type

    def get_location(self):
        """
        Purpose:
        - Returns facility location.

        Logical reason:
        Users need to know where to go.
        """
        return self.__location

    def set_location(self, location):
        """
        Purpose:
        - Updates facility location.

        Logical reason:
        Facility details must stay accurate.
        """
        self.__location = location

    def get_capacity(self):
        """
        Purpose:
        - Returns facility capacity.

        Logical reason:
        Users and admins need to know maximum occupancy.
        """
        return self.__capacity

    def set_capacity(self, capacity):
        """
        Purpose:
        - Updates capacity after validation.

        Logical reason:
        Capacity cannot be zero or negative.
        """
        if capacity <= 0:
            raise ValueError("Capacity must be positive.")
        self.__capacity = capacity

    def get_is_available(self):
        """
        Purpose:
        - Returns whether the whole facility is available.

        Logical reason:
        Unavailable facilities should not appear as bookable.
        """
        return self.__is_available

    def set_is_available(self, is_available):
        """
        Purpose:
        - Opens or closes the whole facility.

        Logical reason:
        Admin needs to disable facilities during maintenance.
        """
        self.__is_available = is_available

    def get_price_per_hour(self):
        """
        Purpose:
        - Returns facility price.

        Logical reason:
        The payment screen must show and calculate cost.
        """
        return self.__price_per_hour

    def set_price_per_hour(self, price_per_hour):
        """
        Purpose:
        - Updates price after validation.

        Logical reason:
        Prices cannot be negative.
        """
        if price_per_hour < 0:
            raise ValueError("Price cannot be negative.")
        self.__price_per_hour = price_per_hour

    def get_rules(self):
        """
        Purpose:
        - Returns facility rules.

        Logical reason:
        Users should see rules before confirming bookings.
        """
        return self.__rules

    def set_rules(self, rules):
        """
        Purpose:
        - Updates facility rules.

        Logical reason:
        Admin can change policies or instructions.
        """
        self.__rules = rules

    def get_time_slots(self):
        """
        Purpose:
        - Returns all possible facility time slots.

        Logical reason:
        The GUI needs this list for booking choices.
        """
        return self.__time_slots

    def set_time_slots(self, time_slots):
        """
        Purpose:
        - Updates available time slot list.

        Logical reason:
        Admin may adjust operating hours.
        """
        self.__time_slots = time_slots

    def get_booked_slots_by_date(self):
        """
        Purpose:
        - Returns confirmed booked slots by date.

        Logical reason:
        The system must remember which slots are already taken.
        """
        return self.__booked_slots_by_date

    def set_booked_slots_by_date(self, booked_slots_by_date):
        """
        Purpose:
        - Restores booked slot data.

        Logical reason:
        This helps repair or load pickle data.
        """
        self.__booked_slots_by_date = booked_slots_by_date

    def get_closed_slots_by_date(self):
        """
        Purpose:
        - Returns admin-closed slots by date.

        Logical reason:
        Maintenance closures must be stored separately from booked slots.
        """
        return self.__closed_slots_by_date

    def set_closed_slots_by_date(self, closed_slots_by_date):
        """
        Purpose:
        - Restores closed slot data.

        Logical reason:
        This helps old saved data work with updated code.
        """
        self.__closed_slots_by_date = closed_slots_by_date

    def get_closed_slots(self, booking_date):
        """
        Purpose:
        - Returns closed slots for one date.

        Logical reason:
        The booking screen must block closed slots.
        """
        if booking_date not in self.__closed_slots_by_date:
            self.__closed_slots_by_date[booking_date] = []
        return self.__closed_slots_by_date[booking_date]

    def close_slot(self, booking_date, slot):
        """
        Purpose:
        - Closes one specific slot for a facility.

        Logical reason:
        Admin can block a time for maintenance without closing the whole facility.
        """
        if slot not in self.__time_slots:
            raise ValueError("Invalid time slot.")
        if booking_date not in self.__closed_slots_by_date:
            self.__closed_slots_by_date[booking_date] = []
        if slot not in self.__closed_slots_by_date[booking_date]:
            self.__closed_slots_by_date[booking_date].append(slot)

    def open_slot(self, booking_date, slot):
        """
        Purpose:
        - Reopens a previously closed slot.

        Logical reason:
        Admin needs to undo maintenance closures.
        """
        if booking_date in self.__closed_slots_by_date:
            if slot in self.__closed_slots_by_date[booking_date]:
                self.__closed_slots_by_date[booking_date].remove(slot)

    def get_available_slots(self, booking_date):
        """
        Purpose:
        - Returns slots that are not booked and not closed.

        Logical reason:
        Users should only be offered valid booking options.
        """
        if booking_date not in self.__booked_slots_by_date:
            self.__booked_slots_by_date[booking_date] = {}
        if booking_date not in self.__closed_slots_by_date:
            self.__closed_slots_by_date[booking_date] = []

        booked = self.__booked_slots_by_date[booking_date]
        closed = self.__closed_slots_by_date[booking_date]

        return [
            slot for slot in self.__time_slots
            if slot not in booked and slot not in closed
        ]

    def book_slot(self, booking_date, slot, booking_id):
        """
        Purpose:
        - Marks a slot as booked for a date.

        Logical reason:
        This prevents another user from booking the same facility slot.
        """
        if slot not in self.__time_slots:
            raise ValueError("Invalid time slot.")
        if booking_date not in self.__booked_slots_by_date:
            self.__booked_slots_by_date[booking_date] = {}
        if slot in self.__booked_slots_by_date[booking_date]:
            raise ValueError("This slot is already booked.")
        if slot in self.get_closed_slots(booking_date):
            raise ValueError("This slot is closed by admin.")
        self.__booked_slots_by_date[booking_date][slot] = booking_id

    def release_slot(self, booking_date, slot):
        """
        Purpose:
        - Frees a slot after cancellation, modification, or no-show.

        Logical reason:
        Released slots should become available for other users.
        """
        if booking_date in self.__booked_slots_by_date:
            if slot in self.__booked_slots_by_date[booking_date]:
                del self.__booked_slots_by_date[booking_date][slot]

    def calculate_cost(self, usage_type):
        """
        Purpose:
        - Calculates facility booking cost.

        Logical reason:
        Payment depends on the selected facility and usage type.
        """
        return self.__price_per_hour

    def display_details(self):
        """
        Purpose:
        - Returns a simple facility summary.

        Logical reason:
        The GUI and admin dashboard need readable facility info.
        """
        return f"{self.__facility_name} | {self.__location} | Capacity: {self.__capacity}"


class StudyRoom(Facility):
    """
    StudyRoom represents study room facilities.

    Purpose:
    - Creates student-only rooms with no booking fee.

    Logical reason:
    Study rooms have their own access rules, so they are represented as a child class of Facility.
    """
    def __init__(self, facility_id, room_code, location, capacity):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        super().__init__(
            facility_id,
            f"Study Room {room_code}",
            "STUDY_ROOM",
            location,
            capacity,
            0,
            "Rules: Students only. Keep the room clean. Arrive on time."
        )
        self.__room_code = room_code

    def get_room_code(self):
        """
        Purpose:
        - Returns study room code.

        Logical reason:
        Room codes help identify specific study rooms.
        """
        return self.__room_code

    def set_room_code(self, room_code):
        """
        Purpose:
        - Updates study room code.

        Logical reason:
        Admin may need to correct room labels.
        """
        self.__room_code = room_code


class SportsCourt(Facility):
    """
    SportsCourt represents sports facilities.

    Purpose:
    - Stores sport type and hourly price.
    - Supports premium student and premium staff bookings.

    Logical reason:
    Sports courts have different access and payment rules from study rooms and event halls.
    """
    def __init__(self, facility_id, sport_type, location, capacity, price_per_hour):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        super().__init__(
            facility_id,
            f"{sport_type} Court",
            "SPORTS_COURT",
            location,
            capacity,
            price_per_hour,
            "Rules: Sports shoes required. Respect the booked time."
        )
        self.__sport_type = sport_type

    def get_sport_type(self):
        """
        Purpose:
        - Returns sport type.

        Logical reason:
        Users need to know what court they are booking.
        """
        return self.__sport_type

    def set_sport_type(self, sport_type):
        """
        Purpose:
        - Updates sport type.

        Logical reason:
        Admin may need to correct court details.
        """
        self.__sport_type = sport_type


class EventHall(Facility):
    """
    EventHall represents large event spaces.

    Purpose:
    - Allows internal and external usage.
    - Makes internal use free and external use paid.

    Logical reason:
    EventHall has special pricing, so it overrides calculate_cost().
    """
    def __init__(self, facility_id, location, capacity, external_rate):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        super().__init__(
            facility_id,
            "Event Hall",
            "EVENT_HALL",
            location,
            capacity,
            external_rate,
            "Rules: Internal use is free. External use requires payment."
        )
        self.__internal_use_free = True
        self.__external_rate = external_rate

    def get_internal_use_free(self):
        """
        Purpose:
        - Returns whether internal EventHall use is free.

        Logical reason:
        This supports EventHall payment rules.
        """
        return self.__internal_use_free

    def set_internal_use_free(self, internal_use_free):
        """
        Purpose:
        - Updates internal-use payment rule.

        Logical reason:
        Admin may need to change hall policy.
        """
        self.__internal_use_free = internal_use_free

    def get_external_rate(self):
        """
        Purpose:
        - Returns external EventHall rate.

        Logical reason:
        External booking payment uses this value.
        """
        return self.__external_rate

    def set_external_rate(self, external_rate):
        """
        Purpose:
        - Updates external EventHall rate.

        Logical reason:
        Admin may adjust event hall pricing.
        """
        self.__external_rate = external_rate
        self.set_price_per_hour(external_rate)

    def calculate_cost(self, usage_type):
        """
        Purpose:
        - Calculates facility booking cost.

        Logical reason:
        Payment depends on the selected facility and usage type.
        """
        if usage_type == UsageType.INTERNAL.value:
            return 0
        return self.__external_rate


class TimeSlot:
    """
    TimeSlot represents a time period.

    Purpose:
    - Stores start and end time.
    - Tracks whether a slot is booked.

    Logical reason:
    This matches the UML TimeSlot class, even though the GUI mainly uses string slots.
    """
    def __init__(self, slot_id, start_time, end_time):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        self.__slot_id = slot_id
        self.__start_time = start_time
        self.__end_time = end_time
        self.__is_booked = False

    def get_slot_id(self):
        """
        Purpose:
        - Returns slot ID.

        Logical reason:
        Time slots need identifiers for tracking.
        """
        return self.__slot_id

    def set_slot_id(self, slot_id):
        """
        Purpose:
        - Updates slot ID.

        Logical reason:
        Slot labels may need correction.
        """
        self.__slot_id = slot_id

    def get_start_time(self):
        """
        Purpose:
        - Returns slot start time.

        Logical reason:
        Time conflicts depend on start and end times.
        """
        return self.__start_time

    def set_start_time(self, start_time):
        """
        Purpose:
        - Updates slot start time.

        Logical reason:
        Admin may adjust slot timing.
        """
        self.__start_time = start_time

    def get_end_time(self):
        """
        Purpose:
        - Returns slot end time.

        Logical reason:
        Time conflicts depend on start and end times.
        """
        return self.__end_time

    def set_end_time(self, end_time):
        """
        Purpose:
        - Updates slot end time.

        Logical reason:
        Admin may adjust slot timing.
        """
        self.__end_time = end_time

    def get_is_booked(self):
        """
        Purpose:
        - Returns whether this TimeSlot is booked.

        Logical reason:
        The UML requires tracking booked state.
        """
        return self.__is_booked

    def set_is_booked(self, is_booked):
        """
        Purpose:
        - Updates booked status.

        Logical reason:
        Booking confirmation and cancellation must update slot state.
        """
        self.__is_booked = is_booked


class Booking:
    """
    Booking represents a confirmed reservation.

    Purpose:
    - Connects User, Facility, slot, date, usage type, cost, and booking status.
    - Generates booking summaries for receipts and GUI display.

    Logical reason:
    Booking is the relationship object proving that a user reserved a facility at a specific time.
    """
    def __init__(self, booking_id, user, facility, slot, booking_date, usage_type, total_cost):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        self.__booking_id = booking_id
        self.__user = user
        self.__facility = facility
        self.__slot = slot
        self.__booking_date = booking_date
        self.__usage_type = usage_type
        self.__total_cost = total_cost
        self.__status = BookingStatus.CONFIRMED

    def get_booking_id(self):
        """
        Purpose:
        - Returns booking ID.

        Logical reason:
        Bookings need IDs for modification, cancellation, and admin reports.
        """
        return self.__booking_id

    def get_user(self):
        """
        Purpose:
        - Returns the user linked to the booking.

        Logical reason:
        The system must know who owns each booking.
        """
        return self.__user

    def get_facility(self):
        """
        Purpose:
        - Finds a facility by ID.

        Logical reason:
        Booking uses facility ID selected from GUI.
        """
        return self.__facility

    def get_slot(self):
        """
        Purpose:
        - Returns the booked slot.

        Logical reason:
        Reports, conflicts, and modifications need the booking time.
        """
        return self.__slot

    def set_slot(self, slot):
        """
        Purpose:
        - Updates the booking slot.

        Logical reason:
        Users can modify their booking time.
        """
        self.__slot = slot

    def get_booking_date(self):
        """
        Purpose:
        - Returns the booking date.

        Logical reason:
        Reports and conflict checks need the date.
        """
        return self.__booking_date

    def set_booking_date(self, booking_date):
        """
        Purpose:
        - Updates the booking date.

        Logical reason:
        Users can modify their booking date.
        """
        self.__booking_date = booking_date

    def get_usage_type(self):
        """
        Purpose:
        - Returns internal or external usage.

        Logical reason:
        EventHall cost depends on usage type.
        """
        return self.__usage_type

    def get_total_cost(self):
        """
        Purpose:
        - Returns booking cost.

        Logical reason:
        Payments and receipts need this amount.
        """
        return self.__total_cost

    def get_status(self):
        """
        Purpose:
        - Returns current object status.

        Logical reason:
        The system needs status for users, bookings, or payments.
        """
        return self.__status

    def set_status(self, status):
        """
        Purpose:
        - Updates status safely.

        Logical reason:
        Object state changes during booking, payment, cancellation, and freezing.
        """
        self.__status = status

    def cancel_booking(self):
        """
        Purpose:
        - Marks a booking as cancelled.

        Logical reason:
        Cancelled bookings should no longer count as active.
        """
        self.__status = BookingStatus.CANCELLED

    def mark_no_show(self):
        """
        Purpose:
        - Marks booking as no-show.

        Logical reason:
        No-shows reduce penalty credits and may freeze the user.
        """
        self.__status = BookingStatus.NO_SHOW

    def generate_booking_summary(self):
        """
        Purpose:
        - Creates readable booking details.

        Logical reason:
        Receipts, tests, and GUI screens need a clear booking summary.
        """
        return (
            f"Booking ID: {self.__booking_id}\n"
            f"User: {self.__user.get_name()}\n"
            f"Facility: {self.__facility.get_facility_name()}\n"
            f"Facility Type: {self.__facility.get_facility_type()}\n"
            f"Location: {self.__facility.get_location()}\n"
            f"Date: {self.__booking_date}\n"
            f"Time Slot: {self.__slot}\n"
            f"Usage Type: {self.__usage_type}\n"
            f"Cost: {self.__total_cost} AED\n"
            f"Status: {self.__status.value}\n"
            f"{self.__facility.get_rules()}"
        )


class Payment:
    """
    Payment represents money handling for bookings or upgrades.

    Purpose:
    - Stores amount, date, status, and payment purpose.
    - Processes paid and free transactions.

    Logical reason:
    The system needs payment records to create receipts and show financial proof.
    """
    def __init__(self, payment_id, amount, purpose):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        self.__payment_id = payment_id
        self.__amount = amount
        self.__payment_date = str(date.today())
        self.__status = PaymentStatus.PENDING if amount > 0 else PaymentStatus.NOT_REQUIRED
        self.__purpose = purpose

    def get_payment_id(self):
        """
        Purpose:
        - Returns payment ID.

        Logical reason:
        Payments need IDs for storage and receipts.
        """
        return self.__payment_id

    def get_amount(self):
        """
        Purpose:
        - Returns payment amount.

        Logical reason:
        Receipts and payment records need the amount.
        """
        return self.__amount

    def get_payment_date(self):
        """
        Purpose:
        - Returns payment date.

        Logical reason:
        Receipts need transaction date.
        """
        return self.__payment_date

    def get_status(self):
        """
        Purpose:
        - Returns current object status.

        Logical reason:
        The system needs status for users, bookings, or payments.
        """
        return self.__status

    def set_status(self, status):
        """
        Purpose:
        - Updates status safely.

        Logical reason:
        Object state changes during booking, payment, cancellation, and freezing.
        """
        self.__status = status

    def get_purpose(self):
        """
        Purpose:
        - Returns payment purpose.

        Logical reason:
        The system should know if the payment was for booking or upgrade.
        """
        return self.__purpose

    def process_payment(self):
        """
        Purpose:
        - Completes payment processing.

        Logical reason:
        Paid bookings should become PAID, while free bookings should be NOT_REQUIRED.
        """
        if self.__amount == 0:
            self.__status = PaymentStatus.NOT_REQUIRED
        else:
            self.__status = PaymentStatus.PAID
        return True


class Receipt:
    """
    Receipt represents proof of a completed booking/payment.

    Purpose:
    - Combines booking and payment information into a readable receipt.

    Logical reason:
    Users need confirmation that their booking and payment were completed.
    """
    def __init__(self, receipt_id, booking, payment):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        self.__receipt_id = receipt_id
        self.__booking = booking
        self.__payment = payment
        self.__issue_date = str(date.today())

    def get_receipt_id(self):
        """
        Purpose:
        - Returns receipt ID.

        Logical reason:
        Receipts need IDs for storage.
        """
        return self.__receipt_id

    def generate_receipt(self):
        """
        Purpose:
        - Creates receipt text.

        Logical reason:
        Users need proof of booking and payment.
        """
        return (
            "BOOKING RECEIPT\n"
            "----------------------\n"
            f"Receipt ID: {self.__receipt_id}\n"
            f"Issue Date: {self.__issue_date}\n"
            f"Payment Status: {self.__payment.get_status().value}\n"
            f"Payment Amount: {self.__payment.get_amount()} AED\n\n"
            f"{self.__booking.generate_booking_summary()}"
        )


class Notification:
    """
    Notification represents a system message.

    Purpose:
    - Stores booking confirmations, freeze warnings, and system updates.

    Logical reason:
    Users must be informed when important actions happen.
    """
    def __init__(self, notification_id, message):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        self.__notification_id = notification_id
        self.__message = message
        self.__send_date = str(date.today())
        self.__is_read = False

    def get_notification_id(self):
        """
        Purpose:
        - Returns notification ID.

        Logical reason:
        Notifications need IDs for storage.
        """
        return self.__notification_id

    def get_message(self):
        """
        Purpose:
        - Returns notification message.

        Logical reason:
        The GUI displays this text to users.
        """
        return self.__message

    def get_send_date(self):
        """
        Purpose:
        - Returns notification date.

        Logical reason:
        Users and admins may need message history.
        """
        return self.__send_date

    def get_is_read(self):
        """
        Purpose:
        - Returns read/unread state.

        Logical reason:
        The GUI can separate unread notifications.
        """
        return self.__is_read

    def mark_as_read(self):
        """
        Purpose:
        - Marks notification as read.

        Logical reason:
        Users should know which notifications are new.
        """
        self.__is_read = True


class AccessPolicy:
    """
    AccessPolicy controls booking permission rules.

    Purpose:
    - Checks role, access type, and facility type.

    Logical reason:
    Putting permission rules in one class makes the code easier to maintain and keeps the business logic clear.
    """
    def can_book(self, user, facility):
        """
        Purpose:
        - Checks whether a user can book a facility.

        Logical reason:
        This enforces role and access rules before booking is created.
        """
        role = user.get_role()
        access = user.get_access_type()
        facility_type = facility.get_facility_type()

        if role == UserRole.ADMIN:
            return False

        if facility_type == "STUDY_ROOM":
            return role == UserRole.STUDENT

        if facility_type == "SPORTS_COURT":
            return access == AccessType.PREMIUM and role in [UserRole.STUDENT, UserRole.STAFF]

        if facility_type == "EVENT_HALL":
            if role == UserRole.STAFF:
                return True
            return role == UserRole.STUDENT and access == AccessType.PREMIUM

        return False


class FacilityCatalog:
    """
    FacilityCatalog stores all facility objects.

    Purpose:
    - Adds, removes, finds, and displays facilities.

    Logical reason:
    BookingSystem needs one organized place to manage facility data.
    """
    def __init__(self):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        self.__facilities = {}

    def get_facilities(self):
        """
        Purpose:
        - Returns all facilities.

        Logical reason:
        Booking and admin screens need the catalog data.
        """
        return self.__facilities

    def set_facilities(self, facilities):
        """
        Purpose:
        - Replaces facility dictionary.

        Logical reason:
        Pickle loading may restore a saved catalog.
        """
        self.__facilities = facilities

    def add_facility(self, facility):
        """
        Purpose:
        - Adds a facility to the catalog.

        Logical reason:
        The system needs a central place to store facilities.
        """
        self.__facilities[facility.get_facility_id()] = facility

    def remove_facility(self, facility_id):
        """
        Purpose:
        - Removes a facility from the catalog.

        Logical reason:
        Admin may remove unavailable or old facilities.
        """
        if facility_id in self.__facilities:
            del self.__facilities[facility_id]

    def get_facility(self, facility_id):
        """
        Purpose:
        - Finds a facility by ID.

        Logical reason:
        Booking uses facility ID selected from GUI.
        """
        return self.__facilities.get(facility_id)

    def display_all_facilities(self):
        """
        Purpose:
        - Returns all facilities for display.

        Logical reason:
        Admin dashboard needs to show facility data.
        """
        return self.__facilities


class BookingSystem:
    """
    BookingSystem is the main controller for the backend system.

    Purpose:
    - Stores users, bookings, payments, receipts, notifications, announcements, and facilities.
    - Handles registration, login, booking, payment, penalties, admin actions, and pickle persistence.

    Logical reason:
    The full system needs one central controller to connect all classes together.
    """
    def __init__(self):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        self.__system_name = "Smart Campus Facility Booking System"
        self.__support_email = "support@campus.ae"
        self.__users = {}
        self.__bookings = {}
        self.__payments = {}
        self.__receipts = {}
        self.__notifications = {}
        self.__announcements = []
        self.__facility_catalog = FacilityCatalog()
        self.__access_policy = AccessPolicy()
        self.__data_file_name = "smart_campus_data.dat"
        self.__session_file_name = "smart_campus_session.dat"
        self.__current_user_id = ""
        self.__remember_login = False
        self.__next_booking_number = 1

        self.load_data_from_file(self.__data_file_name)
        self.repair_loaded_data()

        if not self.__facility_catalog.get_facilities():
            self.create_default_facilities()

        self.load_session()

    def get_system_name(self):
        """
        Purpose:
        - Returns system name.

        Logical reason:
        The GUI can display it as the application title.
        """
        return self.__system_name

    def get_support_email(self):
        """
        Purpose:
        - Returns support email.

        Logical reason:
        Users need contact information for help.
        """
        return self.__support_email

    def get_users(self):
        """
        Purpose:
        - Returns all registered users.

        Logical reason:
        Admin dashboard needs user records.
        """
        return self.__users

    def get_bookings(self):
        """
        Purpose:
        - Returns all bookings.

        Logical reason:
        Users and admins need booking records.
        """
        return self.__bookings

    def get_payments(self):
        """
        Purpose:
        - Returns all payments.

        Logical reason:
        Payment history needs storage and display.
        """
        return self.__payments

    def get_receipts(self):
        """
        Purpose:
        - Returns all receipts.

        Logical reason:
        Receipts must be retrievable after booking.
        """
        return self.__receipts

    def get_notifications(self):
        """
        Purpose:
        - Returns notifications.

        Logical reason:
        Notifications must be saved and displayed.
        """
        return self.__notifications

    def get_announcements(self):
        """
        Purpose:
        - Returns announcements.

        Logical reason:
        Users need to see maintenance messages.
        """
        return self.__announcements

    def get_facility_catalog(self):
        """
        Purpose:
        - Returns catalog object.

        Logical reason:
        Other parts of the system need access to facilities.
        """
        return self.__facility_catalog

    def get_current_user_id(self):
        """
        Purpose:
        - Returns logged-in user ID.

        Logical reason:
        Booking and cancellation depend on current user.
        """
        return self.__current_user_id

    def get_current_user(self):
        """
        Purpose:
        - Returns logged-in user object.

        Logical reason:
        Screens and booking logic depend on the active user.
        """
        if self.__current_user_id in self.__users:
            return self.__users[self.__current_user_id]
        return None

    def create_default_facilities(self):
        """
        Purpose:
        - Creates default facilities when no saved data exists.

        Logical reason:
        The system needs facilities available immediately after first launch.
        """
        self.__facility_catalog.add_facility(StudyRoom("SR-A", "A", "Library - Floor 1", 6))
        self.__facility_catalog.add_facility(StudyRoom("SR-B", "B", "Library - Floor 2", 4))
        self.__facility_catalog.add_facility(SportsCourt("SC-B", "Basketball", "Sports Complex", 10, 20))
        self.__facility_catalog.add_facility(SportsCourt("SC-T", "Tennis", "Sports Complex", 4, 15))
        self.__facility_catalog.add_facility(EventHall("EH-1", "Main Building", 50, 100))
        self.__facility_catalog.add_facility(EventHall("EH-2", "Conference Building", 100, 150))

    def repair_loaded_data(self):
        """
        Purpose:
        - Adds missing attributes to old loaded objects.

        Logical reason:
        Saved pickle data from older code versions may not have new attributes.
        """
        for user in self.__users.values():
            if not hasattr(user, "_User__penalty_credits"):
                user.set_penalty_credits(3)

        for facility in self.__facility_catalog.get_facilities().values():
            if not hasattr(facility, "_Facility__booked_slots_by_date"):
                facility.set_booked_slots_by_date({})
            if not hasattr(facility, "_Facility__closed_slots_by_date"):
                facility.set_closed_slots_by_date({})

    def save_data_to_file(self, file_name):
        """
        Purpose:
        - Saves system data using pickle.

        Logical reason:
        Data must persist after closing the program.
        """
        try:
            with open(file_name, "wb") as file:
                pickle.dump({
                    "users": self.__users,
                    "bookings": self.__bookings,
                    "payments": self.__payments,
                    "receipts": self.__receipts,
                    "notifications": self.__notifications,
                    "announcements": self.__announcements,
                    "facility_catalog": self.__facility_catalog,
                    "next_booking_number": self.__next_booking_number
                }, file)
            return True
        except (pickle.PickleError, OSError) as error:
            raise OSError(f"Saving failed: {error}")

    def load_data_from_file(self, file_name):
        """
        Purpose:
        - Loads saved system data.

        Logical reason:
        Users, bookings, and admin changes should remain after restarting.
        """
        try:
            if os.path.exists(file_name):
                with open(file_name, "rb") as file:
                    data = pickle.load(file)

                self.__users = data.get("users", {})
                self.__bookings = data.get("bookings", {})
                self.__payments = data.get("payments", {})
                self.__receipts = data.get("receipts", {})
                self.__notifications = data.get("notifications", {})
                self.__announcements = data.get("announcements", [])
                self.__facility_catalog = data.get("facility_catalog", FacilityCatalog())
                self.__next_booking_number = data.get("next_booking_number", 1)

            return self
        except (EOFError, pickle.UnpicklingError, OSError, AttributeError):
            self.__users = {}
            self.__bookings = {}
            self.__payments = {}
            self.__receipts = {}
            self.__notifications = {}
            self.__announcements = []
            self.__facility_catalog = FacilityCatalog()
            self.__next_booking_number = 1
            return self

    def save_session(self, user_id):
        """
        Purpose:
        - Saves remembered login session.

        Logical reason:
        The remember-me checkbox needs persistent login data.
        """
        try:
            with open(self.__session_file_name, "wb") as file:
                pickle.dump({"current_user_id": user_id}, file)
            return True
        except OSError:
            return False

    def load_session(self):
        """
        Purpose:
        - Loads remembered login session.

        Logical reason:
        Users can return without logging in again.
        """
        try:
            if os.path.exists(self.__session_file_name):
                with open(self.__session_file_name, "rb") as file:
                    data = pickle.load(file)

                user_id = data.get("current_user_id", "")
                if user_id in self.__users:
                    self.__current_user_id = user_id
                    return self.__users[user_id]
            return None
        except (EOFError, pickle.UnpicklingError, OSError, AttributeError):
            return None

    def logout(self):
        """
        Purpose:
        - Clears current session.

        Logical reason:
        Users need to safely exit their account.
        """
        self.__current_user_id = ""
        if os.path.exists(self.__session_file_name):
            os.remove(self.__session_file_name)

    def generate_user_id(self, role, university_id):
        """
        Purpose:
        - Builds login ID from role and university/work ID.

        Logical reason:
        This makes IDs consistent, like STU20240001 or ADM1000.
        """
        if role == UserRole.STUDENT:
            return "STU" + university_id
        if role == UserRole.STAFF:
            return "STF" + university_id
        if role == UserRole.ADMIN:
            return "ADM" + university_id
        raise ValueError("Invalid role.")

    def register_user(self, role, university_id, name, email, password):
        """
        Purpose:
        - Creates and stores a new user account.

        Logical reason:
        The system needs registration before login and booking.
        """
        user_id = self.generate_user_id(role, university_id)

        if user_id in self.__users:
            raise ValueError("This account already exists.")

        if role == UserRole.STUDENT:
            user = Student(user_id, university_id, name, email, password)
        elif role == UserRole.STAFF:
            user = Staff(user_id, university_id, name, email, password)
        elif role == UserRole.ADMIN:
            user = Administrator(user_id, university_id, name, email, password)
        else:
            raise ValueError("Invalid role.")

        self.__users[user_id] = user
        self.save_data_to_file(self.__data_file_name)
        return user

    def validate_login(self, user_id, password, remember_login=False):
        """
        Purpose:
        - Checks user ID and password.

        Logical reason:
        Only registered users should access the dashboard.
        """
        if user_id not in self.__users:
            raise ValueError(
                "User ID not found.\n\n"
                "Example:\n"
                "Student 20240001 becomes STU20240001\n"
                "Staff 8899 becomes STF8899\n"
                "Admin 1000 becomes ADM1000"
            )

        user = self.__users[user_id]

        if not user.login(user_id, password):
            raise PermissionError("Incorrect password.")

        self.__current_user_id = user_id
        self.__remember_login = remember_login

        if remember_login:
            self.save_session(user_id)

        return user

    def validate_weekday_booking(self, selected_date):
        """
        Purpose:
        - Blocks weekend bookings.

        Logical reason:
        The business rule only allows Monday to Friday reservations.
        """
        booking_day = datetime.strptime(selected_date, "%Y-%m-%d").date()
        if booking_day.weekday() >= 5:
            raise ValueError("Bookings are only allowed from Monday to Friday.")
        return True

    def get_allowed_facilities(self, user):
        """
        Purpose:
        - Returns facilities the current user can book.

        Logical reason:
        The GUI should only show valid options to reduce errors.
        """
        allowed = {}
        for facility_id, facility in self.__facility_catalog.get_facilities().items():
            if facility.get_is_available() and self.__access_policy.can_book(user, facility):
                allowed[facility_id] = facility
        return allowed

    def count_user_bookings_on_date(self, user, selected_date):
        """
        Purpose:
        - Counts user bookings on one date.

        Logical reason:
        STANDARD users only have 1 booking credit per day.
        """
        count = 0
        for booking in self.__bookings.values():
            if (
                booking.get_user().get_user_id() == user.get_user_id()
                and booking.get_booking_date() == selected_date
                and booking.get_status() == BookingStatus.CONFIRMED
            ):
                count += 1
        return count

    def count_user_facility_type_bookings_on_date(self, user, facility_type, selected_date):
        """
        Purpose:
        - Counts premium user bookings by facility type.

        Logical reason:
        PREMIUM users have 3 credits per facility type per day.
        """
        count = 0
        for booking in self.__bookings.values():
            if (
                booking.get_user().get_user_id() == user.get_user_id()
                and booking.get_facility().get_facility_type() == facility_type
                and booking.get_booking_date() == selected_date
                and booking.get_status() == BookingStatus.CONFIRMED
            ):
                count += 1
        return count

    def get_daily_credit_text(self, user, selected_date=None):
        """
        Purpose:
        - Creates readable credit information.

        Logical reason:
        The dashboard should show remaining booking and penalty credits.
        """
        if selected_date is None:
            selected_date = str(date.today())

        if user.get_access_type() == AccessType.STANDARD:
            used = self.count_user_bookings_on_date(user, selected_date)
            return f"Booking Credit: {max(0, 1 - used)} / 1 | Penalty Credit: {user.get_penalty_credits()} / 3"

        allowed_types = set()
        for facility in self.get_allowed_facilities(user).values():
            allowed_types.add(facility.get_facility_type())

        credit_lines = []
        for facility_type in allowed_types:
            used = self.count_user_facility_type_bookings_on_date(user, facility_type, selected_date)
            credit_lines.append(f"{facility_type}: {max(0, 3 - used)} / 3")

        return " | ".join(credit_lines) + f" | Penalty Credit: {user.get_penalty_credits()} / 3"

    def check_booking_credit(self, user, facility, selected_date):
        """
        Purpose:
        - Enforces booking credit limits.

        Logical reason:
        Users should not exceed their daily booking allowance.
        """
        if user.get_access_type() == AccessType.STANDARD:
            if self.count_user_bookings_on_date(user, selected_date) >= 1:
                raise PermissionError("STANDARD access allows only 1 booking per day.")

        if user.get_access_type() == AccessType.PREMIUM:
            used = self.count_user_facility_type_bookings_on_date(
                user,
                facility.get_facility_type(),
                selected_date
            )
            if used >= 3:
                raise PermissionError(
                    f"PREMIUM access allows only 3 bookings per {facility.get_facility_type()} per day."
                )

    def check_booking_conflict(self, user, slot, selected_date):
        """
        Purpose:
        - Prevents same-time double booking.

        Logical reason:
        A user cannot reserve two facilities at the same time.
        """
        for booking in self.__bookings.values():
            if (
                booking.get_user().get_user_id() == user.get_user_id()
                and booking.get_booking_date() == selected_date
                and booking.get_slot() == slot
                and booking.get_status() == BookingStatus.CONFIRMED
            ):
                return True
        return False

    def auto_freeze_if_needed(self, user):
        """
        Purpose:
        - Freezes user automatically after penalty credits reach 0.

        Logical reason:
        Three violations should remove booking privileges.
        """
        if user.get_penalty_credits() <= 0:
            user.set_status(UserStatus.FROZEN)
            notification_id = "N-FREEZE-" + user.get_user_id()
            self.__notifications[notification_id] = Notification(
                notification_id,
                "Your account has been automatically frozen because you reached 3 violations."
            )

    def record_cancellation_violation(self, user):
        """
        Purpose:
        - Applies penalty after cancellation.

        Logical reason:
        Repeated cancellations should affect user status.
        """
        user.decrease_penalty_credit()
        self.auto_freeze_if_needed(user)
        self.save_data_to_file(self.__data_file_name)

    def record_no_show_violation(self, user):
        """
        Purpose:
        - Applies penalty after no-show.

        Logical reason:
        No-shows waste facility slots and should be punished.
        """
        user.decrease_penalty_credit()
        self.auto_freeze_if_needed(user)
        self.save_data_to_file(self.__data_file_name)

    def create_booking_after_payment(self, facility_id, slot, selected_date, usage_type):
        """
        Purpose:
        - Creates booking, payment, receipt, notification, and saves data.

        Logical reason:
        Booking should only be finalized after all rules and payment logic pass.
        """
        user = self.get_current_user()
        facility = self.__facility_catalog.get_facility(facility_id)

        if user.get_status() == UserStatus.FROZEN:
            raise PermissionError("Your account is FROZEN. You cannot book.")

        if not facility.get_is_available():
            raise PermissionError("Facility is unavailable.")

        datetime.strptime(selected_date, "%Y-%m-%d")
        self.validate_weekday_booking(selected_date)

        if datetime.strptime(selected_date, "%Y-%m-%d").date() < date.today():
            raise ValueError("Booking date cannot be in the past.")

        if not self.__access_policy.can_book(user, facility):
            raise PermissionError("You cannot book this facility.")

        if slot not in facility.get_available_slots(selected_date):
            raise ValueError("This slot is already booked or closed.")

        if self.check_booking_conflict(user, slot, selected_date):
            raise ValueError("You already have a booking at this time.")

        self.check_booking_credit(user, facility, selected_date)

        total_cost = facility.calculate_cost(usage_type)

        booking_id = "B" + str(self.__next_booking_number).zfill(4)
        self.__next_booking_number += 1

        booking = Booking(booking_id, user, facility, slot, selected_date, usage_type, total_cost)
        self.__bookings[booking_id] = booking
        facility.book_slot(selected_date, slot, booking_id)

        payment = Payment("P" + booking_id, total_cost, PaymentPurpose.BOOKING_FEE)
        payment.process_payment()
        self.__payments[payment.get_payment_id()] = payment

        receipt = Receipt("R" + booking_id, booking, payment)
        self.__receipts[receipt.get_receipt_id()] = receipt

        notification = Notification("N" + booking_id, f"Booking {booking_id} confirmed.")
        self.__notifications[notification.get_notification_id()] = notification

        self.save_data_to_file(self.__data_file_name)
        return booking, receipt

    def cancel_booking(self, booking_id):
        """
        Purpose:
        - Marks a booking as cancelled.

        Logical reason:
        Cancelled bookings should no longer count as active.
        """
        if booking_id not in self.__bookings:
            raise ValueError("Booking not found.")

        booking = self.__bookings[booking_id]

        if booking.get_user().get_user_id() != self.__current_user_id:
            raise PermissionError("You can only cancel your own booking.")

        booking.get_facility().release_slot(booking.get_booking_date(), booking.get_slot())
        booking.cancel_booking()
        self.record_cancellation_violation(booking.get_user())
        self.save_data_to_file(self.__data_file_name)

    def modify_booking(self, booking_id, new_slot, new_date):
        """
        Purpose:
        - Changes booking slot and date.

        Logical reason:
        Users need flexibility to update bookings while keeping availability correct.
        """
        if booking_id not in self.__bookings:
            raise ValueError("Booking not found.")

        booking = self.__bookings[booking_id]

        if booking.get_user().get_user_id() != self.__current_user_id:
            raise PermissionError("You can only modify your own booking.")

        self.validate_weekday_booking(new_date)

        facility = booking.get_facility()

        if new_slot not in facility.get_available_slots(new_date):
            raise ValueError("New slot is not available.")

        datetime.strptime(new_date, "%Y-%m-%d")

        facility.release_slot(booking.get_booking_date(), booking.get_slot())
        booking.set_slot(new_slot)
        booking.set_booking_date(new_date)
        facility.book_slot(new_date, new_slot, booking_id)

        self.save_data_to_file(self.__data_file_name)

    def mark_booking_no_show(self, booking_id):
        """
        Purpose:
        - Marks a booking as no-show and applies penalty.

        Logical reason:
        Admin must be able to punish missed bookings.
        """
        if booking_id not in self.__bookings:
            raise ValueError("Booking not found.")

        booking = self.__bookings[booking_id]
        user = booking.get_user()

        booking.get_facility().release_slot(booking.get_booking_date(), booking.get_slot())
        booking.mark_no_show()

        self.record_no_show_violation(user)
        self.save_data_to_file(self.__data_file_name)

    def process_upgrade_payment(self):
        """
        Purpose:
        - Processes premium upgrade payment.

        Logical reason:
        Users pay 30 AED before receiving PREMIUM access.
        """
        user = self.get_current_user()

        if user.get_access_type() == AccessType.PREMIUM:
            raise ValueError("User already has PREMIUM access.")

        payment = Payment("P-UP-" + user.get_user_id(), 30, PaymentPurpose.ACCESS_UPGRADE)
        payment.process_payment()
        self.__payments[payment.get_payment_id()] = payment

        user.set_access_type(AccessType.PREMIUM)
        self.save_data_to_file(self.__data_file_name)

    def downgrade_current_user(self):
        """
        Purpose:
        - Downgrades current user to STANDARD.

        Logical reason:
        Users can return to standard access without payment.
        """
        user = self.get_current_user()
        user.set_access_type(AccessType.STANDARD)
        self.save_data_to_file(self.__data_file_name)

    def freeze_user(self, user_id):
        """
        Purpose:
        - Freezes selected user.

        Logical reason:
        Admin can punish users who break rules.
        """
        if user_id not in self.__users:
            raise ValueError("User not found.")
        self.__users[user_id].set_status(UserStatus.FROZEN)
        self.save_data_to_file(self.__data_file_name)

    def unfreeze_user(self, user_id):
        """
        Purpose:
        - Restores selected user account.

        Logical reason:
        Admin can give access back after punishment.
        """
        if user_id not in self.__users:
            raise ValueError("User not found.")
        self.__users[user_id].reset_penalty_credits()
        self.save_data_to_file(self.__data_file_name)

    def upgrade_user_by_admin(self, user_id):
        """
        Purpose:
        - Admin upgrades a user to PREMIUM.

        Logical reason:
        Admin can help when upgrade issues happen.
        """
        if user_id not in self.__users:
            raise ValueError("User not found.")
        self.__users[user_id].set_access_type(AccessType.PREMIUM)
        self.save_data_to_file(self.__data_file_name)

    def downgrade_user_by_admin(self, user_id):
        """
        Purpose:
        - Admin downgrades a user to STANDARD.

        Logical reason:
        Admin can correct or manage access levels.
        """
        if user_id not in self.__users:
            raise ValueError("User not found.")
        self.__users[user_id].set_access_type(AccessType.STANDARD)
        self.save_data_to_file(self.__data_file_name)

    def close_facility_slot(self, facility_id, selected_date, slot):
        """
        Purpose:
        - Closes a specific facility slot.

        Logical reason:
        Admin can block bookings during maintenance.
        """
        facility = self.__facility_catalog.get_facility(facility_id)
        if facility is None:
            raise ValueError("Facility not found.")
        facility.close_slot(selected_date, slot)
        self.save_data_to_file(self.__data_file_name)

    def open_facility_slot(self, facility_id, selected_date, slot):
        """
        Purpose:
        - Reopens a closed slot.

        Logical reason:
        Admin can make a slot available again after maintenance.
        """
        facility = self.__facility_catalog.get_facility(facility_id)
        if facility is None:
            raise ValueError("Facility not found.")
        facility.open_slot(selected_date, slot)
        self.save_data_to_file(self.__data_file_name)

    def add_announcement(self, message):
        """
        Purpose:
        - Adds announcement with timestamp.

        Logical reason:
        Users need to see system updates and maintenance messages.
        """
        if not message:
            raise ValueError("Announcement cannot be empty.")
        stamp = datetime.now().strftime("%Y-%m-%d %H:%M")
        self.__announcements.append(f"{stamp}: {message}")
        self.save_data_to_file(self.__data_file_name)

    def get_bookings_by_date(self, selected_date):
        """
        Purpose:
        - Returns bookings for one selected date.

        Logical reason:
        Admin monitoring depends on daily booking reports.
        """
        return {
            booking_id: booking
            for booking_id, booking in self.__bookings.items()
            if booking.get_booking_date() == selected_date
        }


class CalendarPopup:
    """
    CalendarPopup creates a small calendar selection window.

    Purpose:
    - Lets users select dates instead of typing them manually.
    - Refreshes related data after date selection when needed.

    Logical reason:
    Date format errors can break bookings, so a calendar improves usability and reduces mistakes.
    """
    def __init__(self, parent, target_entry, refresh_function=None):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        self.__target_entry = target_entry
        self.__refresh_function = refresh_function
        self.__today = date.today()
        self.__year = self.__today.year
        self.__month = self.__today.month
        self.__window = tk.Toplevel(parent)
        self.__window.title("Select Date")
        self.__window.geometry("330x300")
        self.draw_calendar()

    def draw_calendar(self):
        """
        Purpose:
        - Draws calendar buttons for the selected month.

        Logical reason:
        Users can choose dates visually instead of typing.
        """
        for widget in self.__window.winfo_children():
            widget.destroy()

        header = tk.Frame(self.__window)
        header.pack(pady=10)

        tk.Button(header, text="<", command=self.previous_month).grid(row=0, column=0, padx=5)
        tk.Label(
            header,
            text=f"{calendar.month_name[self.__month]} {self.__year}",
            font=("Arial", 12, "bold")
        ).grid(row=0, column=1, padx=20)
        tk.Button(header, text=">", command=self.next_month).grid(row=0, column=2, padx=5)

        days_frame = tk.Frame(self.__window)
        days_frame.pack()

        for index, day_name in enumerate(["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]):
            tk.Label(days_frame, text=day_name, width=5, font=("Arial", 9, "bold")).grid(row=0, column=index)

        month_days = calendar.monthcalendar(self.__year, self.__month)

        for row_number, week in enumerate(month_days, start=1):
            for column_number, day_number in enumerate(week):
                if day_number == 0:
                    tk.Label(days_frame, text="", width=5).grid(row=row_number, column=column_number)
                else:
                    tk.Button(
                        days_frame,
                        text=str(day_number),
                        width=5,
                        command=lambda selected_day=day_number: self.select_date(selected_day)
                    ).grid(row=row_number, column=column_number, padx=1, pady=1)

    def previous_month(self):
        """
        Purpose:
        - Moves calendar to previous month.

        Logical reason:
        Users may need to select dates in another month.
        """
        self.__month -= 1
        if self.__month == 0:
            self.__month = 12
            self.__year -= 1
        self.draw_calendar()

    def next_month(self):
        """
        Purpose:
        - Moves calendar to next month.

        Logical reason:
        Users may need to select future dates.
        """
        self.__month += 1
        if self.__month == 13:
            self.__month = 1
            self.__year += 1
        self.draw_calendar()

    def select_date(self, day_number):
        """
        Purpose:
        - Inserts selected date into the target entry.

        Logical reason:
        This connects the calendar popup to booking/admin date fields.
        """
        selected_date = date(self.__year, self.__month, day_number)
        self.__target_entry.delete(0, tk.END)
        self.__target_entry.insert(0, selected_date.strftime("%Y-%m-%d"))

        if self.__refresh_function:
            self.__refresh_function()

        self.__window.destroy()


class SmartCampusGUI:
    """
    SmartCampusGUI controls the Tkinter user interface.

    Purpose:
    - Shows login, registration, dashboard, booking, payment, user booking, and admin screens.
    - Connects button actions to BookingSystem backend logic.

    Logical reason:
    The GUI gives users and admins an easier way to interact with the system instead of using console commands.
    """
    def __init__(self, root):
        """
        Constructor purpose:
        - Creates a new object and prepares its starting data.

        Logical reason:
        Each object must begin with valid attributes before the system can use it.
        """
        self.__root = root
        self.__root.title("Smart Campus Facility Booking System")
        self.__root.geometry("1080x780")
        self.__system = BookingSystem()

        if self.__system.get_current_user() is not None:
            self.show_main_dashboard()
        else:
            self.show_login_screen()

    def clear_window(self):
        """
        Purpose:
        - Removes all widgets from the current GUI screen.

        Logical reason:
        Tkinter screens are rebuilt by clearing old widgets first.
        """
        for widget in self.__root.winfo_children():
            widget.destroy()

    def show_login_screen(self):
        """
        Purpose:
        - Displays login form.

        Logical reason:
        Users must log in before using the system.
        """
        self.clear_window()

        tk.Label(self.__root, text="Smart Campus Facility Booking System", font=("Arial", 22, "bold")).pack(pady=18)
        tk.Label(
            self.__root,
            text="Login Hint: Student 20240001 = STU20240001 | Staff 8899 = STF8899 | Admin 1000 = ADM1000",
            fg="gray"
        ).pack(pady=5)

        frame = tk.Frame(self.__root)
        frame.pack(pady=12)

        tk.Label(frame, text="User ID").grid(row=0, column=0, padx=5, pady=5)
        user_id_entry = tk.Entry(frame, width=35)
        user_id_entry.grid(row=0, column=1, padx=5, pady=5)

        tk.Label(frame, text="Password").grid(row=1, column=0, padx=5, pady=5)
        password_entry = tk.Entry(frame, width=35, show="*")
        password_entry.grid(row=1, column=1, padx=5, pady=5)

        remember_var = tk.BooleanVar()
        tk.Checkbutton(self.__root, text="Remember me", variable=remember_var).pack()

        def login_action():
            try:
                user = self.__system.validate_login(user_id_entry.get(), password_entry.get(), remember_var.get())
                messagebox.showinfo("Login Successful", f"Welcome {user.get_name()}")
                self.show_main_dashboard()
            except (ValueError, PermissionError) as error:
                messagebox.showerror("Login Error", str(error))

        tk.Button(self.__root, text="Login", width=25, command=login_action).pack(pady=5)
        tk.Button(self.__root, text="Create Account", width=25, command=self.show_register_screen).pack(pady=5)

    def show_register_screen(self):
        """
        Purpose:
        - Displays account creation form.

        Logical reason:
        New users need a way to register before login.
        """
        self.clear_window()

        tk.Label(self.__root, text="Create Account", font=("Arial", 18, "bold")).pack(pady=20)

        frame = tk.Frame(self.__root)
        frame.pack(pady=10)

        role_var = tk.StringVar(value="STUDENT")
        fields = {}

        tk.Label(frame, text="Role").grid(row=0, column=0, padx=5, pady=5)
        ttk.Combobox(
            frame,
            textvariable=role_var,
            values=["STUDENT", "STAFF", "ADMIN"],
            state="readonly",
            width=32
        ).grid(row=0, column=1, padx=5, pady=5)

        for index, label in enumerate(["University/Work ID", "Name", "Email", "Password"], start=1):
            tk.Label(frame, text=label).grid(row=index, column=0, padx=5, pady=5)
            entry = tk.Entry(frame, width=35, show="*" if label == "Password" else "")
            entry.grid(row=index, column=1, padx=5, pady=5)
            fields[label] = entry

        def register_action():
            try:
                role = UserRole[role_var.get()]
                user = self.__system.register_user(
                    role,
                    fields["University/Work ID"].get(),
                    fields["Name"].get(),
                    fields["Email"].get(),
                    fields["Password"].get()
                )
                messagebox.showinfo("Account Created", f"Account created successfully.\nYour login User ID is:\n{user.get_user_id()}")
                self.show_login_screen()
            except (ValueError, TypeError) as error:
                messagebox.showerror("Registration Error", str(error))

        tk.Button(self.__root, text="Register", width=25, command=register_action).pack(pady=5)
        tk.Button(self.__root, text="Back", width=25, command=self.show_login_screen).pack(pady=5)

    def show_main_dashboard(self):
        """
        Purpose:
        - Displays the correct dashboard after login.

        Logical reason:
        Students/staff and admins need different buttons and information.
        """
        user = self.__system.get_current_user()
        self.clear_window()

        tk.Label(self.__root, text=f"Welcome, {user.get_name()}", font=("Arial", 18, "bold")).pack(pady=10)

        status_color = "red" if user.get_status() == UserStatus.FROZEN else "black"
        tk.Label(
            self.__root,
            text=f"Role: {user.get_role().value} | Access: {user.get_access_type().value} | Status: {user.get_status().value}",
            font=("Arial", 12, "bold"),
            fg=status_color
        ).pack()

        if user.get_status() == UserStatus.FROZEN:
            tk.Label(
                self.__root,
                text="Your account is frozen. You cannot make new bookings until admin restores your account.",
                fg="red",
                font=("Arial", 11, "bold")
            ).pack(pady=5)

        if user.get_role() != UserRole.ADMIN:
            tk.Label(
                self.__root,
                text=f"Credits Today: {self.__system.get_daily_credit_text(user)}",
                font=("Arial", 11, "bold"),
                fg="blue"
            ).pack(pady=5)

        if self.__system.get_announcements():
            tk.Label(self.__root, text="Announcements", font=("Arial", 13, "bold")).pack(pady=5)
            latest_announcements = "\n".join(self.__system.get_announcements()[-3:])
            tk.Label(self.__root, text=latest_announcements, justify="left", fg="purple").pack()

        frame = tk.Frame(self.__root)
        frame.pack(pady=20)

        if user.get_role() != UserRole.ADMIN:
            tk.Button(frame, text="Book Facility", width=32, command=self.show_booking_screen).grid(row=0, column=0, padx=8, pady=6)
            tk.Button(frame, text="View / Modify / Cancel Bookings", width=32, command=self.show_my_bookings).grid(row=0, column=1, padx=8, pady=6)
            tk.Button(frame, text="Update Profile", width=32, command=self.show_update_profile).grid(row=1, column=0, padx=8, pady=6)
            tk.Button(frame, text="Upgrade to Premium", width=32, command=self.show_upgrade_payment).grid(row=1, column=1, padx=8, pady=6)
            tk.Button(frame, text="Downgrade to Standard", width=32, command=self.downgrade_user).grid(row=2, column=0, padx=8, pady=6)

        if user.get_role() == UserRole.ADMIN:
            tk.Button(frame, text="Admin Dashboard", width=32, command=self.show_admin_dashboard).grid(row=0, column=0, padx=8, pady=6)

        tk.Button(frame, text="Logout", width=32, command=self.logout_action).grid(row=3, column=0, padx=8, pady=6)

    def logout_action(self):
        """
        Purpose:
        - Logs out current user and returns to login screen.

        Logical reason:
        Users need to end their session safely.
        """
        self.__system.logout()
        self.show_login_screen()

    def show_update_profile(self):
        """
        Purpose:
        - Lets users edit name and email.

        Logical reason:
        Profiles should be updateable through the GUI.
        """
        user = self.__system.get_current_user()
        self.clear_window()

        tk.Label(self.__root, text="Update Profile", font=("Arial", 18, "bold")).pack(pady=15)

        frame = tk.Frame(self.__root)
        frame.pack(pady=10)

        tk.Label(frame, text="Name").grid(row=0, column=0, padx=5, pady=5)
        name_entry = tk.Entry(frame, width=35)
        name_entry.insert(0, user.get_name())
        name_entry.grid(row=0, column=1, padx=5, pady=5)

        tk.Label(frame, text="Email").grid(row=1, column=0, padx=5, pady=5)
        email_entry = tk.Entry(frame, width=35)
        email_entry.insert(0, user.get_email())
        email_entry.grid(row=1, column=1, padx=5, pady=5)

        def save_profile():
            try:
                user.update_profile(name_entry.get(), email_entry.get())
                self.__system.save_data_to_file("smart_campus_data.dat")
                messagebox.showinfo("Saved", "Profile updated.")
                self.show_main_dashboard()
            except ValueError as error:
                messagebox.showerror("Profile Error", str(error))

        tk.Button(self.__root, text="Save", width=25, command=save_profile).pack(pady=5)
        tk.Button(self.__root, text="Back", width=25, command=self.show_main_dashboard).pack(pady=5)

    def show_upgrade_payment(self):
        """
        Purpose:
        - Shows upgrade bill and payment confirmation window.

        Logical reason:
        Users should see the cost and benefits before upgrading.
        """
        user = self.__system.get_current_user()

        if user.get_access_type() == AccessType.PREMIUM:
            messagebox.showinfo("Already Premium", "You already have PREMIUM access.")
            return

        window = tk.Toplevel(self.__root)
        window.title("Upgrade Payment")
        window.geometry("460x320")

        bill = (
            "UPGRADE PAYMENT BILL\n"
            "-----------------------------\n"
            f"User: {user.get_name()}\n"
            f"Current Access: {user.get_access_type().value}\n"
            "New Access: PREMIUM\n"
            "Upgrade Fee: 30 AED\n\n"
            "Premium Benefits:\n"
            "- Book multiple facility types\n"
            "- 3 bookings per facility type per day\n"
            "- Priority access"
        )

        tk.Label(window, text=bill, justify="left", font=("Arial", 11)).pack(pady=15)

        def pay_upgrade():
            try:
                self.__system.process_upgrade_payment()
                messagebox.showinfo("Payment Successful", "Payment completed. Your access is now PREMIUM.")
                window.destroy()
                self.show_main_dashboard()
            except ValueError as error:
                messagebox.showerror("Upgrade Error", str(error))

        tk.Button(window, text="Pay 30 AED", width=20, command=pay_upgrade).pack(pady=10)

    def downgrade_user(self):
        """
        Purpose:
        - Downgrades logged-in user through GUI.

        Logical reason:
        Users need a simple dashboard action for downgrade.
        """
        self.__system.downgrade_current_user()
        messagebox.showinfo("Updated", "Your access is now STANDARD.")
        self.show_main_dashboard()

    def show_booking_screen(self):
        """
        Purpose:
        - Shows facility booking form.

        Logical reason:
        Users need to select facility, date, slot, usage type, and payment.
        """
        self.clear_window()
        user = self.__system.get_current_user()

        if user.get_status() == UserStatus.FROZEN:
            messagebox.showerror("Account Frozen", "Your account is frozen. You cannot make a booking.")
            self.show_main_dashboard()
            return

        tk.Label(self.__root, text="Book Facility", font=("Arial", 18, "bold")).pack(pady=10)

        frame = tk.Frame(self.__root)
        frame.pack(pady=10)

        facility_var = tk.StringVar()
        slot_var = tk.StringVar()
        usage_var = tk.StringVar(value=UsageType.INTERNAL.value)

        allowed_facilities = self.__system.get_allowed_facilities(user)
        facility_display = []

        for facility_id, facility in allowed_facilities.items():
            facility_display.append(
                f"{facility_id} - {facility.get_facility_name()} | "
                f"Type: {facility.get_facility_type()} | "
                f"Capacity: {facility.get_capacity()} | "
                f"Price: {facility.get_price_per_hour()} AED"
            )

        tk.Label(frame, text="Facility").grid(row=0, column=0, padx=5, pady=5)
        facility_box = ttk.Combobox(frame, textvariable=facility_var, values=facility_display, state="readonly", width=85)
        facility_box.grid(row=0, column=1, padx=5, pady=5)

        tk.Label(frame, text="Date").grid(row=1, column=0, padx=5, pady=5)
        date_entry = tk.Entry(frame, width=30)
        date_entry.insert(0, str(date.today()))
        date_entry.grid(row=1, column=1, sticky="w", padx=5, pady=5)

        tk.Label(frame, text="Time Slot").grid(row=2, column=0, padx=5, pady=5)
        slot_box = ttk.Combobox(frame, textvariable=slot_var, state="readonly", width=30)
        slot_box.grid(row=2, column=1, sticky="w", padx=5, pady=5)

        tk.Label(frame, text="Usage Type").grid(row=3, column=0, padx=5, pady=5)
        usage_box = ttk.Combobox(
            frame,
            textvariable=usage_var,
            values=[UsageType.INTERNAL.value, UsageType.EXTERNAL.value],
            state="disabled",
            width=30
        )
        usage_box.grid(row=3, column=1, sticky="w", padx=5, pady=5)

        credit_label = tk.Label(self.__root, text="", font=("Arial", 11, "bold"), fg="blue")
        credit_label.pack(pady=5)

        details_label = tk.Label(self.__root, text="", justify="left", font=("Arial", 11))
        details_label.pack(pady=10)

        def update_facility_details(event=None):
            credit_label.config(
                text=f"Credits for {date_entry.get()}: {self.__system.get_daily_credit_text(user, date_entry.get())}"
            )

            if not facility_var.get():
                return

            facility_id = facility_var.get().split(" - ")[0]
            facility = self.__system.get_facility_catalog().get_facility(facility_id)
            available_slots = facility.get_available_slots(date_entry.get())

            slot_box["values"] = available_slots

            if available_slots:
                slot_var.set(available_slots[0])
            else:
                slot_var.set("")

            if facility.get_facility_type() == "EVENT_HALL":
                usage_box.config(state="readonly")
            else:
                usage_var.set(UsageType.INTERNAL.value)
                usage_box.config(state="disabled")

            details_label.config(
                text=f"Facility Details\n"
                     f"Name: {facility.get_facility_name()}\n"
                     f"Type: {facility.get_facility_type()}\n"
                     f"Location: {facility.get_location()}\n"
                     f"Capacity: {facility.get_capacity()}\n"
                     f"Available Slots: {len(available_slots)}\n"
                     f"Available: {facility.get_is_available()}\n"
                     f"{facility.get_rules()}"
            )

        tk.Button(
            frame,
            text="Open Calendar",
            command=lambda: CalendarPopup(self.__root, date_entry, update_facility_details)
        ).grid(row=1, column=1, padx=250)

        facility_box.bind("<<ComboboxSelected>>", update_facility_details)
        usage_box.bind("<<ComboboxSelected>>", update_facility_details)
        update_facility_details()

        def show_payment_bill():
            try:
                if not facility_var.get():
                    raise ValueError("Please select a facility.")
                if not slot_var.get():
                    raise ValueError("Please select an available time slot.")

                facility_id = facility_var.get().split(" - ")[0]
                facility = self.__system.get_facility_catalog().get_facility(facility_id)
                cost = facility.calculate_cost(usage_var.get())

                bill_window = tk.Toplevel(self.__root)
                bill_window.title("Booking Bill")
                bill_window.geometry("540x480")

                bill_text = (
                    "BOOKING BILL\n"
                    "-----------------------------\n"
                    f"User: {user.get_name()}\n"
                    f"Facility: {facility.get_facility_name()}\n"
                    f"Location: {facility.get_location()}\n"
                    f"Capacity: {facility.get_capacity()}\n"
                    f"Date: {date_entry.get()}\n"
                    f"Time Slot: {slot_var.get()}\n"
                    f"Usage Type: {usage_var.get()}\n"
                    f"Total Cost: {cost} AED\n\n"
                    f"{facility.get_rules()}\n\n"
                    "Click confirm to finalize the booking."
                )

                tk.Label(bill_window, text=bill_text, justify="left", font=("Arial", 11)).pack(pady=10)

                def pay_and_confirm():
                    try:
                        booking, receipt = self.__system.create_booking_after_payment(
                            facility_id,
                            slot_var.get(),
                            date_entry.get(),
                            usage_var.get()
                        )
                        messagebox.showinfo("Receipt", receipt.generate_receipt())
                        bill_window.destroy()
                        self.show_main_dashboard()
                    except (ValueError, PermissionError) as error:
                        messagebox.showerror("Booking Error", str(error))

                button_text = "Pay and Confirm" if cost > 0 else "Confirm Free Booking"
                tk.Button(bill_window, text=button_text, width=25, command=pay_and_confirm).pack(pady=10)

            except (ValueError, PermissionError) as error:
                messagebox.showerror("Booking Error", str(error))

        tk.Button(self.__root, text="Continue to Payment / Confirmation", width=35, command=show_payment_bill).pack(pady=5)
        tk.Button(self.__root, text="Back", width=25, command=self.show_main_dashboard).pack(pady=5)

    def show_my_bookings(self):
        """
        Purpose:
        - Displays user bookings and allows modify/cancel.

        Logical reason:
        Users need to manage their own reservations.
        """
        self.clear_window()
        user = self.__system.get_current_user()

        tk.Label(self.__root, text="My Bookings", font=("Arial", 18, "bold")).pack(pady=10)

        text = tk.Text(self.__root, width=120, height=22)
        text.pack(pady=8)

        my_bookings = []
        for booking in self.__system.get_bookings().values():
            if booking.get_user().get_user_id() == user.get_user_id():
                my_bookings.append(booking)

        if not my_bookings:
            text.insert(tk.END, "No bookings found.")
        else:
            for booking in my_bookings:
                text.insert(tk.END, booking.generate_booking_summary() + "\n-----------------------------\n")

        action_frame = tk.Frame(self.__root)
        action_frame.pack(pady=5)

        tk.Label(action_frame, text="Booking ID").grid(row=0, column=0, padx=5)
        booking_entry = tk.Entry(action_frame, width=15)
        booking_entry.grid(row=0, column=1, padx=5)

        tk.Label(action_frame, text="New Slot").grid(row=0, column=2, padx=5)
        new_slot_entry = tk.Entry(action_frame, width=15)
        new_slot_entry.grid(row=0, column=3, padx=5)

        tk.Label(action_frame, text="New Date").grid(row=0, column=4, padx=5)
        new_date_entry = tk.Entry(action_frame, width=15)
        new_date_entry.insert(0, str(date.today()))
        new_date_entry.grid(row=0, column=5, padx=5)

        def cancel_action():
            try:
                self.__system.cancel_booking(booking_entry.get())
                messagebox.showinfo("Cancelled", "Booking cancelled. Penalty credit decreased.")
                self.show_my_bookings()
            except (ValueError, PermissionError) as error:
                messagebox.showerror("Cancel Error", str(error))

        def modify_action():
            try:
                self.__system.modify_booking(booking_entry.get(), new_slot_entry.get(), new_date_entry.get())
                messagebox.showinfo("Modified", "Booking modified.")
                self.show_my_bookings()
            except (ValueError, PermissionError) as error:
                messagebox.showerror("Modify Error", str(error))

        tk.Button(action_frame, text="Cancel", command=cancel_action).grid(row=0, column=6, padx=5)
        tk.Button(action_frame, text="Modify", command=modify_action).grid(row=0, column=7, padx=5)
        tk.Button(self.__root, text="Back", width=25, command=self.show_main_dashboard).pack(pady=8)

    def show_admin_dashboard(self):
        """
        Purpose:
        - Displays admin tabs for user, booking, facility, and announcement management.

        Logical reason:
        Admins need organized tools for monitoring and control.
        """
        self.clear_window()

        tk.Label(self.__root, text="Admin Dashboard", font=("Arial", 18, "bold")).pack(pady=8)

        notebook = ttk.Notebook(self.__root)
        notebook.pack(fill="both", expand=True, padx=10, pady=10)

        users_tab = tk.Frame(notebook)
        bookings_tab = tk.Frame(notebook)
        facilities_tab = tk.Frame(notebook)
        announcements_tab = tk.Frame(notebook)

        notebook.add(users_tab, text="Users Dashboard")
        notebook.add(bookings_tab, text="Bookings Calendar")
        notebook.add(facilities_tab, text="Facilities")
        notebook.add(announcements_tab, text="Announcements")

        self.build_users_tab(users_tab)
        self.build_bookings_tab(bookings_tab)
        self.build_facilities_tab(facilities_tab)
        self.build_announcements_tab(announcements_tab)

        tk.Button(self.__root, text="Back", width=25, command=self.show_main_dashboard).pack(pady=5)

    def build_users_tab(self, parent):
        """
        Purpose:
        - Builds admin user management tab.

        Logical reason:
        Admin must upgrade, downgrade, freeze, and unfreeze users.
        """
        text = tk.Text(parent, width=120, height=20)
        text.pack(pady=8)

        text.insert(tk.END, "USER DASHBOARD\n")
        text.insert(tk.END, "User ID | Name | Role | Access | Status | Booking Credit | Penalty Credit\n")
        text.insert(tk.END, "-" * 120 + "\n")

        for user_id, user in self.__system.get_users().items():
            text.insert(
                tk.END,
                f"{user_id} | {user.get_name()} | {user.get_role().value} | "
                f"{user.get_access_type().value} | {user.get_status().value} | "
                f"{self.__system.get_daily_credit_text(user)} | Penalty: {user.get_penalty_credits()} / 3\n"
            )

        frame = tk.Frame(parent)
        frame.pack(pady=5)

        tk.Label(frame, text="User ID").grid(row=0, column=0, padx=5)
        user_entry = tk.Entry(frame, width=20)
        user_entry.grid(row=0, column=1, padx=5)

        def do_action(action):
            try:
                user_id = user_entry.get()

                if action == "upgrade":
                    self.__system.upgrade_user_by_admin(user_id)
                elif action == "downgrade":
                    self.__system.downgrade_user_by_admin(user_id)
                elif action == "freeze":
                    self.__system.freeze_user(user_id)
                elif action == "unfreeze":
                    self.__system.unfreeze_user(user_id)

                messagebox.showinfo("Updated", "User updated.")
                self.show_admin_dashboard()

            except (ValueError, KeyError) as error:
                messagebox.showerror("Admin Error", str(error))

        tk.Button(frame, text="Upgrade", command=lambda: do_action("upgrade")).grid(row=0, column=2, padx=5)
        tk.Button(frame, text="Downgrade", command=lambda: do_action("downgrade")).grid(row=0, column=3, padx=5)
        tk.Button(frame, text="Freeze", command=lambda: do_action("freeze")).grid(row=0, column=4, padx=5)
        tk.Button(frame, text="Unfreeze / Reset Penalty", command=lambda: do_action("unfreeze")).grid(row=0, column=5, padx=5)

    def build_bookings_tab(self, parent):
        """
        Purpose:
        - Builds admin booking monitoring tab.

        Logical reason:
        Admin needs to review bookings and mark no-shows.
        """
        control = tk.Frame(parent)
        control.pack(pady=5)

        tk.Label(control, text="Date").grid(row=0, column=0, padx=5)

        date_entry = tk.Entry(control, width=15)
        date_entry.insert(0, str(date.today()))
        date_entry.grid(row=0, column=1, padx=5)

        output = tk.Text(parent, width=120, height=21)
        output.pack(pady=5)

        def refresh():
            output.delete("1.0", tk.END)
            selected_date = date_entry.get()

            output.insert(tk.END, f"BOOKINGS ON {selected_date}\n")
            output.insert(tk.END, "=" * 90 + "\n\n")

            bookings = self.__system.get_bookings_by_date(selected_date)

            if not bookings:
                output.insert(tk.END, "No bookings on this date.\n\n")
            else:
                for booking in bookings.values():
                    output.insert(
                        tk.END,
                        f"{booking.get_booking_id()} | "
                        f"{booking.get_user().get_name()} | "
                        f"{booking.get_facility().get_facility_name()} | "
                        f"{booking.get_slot()} | "
                        f"{booking.get_status().value} | "
                        f"{booking.get_total_cost()} AED\n"
                    )

            output.insert(tk.END, "\nFACILITY SLOT STATUS\n")
            output.insert(tk.END, "=" * 90 + "\n")

            for facility in self.__system.get_facility_catalog().get_facilities().values():
                output.insert(tk.END, f"\n{facility.get_facility_name()} - {facility.get_location()}\n")
                available_slots = facility.get_available_slots(selected_date)
                closed_slots = facility.get_closed_slots(selected_date)

                for slot in facility.get_time_slots():
                    if slot in closed_slots:
                        output.insert(tk.END, f"  {slot}: CLOSED BY ADMIN\n")
                    elif slot in available_slots:
                        output.insert(tk.END, f"  {slot}: Available\n")
                    else:
                        output.insert(tk.END, f"  {slot}: BOOKED\n")

        tk.Button(control, text="Open Calendar", command=lambda: CalendarPopup(self.__root, date_entry, refresh)).grid(row=0, column=2, padx=5)
        tk.Button(control, text="Refresh", command=refresh).grid(row=0, column=3, padx=5)

        action_frame = tk.Frame(parent)
        action_frame.pack(pady=5)

        tk.Label(action_frame, text="Booking ID").grid(row=0, column=0, padx=5)
        no_show_entry = tk.Entry(action_frame, width=15)
        no_show_entry.grid(row=0, column=1, padx=5)

        def mark_no_show_action():
            try:
                self.__system.mark_booking_no_show(no_show_entry.get())
                messagebox.showinfo("No-Show Recorded", "No-show recorded. Penalty credit decreased. User may be frozen automatically.")
                self.show_admin_dashboard()
            except ValueError as error:
                messagebox.showerror("No-Show Error", str(error))

        tk.Button(action_frame, text="Mark No-Show", command=mark_no_show_action).grid(row=0, column=2, padx=5)

        refresh()

    def build_facilities_tab(self, parent):
        """
        Purpose:
        - Builds admin facility management tab.

        Logical reason:
        Admin needs to open, close, and manage facility availability.
        """
        text = tk.Text(parent, width=120, height=16)
        text.pack(pady=8)

        for facility_id, facility in self.__system.get_facility_catalog().get_facilities().items():
            text.insert(
                tk.END,
                f"{facility_id} | {facility.get_facility_name()} | {facility.get_facility_type()} | "
                f"{facility.get_location()} | Capacity: {facility.get_capacity()} | "
                f"Price: {facility.get_price_per_hour()} AED | Available: {facility.get_is_available()}\n"
            )

        frame = tk.Frame(parent)
        frame.pack(pady=5)

        facility_var = tk.StringVar()
        options = []

        for facility_id, facility in self.__system.get_facility_catalog().get_facilities().items():
            options.append(f"{facility_id} - {facility.get_facility_name()}")

        tk.Label(frame, text="Facility").grid(row=0, column=0, padx=5)

        ttk.Combobox(frame, textvariable=facility_var, values=options, state="readonly", width=40).grid(row=0, column=1, padx=5)

        def set_availability(flag):
            try:
                if not facility_var.get():
                    raise ValueError("Select a facility first.")

                facility_id = facility_var.get().split(" - ")[0]
                facility = self.__system.get_facility_catalog().get_facility(facility_id)
                facility.set_is_available(flag)
                self.__system.save_data_to_file("smart_campus_data.dat")

                messagebox.showinfo("Updated", "Facility availability updated.")
                self.show_admin_dashboard()

            except ValueError as error:
                messagebox.showerror("Facility Error", str(error))

        tk.Button(frame, text="Set Available", command=lambda: set_availability(True)).grid(row=0, column=2, padx=5)
        tk.Button(frame, text="Set Unavailable", command=lambda: set_availability(False)).grid(row=0, column=3, padx=5)

        closure_frame = tk.Frame(parent)
        closure_frame.pack(pady=10)

        tk.Label(closure_frame, text="Closure Date").grid(row=0, column=0, padx=5)

        closure_date_entry = tk.Entry(closure_frame, width=15)
        closure_date_entry.insert(0, str(date.today()))
        closure_date_entry.grid(row=0, column=1, padx=5)

        tk.Button(closure_frame, text="Open Calendar", command=lambda: CalendarPopup(self.__root, closure_date_entry)).grid(row=0, column=2, padx=5)

        tk.Label(closure_frame, text="Time Slot").grid(row=0, column=3, padx=5)

        slot_var = tk.StringVar()
        slot_box = ttk.Combobox(
            closure_frame,
            textvariable=slot_var,
            values=[
                "09:00-10:00",
                "10:00-11:00",
                "11:00-12:00",
                "12:00-13:00",
                "14:00-15:00",
                "15:00-16:00"
            ],
            state="readonly",
            width=18
        )
        slot_box.grid(row=0, column=4, padx=5)

        def close_selected_slot():
            try:
                if not facility_var.get():
                    raise ValueError("Select a facility first.")
                if not slot_var.get():
                    raise ValueError("Select a time slot first.")

                facility_id = facility_var.get().split(" - ")[0]
                self.__system.close_facility_slot(facility_id, closure_date_entry.get(), slot_var.get())
                messagebox.showinfo("Closed", "The selected facility slot is now closed.")
                self.show_admin_dashboard()

            except ValueError as error:
                messagebox.showerror("Closure Error", str(error))

        def open_selected_slot():
            try:
                if not facility_var.get():
                    raise ValueError("Select a facility first.")
                if not slot_var.get():
                    raise ValueError("Select a time slot first.")

                facility_id = facility_var.get().split(" - ")[0]
                self.__system.open_facility_slot(facility_id, closure_date_entry.get(), slot_var.get())
                messagebox.showinfo("Opened", "The selected facility slot is now available again.")
                self.show_admin_dashboard()

            except ValueError as error:
                messagebox.showerror("Opening Error", str(error))

        tk.Button(closure_frame, text="Close Selected Slot", command=close_selected_slot).grid(row=0, column=5, padx=5)
        tk.Button(closure_frame, text="Open Selected Slot", command=open_selected_slot).grid(row=0, column=6, padx=5)

    def build_announcements_tab(self, parent):
        """
        Purpose:
        - Builds admin announcement tab.

        Logical reason:
        Admin needs to post maintenance or system messages.
        """
        text = tk.Text(parent, width=120, height=18)
        text.pack(pady=8)

        if not self.__system.get_announcements():
            text.insert(tk.END, "No announcements yet.")
        else:
            for announcement in self.__system.get_announcements():
                text.insert(tk.END, announcement + "\n")

        frame = tk.Frame(parent)
        frame.pack(pady=5)

        tk.Label(frame, text="New Announcement").grid(row=0, column=0, padx=5)
        announcement_entry = tk.Entry(frame, width=70)
        announcement_entry.grid(row=0, column=1, padx=5)

        def add_announcement():
            """
            Purpose:
            - Adds announcement with timestamp.

            Logical reason:
            Users need to see system updates and maintenance messages.
            """
            try:
                self.__system.add_announcement(announcement_entry.get())
                messagebox.showinfo("Added", "Announcement added.")
                self.show_admin_dashboard()
            except ValueError as error:
                messagebox.showerror("Announcement Error", str(error))

        tk.Button(frame, text="Add Announcement", command=add_announcement).grid(row=0, column=2, padx=5)


if __name__ == "__main__":
    try:
        root = tk.Tk()
        app = SmartCampusGUI(root)
        root.mainloop()

    except tk.TclError:
        print("Tkinter GUI cannot open here. Run it locally in PyCharm.")

    except KeyboardInterrupt:
        print("Program closed by user.")

Test Cases

Test Cases 1 – 3 :

In [ ]:
# ============================================================
# TEST CASES GROUP 1: USER + BOOKING + PAYMENT TESTS
# Includes:
# 1) Registration/Login/Upgrade/Downgrade
# 2) Successful booking + failed Standard limit booking
# 3) Paid booking + successful double booking
# ============================================================

def reset_test_files():
    for file_name in ["smart_campus_data.dat", "smart_campus_session.dat"]:
        if os.path.exists(file_name):
            os.remove(file_name)


def test_case_1(system):
    """
    TEST CASE 1: Registration, Login, Upgrade, Failed Upgrade, Downgrade

    Scenario:
    Dana Student creates an account, logs in successfully, upgrades from STANDARD
    to PREMIUM, tries to upgrade again, then downgrades back to STANDARD.

    Functions tested:
    register_user(), validate_login(), process_upgrade_payment(),
    downgrade_current_user()
    """

    print("\n--- TEST CASE 1: Registration/Login/Upgrade/Downgrade ---")

    dana = system.register_user(UserRole.STUDENT, "3001", "Dana Student", "dana@student.com", "1234")
    maryam = system.register_user(UserRole.STAFF, "4001", "Maryam Staff", "maryam@staff.com", "1234")
    admin = system.register_user(UserRole.ADMIN, "9009", "Admin User", "admin@campus.com", "1234")

    print("Created Student:", dana.get_user_id(), "| Access:", dana.get_access_type().value)
    print("Created Staff:", maryam.get_user_id(), "| Access:", maryam.get_access_type().value)
    print("Created Admin:", admin.get_user_id())

    system.validate_login("STU3001", "1234")
    print("Login successful for:", system.get_current_user().get_name())

    system.process_upgrade_payment()
    print("Upgrade successful. New access:", system.get_current_user().get_access_type().value)

    try:
        system.process_upgrade_payment()
    except ValueError as error:
        print("Failed upgrade correctly blocked:", error)

    system.downgrade_current_user()
    print("Downgrade successful. Current access:", system.get_current_user().get_access_type().value)


def test_case_2(system):
    """
    TEST CASE 2: Successful Booking + Failed Booking Due to Standard Limit

    Scenario:
    Dana is a STANDARD Student. She books Study Room A successfully. Then she
    attempts another Study Room booking on the same date. The system rejects
    the second booking because STANDARD users only have 1 daily booking credit.

    Functions tested:
    create_booking_after_payment(), check_booking_credit(),
    count_user_bookings_on_date(), get_daily_credit_text()
    """

    print("\n--- TEST CASE 2: Successful Booking + Failed Standard Limit ---")

    system.validate_login("STU3001", "1234")

    booking_date = "2026-05-11"

    booking, receipt = system.create_booking_after_payment(
        "SR-A",
        "09:00-10:00",
        booking_date,
        UsageType.INTERNAL.value
    )

    print("Booking confirmed:", booking.get_booking_id())
    print("Facility:", booking.get_facility().get_facility_name())
    print("Cost:", booking.get_total_cost(), "AED")
    print("Credit after booking:", system.get_daily_credit_text(system.get_current_user(), booking_date))

    try:
        system.create_booking_after_payment(
            "SR-B",
            "10:00-11:00",
            booking_date,
            UsageType.INTERNAL.value
        )
    except PermissionError as error:
        print("Second booking correctly rejected:", error)


def test_case_3(system):
    """
    TEST CASE 3: Paid Booking + Successful Double Booking

    Scenario:
    Maryam Staff logs in and upgrades to PREMIUM. She books Event Hall 1 for
    EXTERNAL use, so the system calculates a payment. Then she books Sports
    Court B on the same date but at a different time, so the system accepts it.

    Functions tested:
    process_upgrade_payment(), create_booking_after_payment(),
    calculate_cost(), process_payment(), generate_receipt(),
    check_booking_conflict()
    """

    print("\n--- TEST CASE 3: Paid Booking + Successful Double Booking ---")

    system.logout()
    system.validate_login("STF4001", "1234")

    system.process_upgrade_payment()
    print("Maryam upgraded to:", system.get_current_user().get_access_type().value)

    booking_date = "2026-05-12"

    hall_booking, hall_receipt = system.create_booking_after_payment(
        "EH-1",
        "09:00-10:00",
        booking_date,
        UsageType.EXTERNAL.value
    )

    print("Paid EventHall booking confirmed:", hall_booking.get_booking_id())
    print("EventHall cost:", hall_booking.get_total_cost(), "AED")

    court_booking, court_receipt = system.create_booking_after_payment(
        "SC-B",
        "10:00-11:00",
        booking_date,
        UsageType.INTERNAL.value
    )

    print("Second booking confirmed:", court_booking.get_booking_id())
    print("SportsCourt cost:", court_booking.get_total_cost(), "AED")
    print("Double booking accepted because slots are different.")


def run_test_group_1():
    reset_test_files()
    system = BookingSystem()

    test_case_1(system)
    test_case_2(system)
    test_case_3(system)

    print("\n--- GROUP 1 TESTS COMPLETED ---")


run_test_group_1()


--- TEST CASE 1: Registration/Login/Upgrade/Downgrade ---
Created Student: STU3001 | Access: STANDARD
Created Staff: STF4001 | Access: STANDARD
Created Admin: ADM9009
Login successful for: Dana Student
Upgrade successful. New access: PREMIUM
Failed upgrade correctly blocked: User already has PREMIUM access.
Downgrade successful. Current access: STANDARD

--- TEST CASE 2: Successful Booking + Failed Standard Limit ---
Booking confirmed: B0001
Facility: Study Room A
Cost: 0 AED
Credit after booking: Booking Credit: 0 / 1 | Penalty Credit: 3 / 3
Second booking correctly rejected: STANDARD access allows only 1 booking per day.

--- TEST CASE 3: Paid Booking + Successful Double Booking ---
Maryam upgraded to: PREMIUM
Paid EventHall booking confirmed: B0002
EventHall cost: 100 AED
Second booking confirmed: B0003
SportsCourt cost: 20 AED
Double booking accepted because slots are different.

--- GROUP 1 TESTS COMPLETED ---


Test Cases 4 – 6 :

In [ ]:
# ============================================================
# TEST CASES GROUP 2: ADMIN + FREEZING + MAINTENANCE TESTS
# Includes:
# 4) Admin upgrade + booking monitoring
# 5) Freezing punishment
# 6) Announcement + maintenance closure
# ============================================================

def reset_test_files():
    for file_name in ["smart_campus_data.dat", "smart_campus_session.dat"]:
        if os.path.exists(file_name):
            os.remove(file_name)


def prepare_group_2_data():
    """
    This prepares clean users and one booking so the admin tests have data.
    """
    reset_test_files()
    system = BookingSystem()

    system.register_user(UserRole.STUDENT, "3001", "Dana Student", "dana@student.com", "1234")
    system.register_user(UserRole.STAFF, "4001", "Maryam Staff", "maryam@staff.com", "1234")
    system.register_user(UserRole.ADMIN, "9009", "Admin User", "admin@campus.com", "1234")

    system.validate_login("STU3001", "1234")
    system.create_booking_after_payment(
        "SR-A",
        "09:00-10:00",
        "2026-05-12",
        UsageType.INTERNAL.value
    )

    return system


def test_case_4(system):
    """
    TEST CASE 4: Admin Helping with Upgrade + Admin Monitoring

    Scenario:
    Admin logs in, upgrades Dana Student to PREMIUM, then checks booking
    activity for 2026-05-12.

    Functions tested:
    validate_login(), upgrade_user_by_admin(), get_users(),
    get_bookings_by_date()
    """

    print("\n--- TEST CASE 4: Admin Upgrade + Monitoring ---")

    system.logout()
    system.validate_login("ADM9009", "1234")

    system.upgrade_user_by_admin("STU3001")
    print("Dana access after admin upgrade:", system.get_users()["STU3001"].get_access_type().value)

    bookings = system.get_bookings_by_date("2026-05-12")

    print("Admin monitoring bookings on 2026-05-12:")
    for booking_id, booking in bookings.items():
        print(
            booking_id,
            "| User:", booking.get_user().get_name(),
            "| Facility:", booking.get_facility().get_facility_name(),
            "| Slot:", booking.get_slot(),
            "| Cost:", booking.get_total_cost(), "AED"
        )


def test_case_5(system):
    """
    TEST CASE 5: Cancellation Penalty + Automatic Freezing

    Scenario:
    Dana creates and cancels three bookings on different dates. Each cancellation
    decreases penalty credits. After the third cancellation, the system freezes
    Dana automatically. A new booking attempt is then rejected.

    Functions tested:
    create_booking_after_payment(), cancel_booking(),
    record_cancellation_violation(), auto_freeze_if_needed(),
    get_penalty_credits(), get_status()
    """

    print("\n--- TEST CASE 5: Freezing Punishment ---")

    system.logout()
    system.validate_login("STU3001", "1234")

    cancel_dates = ["2026-05-13", "2026-05-14", "2026-05-15"]

    for index, cancel_date in enumerate(cancel_dates, start=1):
        booking, receipt = system.create_booking_after_payment(
            "SR-A",
            "11:00-12:00",
            cancel_date,
            UsageType.INTERNAL.value
        )

        print("Created booking:", booking.get_booking_id())
        system.cancel_booking(booking.get_booking_id())

        print(
            "Cancellation",
            index,
            "| Penalty credits:",
            system.get_current_user().get_penalty_credits(),
            "| Status:",
            system.get_current_user().get_status().value
        )

    try:
        system.create_booking_after_payment(
            "SR-A",
            "12:00-13:00",
            "2026-05-18",
            UsageType.INTERNAL.value
        )
    except PermissionError as error:
        print("Frozen account booking correctly rejected:", error)


def test_case_6(system):
    """
    TEST CASE 6: Admin Announcement + Maintenance Closure + Blocked Booking

    Scenario:
    Admin adds a maintenance announcement and closes Event Hall 1 from
    09:00-10:00 on 2026-05-19. Maryam Staff logs in, sees the announcement,
    and tries to book the closed slot. The system rejects the booking.

    Functions tested:
    add_announcement(), close_facility_slot(), get_announcements(),
    get_available_slots(), create_booking_after_payment()
    """

    print("\n--- TEST CASE 6: Announcement + Maintenance Closure ---")

    system.logout()
    system.validate_login("ADM9009", "1234")

    maintenance_date = "2026-05-19"

    system.add_announcement(
        "Maintenance notice: Event Hall 1 is unavailable from 09:00-10:00."
    )

    system.close_facility_slot(
        "EH-1",
        maintenance_date,
        "09:00-10:00"
    )

    print("System announcements:")
    for announcement in system.get_announcements():
        print(announcement)

    system.logout()
    system.validate_login("STF4001", "1234")

    print("Maryam sees announcement before booking:")
    for announcement in system.get_announcements():
        print(announcement)

    facility = system.get_facility_catalog().get_facility("EH-1")
    print("Available EventHall slots:", facility.get_available_slots(maintenance_date))

    try:
        system.create_booking_after_payment(
            "EH-1",
            "09:00-10:00",
            maintenance_date,
            UsageType.EXTERNAL.value
        )
    except ValueError as error:
        print("Closed maintenance slot correctly rejected:", error)


def run_test_group_2():
    system = prepare_group_2_data()

    test_case_4(system)
    test_case_5(system)
    test_case_6(system)

    print("\n--- GROUP 2 TESTS COMPLETED ---")


run_test_group_2()


--- TEST CASE 4: Admin Upgrade + Monitoring ---
Dana access after admin upgrade: PREMIUM
Admin monitoring bookings on 2026-05-12:
B0001 | User: Dana Student | Facility: Study Room A | Slot: 09:00-10:00 | Cost: 0 AED

--- TEST CASE 5: Freezing Punishment ---
Created booking: B0002
Cancellation 1 | Penalty credits: 2 | Status: ACTIVE
Created booking: B0003
Cancellation 2 | Penalty credits: 1 | Status: ACTIVE
Created booking: B0004
Cancellation 3 | Penalty credits: 0 | Status: FROZEN
Frozen account booking correctly rejected: Your account is FROZEN. You cannot book.

--- TEST CASE 6: Announcement + Maintenance Closure ---
System announcements:
2026-05-08 10:39: Maintenance notice: Event Hall 1 is unavailable from 09:00-10:00.
Maryam sees announcement before booking:
2026-05-08 10:39: Maintenance notice: Event Hall 1 is unavailable from 09:00-10:00.
Available EventHall slots: ['10:00-11:00', '11:00-12:00', '12:00-13:00', '14:00-15:00', '15:00-16:00']
Closed maintenance slot correctly reje

Test Case 7

In [ ]:
# ============================================================
# PRIORITY TEST CASE: Premium User Gets the Slot First
# ============================================================
from enum import Enum
import pickle
import os
from datetime import datetime, date


class UserRole(Enum):
    """
    UserRole defines the three actors in the system.

    Purpose:
    - STUDENT users book student-access facilities.
    - STAFF users book staff-access facilities.
    - ADMIN users manage users, bookings, and facility availability.

    Logical reason:
    Using Enum prevents spelling mistakes like "student", "Student", or "STUDNT".
    The system only accepts the exact role values defined here.
    """
    STUDENT = "STUDENT"
    STAFF = "STAFF"
    ADMIN = "ADMIN"


class AccessType(Enum):
    """
    AccessType defines the user's permission level.

    Purpose:
    - STANDARD users have limited booking access.
    - PREMIUM users have more booking access.

    Logical reason:
    The booking rules depend on whether the user is STANDARD or PREMIUM,
    so this enum keeps the permission levels consistent across the whole system.
    """
    STANDARD = "STANDARD"
    PREMIUM = "PREMIUM"


class UserStatus(Enum):
    """
    UserStatus tracks whether a user is allowed to use the system.

    Purpose:
    - ACTIVE users can book normally.
    - FROZEN users are blocked from booking.

    Logical reason:
    This supports the punishment rule where users can be frozen after violations.
    """
    ACTIVE = "ACTIVE"
    FROZEN = "FROZEN"


class BookingStatus(Enum):
    """
    BookingStatus tracks the current state of a booking.

    Purpose:
    - CONFIRMED means the booking is active.
    - CANCELLED means the user cancelled it.
    - NO_SHOW means the user missed it.

    Logical reason:
    Booking status is needed for reports, penalties, and freeing booked slots.
    """
    PENDING = "PENDING"
    CONFIRMED = "CONFIRMED"
    CANCELLED = "CANCELLED"
    NO_SHOW = "NO_SHOW"


class UsageType(Enum):
    """
    UsageType explains how a facility is being used.

    Purpose:
    - INTERNAL use can be free for EventHall.
    - EXTERNAL use may require payment.

    Logical reason:
    EventHall cost depends on usage type, so this enum helps calculate fees correctly.
    """
    INTERNAL = "INTERNAL"
    EXTERNAL = "EXTERNAL"


class PaymentStatus(Enum):
    """
    PaymentStatus tracks the payment result.

    Purpose:
    - NOT_REQUIRED is used when the cost is 0.
    - PAID is used after successful payment.
    - REFUNDED is used when money is returned.

    Logical reason:
    The system needs payment status for receipts and proof of payment.
    """
    NOT_REQUIRED = "NOT_REQUIRED"
    PENDING = "PENDING"
    PAID = "PAID"
    FAILED = "FAILED"
    REFUNDED = "REFUNDED"


class PaymentPurpose(Enum):
    """
    PaymentPurpose explains why the payment exists.

    Purpose:
    - BOOKING_FEE is for facility bookings.
    - ACCESS_UPGRADE is for changing STANDARD to PREMIUM.

    Logical reason:
    Separating payment purposes makes reports and receipts clearer.
    """
    BOOKING_FEE = "BOOKING_FEE"
    ACCESS_UPGRADE = "ACCESS_UPGRADE"


class User:
    """
    User is the parent class for Student, Staff, and Administrator.

    Purpose:
    - Stores shared user information such as ID, name, email, password, role, access type, and status.
    - Provides shared actions such as login, profile update, support contact, and access requests.

    Logical reason:
    Student, Staff, and Administrator share common data. Using inheritance avoids repeating the same attributes and methods in every child class.
    """

    def __init__(self, user_id, university_id, name, email, password, role):
        # Validation protects the system from creating incomplete or invalid user objects.
        if not user_id:
            raise ValueError("User ID cannot be empty.")
        if not university_id:
            raise ValueError("University/work ID cannot be empty.")
        if not name:
            raise ValueError("Name cannot be empty.")
        if "@" not in email:
            raise ValueError("Invalid email format.")
        if len(password) < 4:
            raise ValueError("Password must be at least 4 characters.")

        # Private attributes match UML encapsulation.
        self.__user_id = user_id
        self.__university_id = university_id
        self.__name = name
        self.__email = email
        self.__password = password
        self.__role = role

        # Every new user starts with STANDARD access and ACTIVE status.
        # This matches the business rule that upgrades happen later.
        self.__access_type = AccessType.STANDARD
        self.__status = UserStatus.ACTIVE

        # Penalty credits start at 3.
        # Each violation decreases this by 1, and 0 causes freezing.
        self.__penalty_credits = 3

    def get_user_id(self):
        # Returns the user ID so the system can identify and search for a user.
        return self.__user_id

    def set_user_id(self, user_id):
        # Updates user ID only after checking it is not empty.
        if not user_id:
            raise ValueError("User ID cannot be empty.")
        self.__user_id = user_id

    def get_university_id(self):
        # Returns university or work ID for profile and ID generation tracking.
        return self.__university_id

    def set_university_id(self, university_id):
        # Updates university/work ID and prevents empty IDs.
        if not university_id:
            raise ValueError("University/work ID cannot be empty.")
        self.__university_id = university_id

    def get_name(self):
        # Returns the user's name for profile display, receipts, and booking summaries.
        return self.__name

    def set_name(self, name):
        # Updates the name after validation.
        if not name:
            raise ValueError("Name cannot be empty.")
        self.__name = name

    def get_email(self):
        # Returns email for support messages and user profile display.
        return self.__email

    def set_email(self, email):
        # Updates email and checks that it looks like an email address.
        if "@" not in email:
            raise ValueError("Invalid email format.")
        self.__email = email

    def get_password(self):
        # Returns password for login checking.
        return self.__password

    def set_password(self, password):
        # Updates password while keeping a minimum length rule.
        if len(password) < 4:
            raise ValueError("Password must be at least 4 characters.")
        self.__password = password

    def get_role(self):
        # Returns the role because access rules depend on Student, Staff, or Admin.
        return self.__role

    def set_role(self, role):
        # Updates the role only if it is a valid UserRole enum.
        if not isinstance(role, UserRole):
            raise TypeError("role must be UserRole.")
        self.__role = role

    def get_access_type(self):
        # Returns STANDARD or PREMIUM because booking permissions depend on it.
        return self.__access_type

    def set_access_type(self, access_type):
        # Updates access type after checking it is a valid AccessType enum.
        if not isinstance(access_type, AccessType):
            raise TypeError("accessType must be AccessType.")
        self.__access_type = access_type

    def get_status(self):
        # Returns ACTIVE or FROZEN. Frozen users cannot book.
        return self.__status

    def set_status(self, status):
        # Updates account status and prevents invalid status values.
        if not isinstance(status, UserStatus):
            raise TypeError("status must be UserStatus.")
        self.__status = status

    def get_penalty_credits(self):
        # Returns remaining penalty credits for admin monitoring and user display.
        return self.__penalty_credits

    def set_penalty_credits(self, penalty_credits):
        # Updates penalty credits but prevents negative values.
        if penalty_credits < 0:
            raise ValueError("Penalty credits cannot be negative.")
        self.__penalty_credits = penalty_credits

    def decrease_penalty_credit(self):
        # Decreases penalty credit after cancellation or no-show.
        # This supports the rule that repeated violations can freeze the user.
        if self.__penalty_credits > 0:
            self.__penalty_credits -= 1

    def reset_penalty_credits(self):
        # Restores penalty credits and makes the user active again.
        # This is used when admin unfreezes the account.
        self.__penalty_credits = 3
        self.__status = UserStatus.ACTIVE

    def login(self, user_id, password):
        # Checks if the entered login information matches this user.
        # This function exists so BookingSystem can validate users without directly accessing private password data.
        return self.__user_id == user_id and self.__password == password

    def update_profile(self, name, email):
        # Updates user profile using setters so validation still happens.
        self.set_name(name)
        self.set_email(email)

    def request_access_upgrade(self):
        # Represents a user asking to upgrade.
        # The actual payment and access update happens inside BookingSystem.
        return "Upgrade request submitted."

    def request_access_downgrade(self):
        # Allows user to downgrade back to STANDARD.
        # Downgrade does not require payment in the business rules.
        self.__access_type = AccessType.STANDARD

    def contact_support(self, subject, message):
        # Lets user send a support issue.
        # The subject and message are required so support receives meaningful details.
        if not subject or not message:
            raise ValueError("Subject and message are required.")
        return f"Support request submitted: {subject} - {message}"

# RELATIONSHIP EVIDENCE: Student inherits from User.
# This shows inheritance because Student reuses User attributes and methods.
class Student(User):
    """
    Student is a child class of User.

    Purpose:
    - Represents student users.
    - Stores student-specific bookings.
    - Supports the business rules for student facility access.

    Logical reason:
    Students have different permissions from Staff and Admin, so they need their own child class while still sharing User data.
    """

    def __init__(self, user_id, university_id, name, email, password):
        # super() calls the User constructor and sets role to STUDENT.
        super().__init__(user_id, university_id, name, email, password, UserRole.STUDENT)
        self.__student_bookings = {}

    def get_student_bookings(self):
        # Returns student booking records for profile display or testing.
        return self.__student_bookings

    def set_student_bookings(self, student_bookings):
        # Updates student bookings and checks the value is a dictionary.
        if not isinstance(student_bookings, dict):
            raise TypeError("studentBookings must be dictionary.")
        self.__student_bookings = student_bookings

    def view_student_bookings(self):
        # Shows all bookings connected to this student.
        return self.__student_bookings

    def book_study_room(self):
        # This represents the student's ability to book StudyRoom.
        # Actual booking validation is handled in BookingSystem and AccessPolicy.
        return "Student can book StudyRoom."

    def book_sports_court(self):
        # This represents the rule that only PREMIUM students can book SportsCourt.
        # AccessPolicy enforces the real permission check.
        return "Premium Student can book SportsCourt."

    def book_event_hall(self):
        # This represents the rule that only PREMIUM students can book EventHall.
        # BookingSystem checks this before creating a booking.
        return "Premium Student can book EventHall."

# RELATIONSHIP EVIDENCE: Staff inherits from User.
# This shows inheritance because Staff is a specialized type of User.
class Staff(User):
    """
    Staff is a child class of User.

    Purpose:
    - Represents staff users.
    - Stores staff-specific bookings.
    - Supports staff booking permissions.

    Logical reason:
    Staff rules are different from student rules. Staff can book EventHall, and PREMIUM staff can book SportsCourt, but staff can never book StudyRoom.
    """

    def __init__(self, user_id, university_id, name, email, password):
        # super() calls the User constructor and sets role to STAFF.
        super().__init__(user_id, university_id, name, email, password, UserRole.STAFF)
        self.__staff_bookings = {}

    def get_staff_bookings(self):
        # Returns staff booking records.
        return self.__staff_bookings

    def set_staff_bookings(self, staff_bookings):
        # Updates staff bookings and ensures dictionary structure.
        if not isinstance(staff_bookings, dict):
            raise TypeError("staffBookings must be dictionary.")
        self.__staff_bookings = staff_bookings

    def view_staff_bookings(self):
        # Displays staff bookings for profile or report use.
        return self.__staff_bookings

    def book_event_hall(self):
        # Represents staff permission for EventHall.
        # Standard and Premium staff can both book EventHall.
        return "Staff can book EventHall."

    def book_sports_court(self):
        # Represents the rule that only PREMIUM staff can book SportsCourt.
        return "Premium Staff can book SportsCourt."

# RELATIONSHIP EVIDENCE: Administrator inherits from User.
# This shows inheritance because Admin is also a User but has extra management functions.
class Administrator(User):
    """
    Administrator is a child class of User.

    Purpose:
    - Manages users.
    - Freezes or unfreezes accounts.
    - Upgrades or downgrades access.
    - Makes announcements.
    - Opens or closes facility slots.

    Logical reason:
    Admin has management permissions that normal users should not have.
    """

    def __init__(self, user_id, university_id, name, email, password):
        # super() calls the User constructor and sets role to ADMIN.
        super().__init__(user_id, university_id, name, email, password, UserRole.ADMIN)
        self.__admin_bookings_view = {}

    def get_admin_bookings_view(self):
        # Returns admin view of bookings.
        return self.__admin_bookings_view

    def set_admin_bookings_view(self, admin_bookings_view):
        # Updates admin booking view and ensures dictionary structure.
        if not isinstance(admin_bookings_view, dict):
            raise TypeError("adminBookingsView must be dictionary.")
        self.__admin_bookings_view = admin_bookings_view

    def freeze_user(self, user):
        # Freezes a user after violations.
        # Frozen users cannot book facilities.
        user.set_status(UserStatus.FROZEN)

    def unfreeze_user(self, user):
        # Unfreezes a user by resetting penalty credits and status.
        user.reset_penalty_credits()

    def upgrade_user_access(self, user):
        # Admin can help users upgrade when needed.
        user.set_access_type(AccessType.PREMIUM)

    def downgrade_user_access(self, user):
        # Admin can downgrade users back to STANDARD.
        user.set_access_type(AccessType.STANDARD)

    def make_announcement(self, announcements, message):
        # Adds announcement with timestamp.
        # Used for maintenance messages and system updates.
        if not message:
            raise ValueError("Announcement cannot be empty.")
        stamp = datetime.now().strftime("%Y-%m-%d %H:%M")
        announcements.append(f"{stamp}: {message}")

    def close_facility_slot(self, facility, selected_date, slot):
        # Admin closes a facility slot so users cannot book it.
        # Used for maintenance or unavailable times.
        facility.close_slot(selected_date, slot)

    def open_facility_slot(self, facility, selected_date, slot):
        # Admin reopens a previously closed slot.
        facility.open_slot(selected_date, slot)


class Facility:
    """
    Facility is the parent class for StudyRoom, SportsCourt, and EventHall.

    Purpose:
    - Stores shared facility information.
    - Manages available, booked, and closed time slots.
    - Calculates basic facility cost.

    Logical reason:
    All facility types share common data like location, capacity, slots, and price, so a parent class avoids repeated code.
    """

    def __init__(self, facility_id, facility_name, facility_type, location, capacity, price_per_hour, rules):
        self.__facility_id = facility_id
        self.__facility_name = facility_name
        self.__facility_type = facility_type
        self.__location = location
        self.__capacity = capacity
        self.__is_available = True
        self.__price_per_hour = price_per_hour
        self.__rules = rules

        # Fixed slots are used so the GUI can show clear booking options.
        self.__time_slots = [
            "09:00-10:00",
            "10:00-11:00",
            "11:00-12:00",
            "12:00-13:00",
            "14:00-15:00",
            "15:00-16:00"
        ]

        # booked_slots_by_date stores confirmed bookings per date.
        # closed_slots_by_date stores admin closed slots per date.
        self.__booked_slots_by_date = {}
        self.__closed_slots_by_date = {}

    def get_facility_id(self):
        # Returns facility ID so the catalog can find this facility.
        return self.__facility_id

    def set_facility_id(self, facility_id):
        # Updates facility ID.
        self.__facility_id = facility_id

    def get_facility_name(self):
        # Returns facility name for GUI and booking summaries.
        return self.__facility_name

    def set_facility_name(self, facility_name):
        # Updates displayed facility name.
        self.__facility_name = facility_name

    def get_facility_type(self):
        # Returns facility type because access rules depend on it.
        return self.__facility_type

    def set_facility_type(self, facility_type):
        # Updates facility type.
        self.__facility_type = facility_type

    def get_location(self):
        # Returns facility location for user display.
        return self.__location

    def set_location(self, location):
        # Updates facility location.
        self.__location = location

    def get_capacity(self):
        # Returns maximum number of people allowed.
        return self.__capacity

    def set_capacity(self, capacity):
        # Capacity must be positive because a facility cannot hold 0 or negative people.
        if capacity <= 0:
            raise ValueError("Capacity must be positive.")
        self.__capacity = capacity

    def get_is_available(self):
        # Returns general facility availability.
        return self.__is_available

    def set_is_available(self, is_available):
        # Opens or closes the whole facility.
        self.__is_available = is_available

    def get_price_per_hour(self):
        # Returns the basic hourly price.
        return self.__price_per_hour

    def set_price_per_hour(self, price_per_hour):
        # Price cannot be negative because cost must be valid.
        if price_per_hour < 0:
            raise ValueError("Price cannot be negative.")
        self.__price_per_hour = price_per_hour

    def get_rules(self):
        # Returns facility rules to show users before booking.
        return self.__rules

    def set_rules(self, rules):
        # Updates facility rules.
        self.__rules = rules

    def get_time_slots(self):
        # Returns all possible time slots.
        return self.__time_slots

    def set_time_slots(self, time_slots):
        # Updates available slot list if admin changes schedule.
        self.__time_slots = time_slots

    def get_booked_slots_by_date(self):
        # Returns booked slots grouped by date.
        return self.__booked_slots_by_date

    def set_booked_slots_by_date(self, booked_slots_by_date):
        # Used when repairing loaded pickle data.
        self.__booked_slots_by_date = booked_slots_by_date

    def get_closed_slots_by_date(self):
        # Returns admin-closed slots grouped by date.
        return self.__closed_slots_by_date

    def set_closed_slots_by_date(self, closed_slots_by_date):
        # Used when repairing old pickle data.
        self.__closed_slots_by_date = closed_slots_by_date

    def get_closed_slots(self, booking_date):
        # Makes sure every date has a closed-slot list.
        # This prevents key errors when checking closed slots.
        if booking_date not in self.__closed_slots_by_date:
            self.__closed_slots_by_date[booking_date] = []
        return self.__closed_slots_by_date[booking_date]

    def close_slot(self, booking_date, slot):
        # Closes one specific slot for maintenance or admin reasons.
        # Users cannot book slots in this closed list.
        if slot not in self.__time_slots:
            raise ValueError("Invalid time slot.")
        if booking_date not in self.__closed_slots_by_date:
            self.__closed_slots_by_date[booking_date] = []
        if slot not in self.__closed_slots_by_date[booking_date]:
            self.__closed_slots_by_date[booking_date].append(slot)

    def open_slot(self, booking_date, slot):
        # Reopens a closed slot so users can book it again.
        if booking_date in self.__closed_slots_by_date:
            if slot in self.__closed_slots_by_date[booking_date]:
                self.__closed_slots_by_date[booking_date].remove(slot)

    def get_available_slots(self, booking_date):
        # Returns slots that are not booked and not closed.
        # This is needed so users only see valid booking choices.
        if booking_date not in self.__booked_slots_by_date:
            self.__booked_slots_by_date[booking_date] = {}
        if booking_date not in self.__closed_slots_by_date:
            self.__closed_slots_by_date[booking_date] = []

        booked = self.__booked_slots_by_date[booking_date]
        closed = self.__closed_slots_by_date[booking_date]

        return [
            slot for slot in self.__time_slots
            if slot not in booked and slot not in closed
        ]

    def book_slot(self, booking_date, slot, booking_id):
        # Marks a slot as booked for a specific date.
        # This prevents another user from taking the same slot.
        if slot not in self.__time_slots:
            raise ValueError("Invalid time slot.")
        if booking_date not in self.__booked_slots_by_date:
            self.__booked_slots_by_date[booking_date] = {}
        if slot in self.__booked_slots_by_date[booking_date]:
            raise ValueError("This slot is already booked.")
        if slot in self.get_closed_slots(booking_date):
            raise ValueError("This slot is closed by admin.")
        self.__booked_slots_by_date[booking_date][slot] = booking_id

    def release_slot(self, booking_date, slot):
        # Frees a slot after cancellation or no-show.
        # This allows another user to book it later.
        if booking_date in self.__booked_slots_by_date:
            if slot in self.__booked_slots_by_date[booking_date]:
                del self.__booked_slots_by_date[booking_date][slot]

    def calculate_cost(self, usage_type):
        # Returns the normal price per hour.
        # Child classes can override this for special pricing.
        return self.__price_per_hour

    def display_details(self):
        # Returns a simple facility summary for GUI lists.
        return f"{self.__facility_name} | {self.__location} | Capacity: {self.__capacity}"

# RELATIONSHIP EVIDENCE: StudyRoom inherits from Facility.
# This shows inheritance because StudyRoom uses common facility details from Facility.
class StudyRoom(Facility):
    """
    StudyRoom represents a student-only facility.

    Purpose:
    - Allows students to book study spaces.
    - Has no booking fee.

    Logical reason:
    Study rooms are restricted to students only, so this child class separates it from SportsCourt and EventHall.
    """

    def __init__(self, facility_id, room_code, location, capacity):
        super().__init__(
            facility_id,
            f"Study Room {room_code}",
            "STUDY_ROOM",
            location,
            capacity,
            0,
            "Rules: Students only. Keep the room clean. Arrive on time."
        )
        self.__room_code = room_code

    def get_room_code(self):
        # Returns room code such as A or B.
        return self.__room_code

    def set_room_code(self, room_code):
        # Updates room code.
        self.__room_code = room_code

# RELATIONSHIP EVIDENCE: SportsCourt inherits from Facility.
# This shows inheritance because SportsCourt is a specialized type of Facility.
class SportsCourt(Facility):
    """
    SportsCourt represents sports facilities.

    Purpose:
    - Allows PREMIUM students and PREMIUM staff to book courts.
    - Stores sport type and hourly price.

    Logical reason:
    SportsCourt has different access rules from StudyRoom and EventHall.
    """

    def __init__(self, facility_id, sport_type, location, capacity, price_per_hour):
        super().__init__(
            facility_id,
            f"{sport_type} Court",
            "SPORTS_COURT",
            location,
            capacity,
            price_per_hour,
            "Rules: Sports shoes required. Respect the booked time."
        )
        self.__sport_type = sport_type

    def get_sport_type(self):
        # Returns sport type such as Basketball or Tennis.
        return self.__sport_type

    def set_sport_type(self, sport_type):
        # Updates sport type.
        self.__sport_type = sport_type

# RELATIONSHIP EVIDENCE: EventHall inherits from Facility.
# This shows inheritance because EventHall shares facility data but overrides cost logic.
class EventHall(Facility):
    """
    EventHall represents large event spaces.

    Purpose:
    - Allows staff to book halls.
    - Allows PREMIUM students to book halls.
    - Charges external use but keeps internal use free.

    Logical reason:
    EventHall has special payment logic, so it overrides calculate_cost().
    """

    def __init__(self, facility_id, location, capacity, external_rate):
        super().__init__(
            facility_id,
            "Event Hall",
            "EVENT_HALL",
            location,
            capacity,
            external_rate,
            "Rules: Internal use is free. External use requires payment."
        )
        self.__internal_use_free = True
        self.__external_rate = external_rate

    def get_internal_use_free(self):
        # Returns whether internal use is free.
        return self.__internal_use_free

    def set_internal_use_free(self, internal_use_free):
        # Updates internal use free rule.
        self.__internal_use_free = internal_use_free

    def get_external_rate(self):
        # Returns price charged for external event hall use.
        return self.__external_rate

    def set_external_rate(self, external_rate):
        # Updates external rate and parent price.
        self.__external_rate = external_rate
        self.set_price_per_hour(external_rate)

    def calculate_cost(self, usage_type):
        # Internal use is free based on business rules.
        if usage_type == UsageType.INTERNAL.value:
            return 0

        # External use requires payment.
        return self.__external_rate


class TimeSlot:
    """
    TimeSlot represents a simple time period.

    Purpose:
    - Stores start and end time.
    - Tracks if the slot is booked.

    Logical reason:
    This supports the UML TimeSlot class, but the current system mainly uses string slots like "10:00-11:00" inside Facility.
    """

    def __init__(self, slot_id, start_time, end_time):
        self.__slot_id = slot_id
        self.__start_time = start_time
        self.__end_time = end_time
        self.__is_booked = False

    def get_slot_id(self):
        # Returns slot ID.
        return self.__slot_id

    def set_slot_id(self, slot_id):
        # Updates slot ID.
        self.__slot_id = slot_id

    def get_start_time(self):
        # Returns slot start time.
        return self.__start_time

    def set_start_time(self, start_time):
        # Updates slot start time.
        self.__start_time = start_time

    def get_end_time(self):
        # Returns slot end time.
        return self.__end_time

    def set_end_time(self, end_time):
        # Updates slot end time.
        self.__end_time = end_time

    def get_is_booked(self):
        # Returns whether this TimeSlot object is booked.
        return self.__is_booked

    def set_is_booked(self, is_booked):
        # Updates booking state for this TimeSlot.
        self.__is_booked = is_booked


class Booking:
    """
    Booking connects User, Facility, and selected slot.

    Purpose:
    - Stores reservation details.
    - Tracks booking cost and status.
    - Generates booking summaries.

    Logical reason:
    A booking is the main relationship object that proves a user reserved a facility at a specific time.
    """

    def __init__(self, booking_id, user, facility, slot, booking_date, usage_type, total_cost):
        self.__booking_id = booking_id
        # RELATIONSHIP EVIDENCE: Booking is associated with User.
        # A Booking stores one User object, which proves each booking belongs to one user.
        self.__user = user
        # RELATIONSHIP EVIDENCE: Booking is associated with Facility.
        # A Booking stores one Facility object, which proves each booking reserves one facility.
        self.__facility = facility
        # RELATIONSHIP EVIDENCE: Booking is associated with a time slot.
        # The booking stores the selected slot string, so the reservation is tied to one time period.
        self.__slot = slot
        self.__booking_date = booking_date
        self.__usage_type = usage_type
        self.__total_cost = total_cost
        self.__status = BookingStatus.CONFIRMED

    def get_booking_id(self):
        # Returns booking ID for searching, modifying, or cancelling.
        return self.__booking_id

    def get_user(self):
        # Returns the user connected to this booking.
        return self.__user

    def get_facility(self):
        # Returns the facility connected to this booking.
        return self.__facility

    def get_slot(self):
        # Returns the booked time slot.
        return self.__slot

    def set_slot(self, slot):
        # Updates the booking slot when a booking is modified.
        self.__slot = slot

    def get_booking_date(self):
        # Returns booking date for reports and conflict checks.
        return self.__booking_date

    def set_booking_date(self, booking_date):
        # Updates booking date when booking is modified.
        self.__booking_date = booking_date

    def get_usage_type(self):
        # Returns internal or external usage type for cost rules.
        return self.__usage_type

    def get_total_cost(self):
        # Returns total cost for payment and receipt.
        return self.__total_cost

    def get_status(self):
        # Returns current booking status.
        return self.__status

    def set_status(self, status):
        # Updates booking status.
        self.__status = status

    def cancel_booking(self):
        # Marks booking as cancelled.
        # BookingSystem also releases the facility slot and applies penalty.
        self.__status = BookingStatus.CANCELLED

    def mark_no_show(self):
        # Marks booking as no-show.
        # BookingSystem uses this to apply penalty credit loss.
        self.__status = BookingStatus.NO_SHOW

    def generate_booking_summary(self):
        # Creates readable booking details for receipt, testing, and GUI display.
        return (
            f"Booking ID: {self.__booking_id}\n"
            f"User: {self.__user.get_name()}\n"
            f"Facility: {self.__facility.get_facility_name()}\n"
            f"Facility Type: {self.__facility.get_facility_type()}\n"
            f"Location: {self.__facility.get_location()}\n"
            f"Date: {self.__booking_date}\n"
            f"Time Slot: {self.__slot}\n"
            f"Usage Type: {self.__usage_type}\n"
            f"Cost: {self.__total_cost} AED\n"
            f"Status: {self.__status.value}\n"
            f"{self.__facility.get_rules()}"
        )


class Payment:
    """
    Payment represents payment for bookings or upgrades.

    Purpose:
    - Stores payment ID, amount, date, status, and purpose.
    - Processes payments.

    Logical reason:
    Some bookings are free and some require payment, so the system needs a payment object to track the financial result.
    """

    def __init__(self, payment_id, amount, purpose):
        self.__payment_id = payment_id
        self.__amount = amount
        self.__payment_date = str(date.today())
        self.__status = PaymentStatus.PENDING if amount > 0 else PaymentStatus.NOT_REQUIRED
        self.__purpose = purpose

    def get_payment_id(self):
        # Returns payment ID for receipt and payment records.
        return self.__payment_id

    def get_amount(self):
        # Returns payment amount.
        return self.__amount

    def get_payment_date(self):
        # Returns payment date.
        return self.__payment_date

    def get_status(self):
        # Returns payment status.
        return self.__status

    def set_status(self, status):
        # Updates payment status.
        self.__status = status

    def get_purpose(self):
        # Returns whether payment is for booking fee or upgrade.
        return self.__purpose

    def process_payment(self):
        # Processes payment.
        # If amount is 0, payment is not required.
        # If amount is more than 0, payment becomes PAID.
        if self.__amount == 0:
            self.__status = PaymentStatus.NOT_REQUIRED
        else:
            self.__status = PaymentStatus.PAID
        return True


class Receipt:
    """
    Receipt is proof of booking/payment.

    Purpose:
    - Combines booking and payment details.
    - Generates receipt text.

    Logical reason:
    After booking and payment, the user needs evidence that the transaction was completed.
    """

    def __init__(self, receipt_id, booking, payment):
        self.__receipt_id = receipt_id
        # RELATIONSHIP EVIDENCE: Receipt is associated with Booking.
        # Receipt stores the booking object so it can display booking details in the receipt.
        self.__booking = booking
        # RELATIONSHIP EVIDENCE: Receipt is connected to Payment.
        # Receipt stores the payment object because receipt information depends on payment result.
        self.__payment = payment
        self.__issue_date = str(date.today())

    def get_receipt_id(self):
        # Returns receipt ID for storage and search.
        return self.__receipt_id

    def generate_receipt(self):
        # Generates formatted receipt details for the user.
        return (
            "BOOKING RECEIPT\n"
            "----------------------\n"
            f"Receipt ID: {self.__receipt_id}\n"
            f"Issue Date: {self.__issue_date}\n"
            f"Payment Status: {self.__payment.get_status().value}\n"
            f"Payment Amount: {self.__payment.get_amount()} AED\n\n"
            f"{self.__booking.generate_booking_summary()}"
        )


class Notification:
    """
    Notification stores messages sent by the system.

    Purpose:
    - Stores booking confirmations.
    - Stores freeze warnings.
    - Stores maintenance or admin messages.

    Logical reason:
    Users need to be informed when important system actions happen.
    """

    def __init__(self, notification_id, message):
        self.__notification_id = notification_id
        self.__message = message
        self.__send_date = str(date.today())
        self.__is_read = False

    def get_notification_id(self):
        # Returns notification ID for storage.
        return self.__notification_id

    def get_message(self):
        # Returns message text for GUI display.
        return self.__message

    def get_send_date(self):
        # Returns notification date.
        return self.__send_date

    def get_is_read(self):
        # Returns whether user has read the notification.
        return self.__is_read

    def mark_as_read(self):
        # Marks notification as read after user views it.
        self.__is_read = True


class AccessPolicy:
    """
    AccessPolicy controls booking permission rules.

    Purpose:
    - Checks role.
    - Checks access type.
    - Checks facility type.

    Logical reason:
    Keeping permission rules in one class makes the system easier to update if business rules change.
    """
    # RELATIONSHIP EVIDENCE: AccessPolicy depends on User and Facility.
    # This method receives user and facility as parameters to check if booking is allowed.
    def can_book(self, user, facility):
        # Get user and facility details needed for permission checking.
        # RELATIONSHIP EVIDENCE: AccessPolicy depends on User.
        # It reads user role and access type to apply permission rules.
        role = user.get_role()
        access = user.get_access_type()
        # RELATIONSHIP EVIDENCE: AccessPolicy depends on Facility.
        # It reads facility type to decide whether the user can book it.
        facility_type = facility.get_facility_type()

        # Admin manages the system but does not make normal facility bookings.
        if role == UserRole.ADMIN:
            return False

        # StudyRoom is student-only.
        if facility_type == "STUDY_ROOM":
            return role == UserRole.STUDENT

        # SportsCourt is only for PREMIUM students and PREMIUM staff.
        if facility_type == "SPORTS_COURT":
            return access == AccessType.PREMIUM and role in [UserRole.STUDENT, UserRole.STAFF]

        # EventHall can be booked by all staff and PREMIUM students.
        if facility_type == "EVENT_HALL":
            if role == UserRole.STAFF:
                return True
            return role == UserRole.STUDENT and access == AccessType.PREMIUM

        # Unknown facility types are rejected.
        return False


class FacilityCatalog:
    """
    FacilityCatalog stores all facility objects.

    Purpose:
    - Adds facilities.
    - Removes facilities.
    - Finds facilities by ID.

    Logical reason:
    BookingSystem needs one organized place to manage all facilities.
    """

    def __init__(self):
        # RELATIONSHIP EVIDENCE: FacilityCatalog aggregates Facilities.
        # The catalog stores many Facility objects using facility ID as the key.
        self.__facilities = {}

    def get_facilities(self):
        # Returns the full facility dictionary.
        return self.__facilities

    def set_facilities(self, facilities):
        # Replaces facility dictionary.
        # Used when loading data from pickle.
        self.__facilities = facilities

    def add_facility(self, facility):
        # Adds facility using its facility ID as the key.
        # RELATIONSHIP EVIDENCE: FacilityCatalog aggregates Facilities.
        # Adding a facility object into the catalog shows the catalog manages many facilities.
        self.__facilities[facility.get_facility_id()] = facility

    def remove_facility(self, facility_id):
        # Removes facility if it exists.
        if facility_id in self.__facilities:
            del self.__facilities[facility_id]

    def get_facility(self, facility_id):
        # Returns one facility by ID.
        return self.__facilities.get(facility_id)

    def display_all_facilities(self):
        # Returns all facilities for GUI display.
        return self.__facilities


class BookingSystem:
    """
    BookingSystem is the main controller class.

    Purpose:
    - Stores all users, bookings, payments, receipts, notifications, and announcements.
    - Handles login, registration, booking, payment, penalties, upgrades, maintenance, and persistence.

    Logical reason:
    The system needs one central controller to connect all other classes together.
    """

    def __init__(self):
        self.__system_name = "Smart Campus Facility Booking System"
        self.__support_email = "support@campus.ae"
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Users.
        # Users are stored in a dictionary so the system can manage many user objects.
        self.__users = {}
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Bookings.
        # Bookings are stored in a dictionary as system records.
        self.__bookings = {}
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Payments.
        # Payments are stored separately so the system can keep financial records.
        self.__payments = {}
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Receipts.
        # Receipts are stored separately as confirmation records.
        self.__receipts = {}
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Notifications.
        # Notifications are stored so the system can keep user messages.
        self.__notifications = {}
        self.__announcements = []
        # RELATIONSHIP EVIDENCE: BookingSystem composes FacilityCatalog.
        # BookingSystem creates and owns the FacilityCatalog object.
        self.__facility_catalog = FacilityCatalog()
        # RELATIONSHIP EVIDENCE: BookingSystem composes AccessPolicy.
        # BookingSystem creates AccessPolicy to control booking permissions.
        self.__access_policy = AccessPolicy()
        self.__data_file_name = "smart_campus_data.dat"
        self.__session_file_name = "smart_campus_session.dat"
        self.__current_user_id = ""
        self.__remember_login = False
        self.__next_booking_number = 1

        # Load saved data so the system remembers users and bookings after closing.
        self.load_data_from_file(self.__data_file_name)

        # Repair old saved data if new attributes were added later.
        self.repair_loaded_data()

        # Create default facilities only if no facilities were loaded.
        if not self.__facility_catalog.get_facilities():
            self.create_default_facilities()

        # Load saved login session if remember login was used.
        self.load_session()

    def get_system_name(self):
        # Returns system name for GUI title or reports.
        return self.__system_name

    def get_support_email(self):
        # Returns support email for contact/support section.
        return self.__support_email

    def get_users(self):
        # Returns registered users.
        return self.__users

    def get_bookings(self):
        # Returns all bookings.
        return self.__bookings

    def get_payments(self):
        # Returns all payment records.
        return self.__payments

    def get_receipts(self):
        # Returns all receipts.
        return self.__receipts

    def get_notifications(self):
        # Returns all notifications.
        return self.__notifications

    def get_announcements(self):
        # Returns system announcements.
        return self.__announcements

    def get_facility_catalog(self):
        # Returns facility catalog object.
        return self.__facility_catalog

    def get_current_user_id(self):
        # Returns the logged-in user ID.
        return self.__current_user_id

    def get_current_user(self):
        # Returns the logged-in user object if one exists.
        if self.__current_user_id in self.__users:
            return self.__users[self.__current_user_id]
        return None

    def create_default_facilities(self):
        # Creates built-in facilities so the system has ready booking options.
        self.__facility_catalog.add_facility(StudyRoom("SR-A", "A", "Library - Floor 1", 6))
        self.__facility_catalog.add_facility(StudyRoom("SR-B", "B", "Library - Floor 2", 4))
        self.__facility_catalog.add_facility(SportsCourt("SC-B", "Basketball", "Sports Complex", 10, 20))
        self.__facility_catalog.add_facility(SportsCourt("SC-T", "Tennis", "Sports Complex", 4, 15))
        self.__facility_catalog.add_facility(EventHall("EH-1", "Main Building", 50, 100))
        self.__facility_catalog.add_facility(EventHall("EH-2", "Conference Building", 100, 150))

    def repair_loaded_data(self):
        # Adds missing attributes to old pickle data.
        # This prevents crashes when the code was updated after data was saved.
        for user in self.__users.values():
            if not hasattr(user, "_User__penalty_credits"):
                user.set_penalty_credits(3)

        for facility in self.__facility_catalog.get_facilities().values():
            if not hasattr(facility, "_Facility__booked_slots_by_date"):
                facility.set_booked_slots_by_date({})
            if not hasattr(facility, "_Facility__closed_slots_by_date"):
                facility.set_closed_slots_by_date({})

    def save_data_to_file(self, file_name):
        # Saves system data in a binary pickle file.
        # This is required for persistence.
        try:
            with open(file_name, "wb") as file:
                pickle.dump({
                    "users": self.__users,
                    "bookings": self.__bookings,
                    "payments": self.__payments,
                    "receipts": self.__receipts,
                    "notifications": self.__notifications,
                    "announcements": self.__announcements,
                    "facility_catalog": self.__facility_catalog,
                    "next_booking_number": self.__next_booking_number
                }, file)
            return True
        except (pickle.PickleError, OSError) as error:
            raise OSError(f"Saving failed: {error}")

    def load_data_from_file(self, file_name):
        # Loads system data from a binary pickle file.
        # If the file is missing or corrupted, the system resets safely.
        try:
            if os.path.exists(file_name):
                with open(file_name, "rb") as file:
                    data = pickle.load(file)

                self.__users = data.get("users", {})
                self.__bookings = data.get("bookings", {})
                self.__payments = data.get("payments", {})
                self.__receipts = data.get("receipts", {})
                self.__notifications = data.get("notifications", {})
                self.__announcements = data.get("announcements", [])
                self.__facility_catalog = data.get("facility_catalog", FacilityCatalog())
                self.__next_booking_number = data.get("next_booking_number", 1)

            return self
        except (EOFError, pickle.UnpicklingError, OSError, AttributeError):
            self.__users = {}
            self.__bookings = {}
            self.__payments = {}
            self.__receipts = {}
            self.__notifications = {}
            self.__announcements = []
            self.__facility_catalog = FacilityCatalog()
            self.__next_booking_number = 1
            return self

    def save_session(self, user_id):
        # Saves current login session using pickle.
        # This supports remember-login functionality.
        try:
            with open(self.__session_file_name, "wb") as file:
                pickle.dump({"current_user_id": user_id}, file)
            return True
        except OSError:
            return False

    def load_session(self):
        # Loads saved login session if it exists.
        try:
            if os.path.exists(self.__session_file_name):
                with open(self.__session_file_name, "rb") as file:
                    data = pickle.load(file)

                user_id = data.get("current_user_id", "")
                if user_id in self.__users:
                    self.__current_user_id = user_id
                    return self.__users[user_id]
            return None
        except (EOFError, pickle.UnpicklingError, OSError, AttributeError):
            return None

    def logout(self):
        # Clears current user and removes saved login session.
        self.__current_user_id = ""
        if os.path.exists(self.__session_file_name):
            os.remove(self.__session_file_name)

    def generate_user_id(self, role, university_id):
        # Creates user ID using role prefix.
        # This keeps user IDs consistent and easy to identify.
        if role == UserRole.STUDENT:
            return "STU" + university_id
        if role == UserRole.STAFF:
            return "STF" + university_id
        if role == UserRole.ADMIN:
            return "ADM" + university_id
        raise ValueError("Invalid role.")

    def register_user(self, role, university_id, name, email, password):
        # Registers a user based on role.
        # The function creates the correct child object automatically.
        user_id = self.generate_user_id(role, university_id)

        if user_id in self.__users:
            raise ValueError("This account already exists.")
        # RELATIONSHIP EVIDENCE: BookingSystem creates different User child objects.
        # This shows polymorphism because the system creates Student, Staff, or Administrator based on role.
        if role == UserRole.STUDENT:
            user = Student(user_id, university_id, name, email, password)
        elif role == UserRole.STAFF:
            user = Staff(user_id, university_id, name, email, password)
        elif role == UserRole.ADMIN:
            user = Administrator(user_id, university_id, name, email, password)
        else:
            raise ValueError("Invalid role.")
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates User objects.
        # The created user object is stored in the users dictionary.
        self.__users[user_id] = user
        self.save_data_to_file(self.__data_file_name)
        return user

    def validate_login(self, user_id, password, remember_login=False):
        # Validates login and stores current user if successful.
        if user_id not in self.__users:
            raise ValueError(
                "User ID not found.\n\n"
                "Example:\n"
                "Student 20240001 becomes STU20240001\n"
                "Staff 8899 becomes STF8899\n"
                "Admin 1000 becomes ADM1000"
            )

        user = self.__users[user_id]

        if not user.login(user_id, password):
            raise PermissionError("Incorrect password.")

        self.__current_user_id = user_id
        self.__remember_login = remember_login

        if remember_login:
            self.save_session(user_id)

        return user

    def validate_weekday_booking(self, selected_date):
        # Stops weekend bookings.
        # The system only allows Monday to Friday bookings.
        booking_day = datetime.strptime(selected_date, "%Y-%m-%d").date()
        if booking_day.weekday() >= 5:
            raise ValueError("Bookings are only allowed from Monday to Friday.")
        return True

    def get_allowed_facilities(self, user):
        # Returns only facilities that the user is allowed to book.
        # This helps the GUI show correct options.
        allowed = {}
        for facility_id, facility in self.__facility_catalog.get_facilities().items():
            if facility.get_is_available() and self.__access_policy.can_book(user, facility):
                allowed[facility_id] = facility
        return allowed

    def count_user_bookings_on_date(self, user, selected_date):
        # Counts confirmed bookings for one user on one date.
        # Used to enforce STANDARD daily booking credit.
        count = 0
        for booking in self.__bookings.values():
            if (
                booking.get_user().get_user_id() == user.get_user_id()
                and booking.get_booking_date() == selected_date
                and booking.get_status() == BookingStatus.CONFIRMED
            ):
                count += 1
        return count

    def count_user_facility_type_bookings_on_date(self, user, facility_type, selected_date):
        # Counts confirmed bookings for one user and one facility type on a date.
        # Used to enforce PREMIUM limit of 3 bookings per facility type per day.
        count = 0
        for booking in self.__bookings.values():
            if (
                booking.get_user().get_user_id() == user.get_user_id()
                and booking.get_facility().get_facility_type() == facility_type
                and booking.get_booking_date() == selected_date
                and booking.get_status() == BookingStatus.CONFIRMED
            ):
                count += 1
        return count

    def get_daily_credit_text(self, user, selected_date=None):
        # Creates readable credit text for GUI display.
        if selected_date is None:
            selected_date = str(date.today())

        if user.get_access_type() == AccessType.STANDARD:
            used = self.count_user_bookings_on_date(user, selected_date)
            return f"Booking Credit: {max(0, 1 - used)} / 1 | Penalty Credit: {user.get_penalty_credits()} / 3"

        allowed_types = set()
        for facility in self.get_allowed_facilities(user).values():
            allowed_types.add(facility.get_facility_type())

        credit_lines = []
        for facility_type in allowed_types:
            used = self.count_user_facility_type_bookings_on_date(user, facility_type, selected_date)
            credit_lines.append(f"{facility_type}: {max(0, 3 - used)} / 3")

        return " | ".join(credit_lines) + f" | Penalty Credit: {user.get_penalty_credits()} / 3"

    def check_booking_credit(self, user, facility, selected_date):
        # Enforces daily booking limits.
        # STANDARD gets 1 booking per day.
        # PREMIUM gets 3 bookings per facility type per day.
        if user.get_access_type() == AccessType.STANDARD:
            if self.count_user_bookings_on_date(user, selected_date) >= 1:
                raise PermissionError("STANDARD access allows only 1 booking per day.")

        if user.get_access_type() == AccessType.PREMIUM:
            used = self.count_user_facility_type_bookings_on_date(
                user,
                facility.get_facility_type(),
                selected_date
            )
            if used >= 3:
                raise PermissionError(
                    f"PREMIUM access allows only 3 bookings per {facility.get_facility_type()} per day."
                )

    def check_booking_conflict(self, user, slot, selected_date):
        # Checks if user already has a confirmed booking at the same date and time.
        # This prevents double-booking conflict.
        for booking in self.__bookings.values():
            if (
                booking.get_user().get_user_id() == user.get_user_id()
                and booking.get_booking_date() == selected_date
                and booking.get_slot() == slot
                and booking.get_status() == BookingStatus.CONFIRMED
            ):
                return True
        return False

    def auto_freeze_if_needed(self, user):
        # Automatically freezes users when penalty credits reach 0.
        # Also creates a notification explaining why the account was frozen.
        if user.get_penalty_credits() <= 0:
            user.set_status(UserStatus.FROZEN)
            notification_id = "N-FREEZE-" + user.get_user_id()
            self.__notifications[notification_id] = Notification(
                notification_id,
                "Your account has been automatically frozen because you reached 3 violations."
            )

    def record_cancellation_violation(self, user):
        # Records cancellation penalty.
        # Each violation decreases penalty credit by 1.
        user.decrease_penalty_credit()
        self.auto_freeze_if_needed(user)
        self.save_data_to_file(self.__data_file_name)

    def record_no_show_violation(self, user):
        # Records no-show penalty.
        # This supports the punishment rule in the assignment.
        user.decrease_penalty_credit()
        self.auto_freeze_if_needed(user)
        self.save_data_to_file(self.__data_file_name)

    def create_booking_after_payment(self, facility_id, slot, selected_date, usage_type):
        # Main booking creation function.
        # It checks login user, permission, date, slot availability, conflicts, credit limit, payment, receipt, and notification.
        user = self.get_current_user()
        facility = self.__facility_catalog.get_facility(facility_id)

        if user.get_status() == UserStatus.FROZEN:
            raise PermissionError("Your account is FROZEN. You cannot book.")

        if not facility.get_is_available():
            raise PermissionError("Facility is unavailable.")

        datetime.strptime(selected_date, "%Y-%m-%d")
        self.validate_weekday_booking(selected_date)

        if datetime.strptime(selected_date, "%Y-%m-%d").date() < date.today():
            raise ValueError("Booking date cannot be in the past.")
        # RELATIONSHIP EVIDENCE: BookingSystem uses AccessPolicy.
        # The system calls AccessPolicy to check if the user is allowed to book this facility.
        if not self.__access_policy.can_book(user, facility):
            raise PermissionError("You cannot book this facility.")

        if slot not in facility.get_available_slots(selected_date):
            raise ValueError("This slot is already booked or closed.")

        if self.check_booking_conflict(user, slot, selected_date):
            raise ValueError("You already have a booking at this time.")

        self.check_booking_credit(user, facility, selected_date)

        total_cost = facility.calculate_cost(usage_type)

        booking_id = "B" + str(self.__next_booking_number).zfill(4)
        self.__next_booking_number += 1
        # RELATIONSHIP EVIDENCE: BookingSystem creates a Booking linked to User and Facility.
        # The booking object receives user and facility objects, proving both associations.
        booking = Booking(booking_id, user, facility, slot, selected_date, usage_type, total_cost)
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Booking objects.
        # The booking is stored in the system booking dictionary.
        self.__bookings[booking_id] = booking
        # RELATIONSHIP EVIDENCE: Facility is associated with booked slots.
        # The facility records the selected slot using booking ID, proving the booking affects facility availability.
        facility.book_slot(selected_date, slot, booking_id)
        # RELATIONSHIP EVIDENCE: BookingSystem creates Payment for Booking.
        # The payment ID is based on the booking ID, showing payment is linked to the booking transac
        payment = Payment("P" + booking_id, total_cost, PaymentPurpose.BOOKING_FEE)
        payment.process_payment()
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Payment objects.
        # The payment object is stored as a system payment record.
        self.__payments[payment.get_payment_id()] = payment
        # RELATIONSHIP EVIDENCE: Receipt is connected to Booking and Payment.
        # The receipt is created using both booking and payment objects.
        receipt = Receipt("R" + booking_id, booking, payment)
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Receipt objects.
        # The receipt object is stored as a confirmation record.
        self.__receipts[receipt.get_receipt_id()] = receipt
        # RELATIONSHIP EVIDENCE: Booking can generate Notification.
        # A booking confirmation notification is created using the booking ID.
        notification = Notification("N" + booking_id, f"Booking {booking_id} confirmed.")
        # RELATIONSHIP EVIDENCE: BookingSystem aggregates Notification objects.
        # The notification is stored in the notification dictionary.
        self.__notifications[notification.get_notification_id()] = notification

        self.save_data_to_file(self.__data_file_name)
        return booking, receipt

    def cancel_booking(self, booking_id):
        # Cancels a booking, releases the slot, and applies a cancellation penalty.
        if booking_id not in self.__bookings:
            raise ValueError("Booking not found.")

        booking = self.__bookings[booking_id]

        if booking.get_user().get_user_id() != self.__current_user_id:
            raise PermissionError("You can only cancel your own booking.")

        booking.get_facility().release_slot(booking.get_booking_date(), booking.get_slot())
        booking.cancel_booking()
        self.record_cancellation_violation(booking.get_user())
        self.save_data_to_file(self.__data_file_name)

    def modify_booking(self, booking_id, new_slot, new_date):
        # Modifies a booking by changing the slot and date.
        # The old slot is released and the new slot is booked.
        if booking_id not in self.__bookings:
            raise ValueError("Booking not found.")

        booking = self.__bookings[booking_id]

        if booking.get_user().get_user_id() != self.__current_user_id:
            raise PermissionError("You can only modify your own booking.")

        self.validate_weekday_booking(new_date)

        facility = booking.get_facility()

        if new_slot not in facility.get_available_slots(new_date):
            raise ValueError("New slot is not available.")

        datetime.strptime(new_date, "%Y-%m-%d")

        facility.release_slot(booking.get_booking_date(), booking.get_slot())
        booking.set_slot(new_slot)
        booking.set_booking_date(new_date)
        facility.book_slot(new_date, new_slot, booking_id)

        self.save_data_to_file(self.__data_file_name)

    def mark_booking_no_show(self, booking_id):
        # Marks booking as no-show, releases the slot, and applies penalty.
        if booking_id not in self.__bookings:
            raise ValueError("Booking not found.")

        booking = self.__bookings[booking_id]
        user = booking.get_user()

        booking.get_facility().release_slot(booking.get_booking_date(), booking.get_slot())
        booking.mark_no_show()

        self.record_no_show_violation(user)
        self.save_data_to_file(self.__data_file_name)

    def process_upgrade_payment(self):
        # Processes STANDARD to PREMIUM upgrade.
        # User pays upgrade fee, then access changes to PREMIUM.
        user = self.get_current_user()

        if user.get_access_type() == AccessType.PREMIUM:
            raise ValueError("User already has PREMIUM access.")

        payment = Payment("P-UP-" + user.get_user_id(), 30, PaymentPurpose.ACCESS_UPGRADE)
        payment.process_payment()
        self.__payments[payment.get_payment_id()] = payment

        user.set_access_type(AccessType.PREMIUM)
        self.save_data_to_file(self.__data_file_name)

    def downgrade_current_user(self):
        # Downgrades the logged-in user to STANDARD.
        user = self.get_current_user()
        user.set_access_type(AccessType.STANDARD)
        self.save_data_to_file(self.__data_file_name)

    def freeze_user(self, user_id):
        # Freezes a selected user manually.
        # Admin can use this after serious violations.
        if user_id not in self.__users:
            raise ValueError("User not found.")
        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            admin.freeze_user(self.__users[user_id])
        else:
            self.__users[user_id].set_status(UserStatus.FROZEN)
        self.save_data_to_file(self.__data_file_name)

    def unfreeze_user(self, user_id):
        # Unfreezes selected user and resets penalty credits.
        if user_id not in self.__users:
            raise ValueError("User not found.")
        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            # RELATIONSHIP EVIDENCE: Admin manages User.
            # The admin object freezes another user object, showing an admin-user management association.
            admin.unfreeze_user(self.__users[user_id])
        else:
            self.__users[user_id].reset_penalty_credits()
        self.save_data_to_file(self.__data_file_name)

    def upgrade_user_by_admin(self, user_id):
        # Admin upgrades any user to PREMIUM.
        if user_id not in self.__users:
            raise ValueError("User not found.")
        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            # RELATIONSHIP EVIDENCE: Admin manages User access.
            # The admin object upgrades a selected user to Premium.
            admin.upgrade_user_access(self.__users[user_id])
        else:
            self.__users[user_id].set_access_type(AccessType.PREMIUM)
        self.save_data_to_file(self.__data_file_name)

    def downgrade_user_by_admin(self, user_id):
        # Admin downgrades any user to STANDARD.
        if user_id not in self.__users:
            raise ValueError("User not found.")
        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            admin.downgrade_user_access(self.__users[user_id])
        else:
            self.__users[user_id].set_access_type(AccessType.STANDARD)
        self.save_data_to_file(self.__data_file_name)

    def close_facility_slot(self, facility_id, selected_date, slot):
        # Closes one facility slot for maintenance.
        # Users cannot book this closed slot.
        # RELATIONSHIP EVIDENCE: BookingSystem uses FacilityCatalog.
        # The system asks the catalog to find the facility object by facility ID.
        facility = self.__facility_catalog.get_facility(facility_id)
        if facility is None:
            raise ValueError("Facility not found.")

        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            admin.close_facility_slot(facility, selected_date, slot)
        else:
            facility.close_slot(selected_date, slot)

        self.save_data_to_file(self.__data_file_name)

    def open_facility_slot(self, facility_id, selected_date, slot):
        # Opens a previously closed facility slot.
        facility = self.__facility_catalog.get_facility(facility_id)
        if facility is None:
            raise ValueError("Facility not found.")

        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            admin.open_facility_slot(facility, selected_date, slot)
        else:
            facility.open_slot(selected_date, slot)

        self.save_data_to_file(self.__data_file_name)

    def add_announcement(self, message):
        # Adds maintenance or system announcement.
        # This supports the admin announcement test case.
        admin = self.get_current_user()
        if isinstance(admin, Administrator):
            admin.make_announcement(self.__announcements, message)
        else:
            if not message:
                raise ValueError("Announcement cannot be empty.")
            stamp = datetime.now().strftime("%Y-%m-%d %H:%M")
            self.__announcements.append(f"{stamp}: {message}")

        self.save_data_to_file(self.__data_file_name)

    def get_bookings_by_date(self, selected_date):
        # Returns all bookings on a selected date.
        # Admin uses this for monitoring daily booking activity.
        return {
            booking_id: booking
            for booking_id, booking in self.__bookings.items()
            if booking.get_booking_date() == selected_date
        }


'''
Explanation Box:

Scenario:
This test checks the priority rule when two users try to reserve the same facility
at the same date and time. We created two staff users because both STANDARD Staff
and PREMIUM Staff are allowed to book Event Halls. Salama is a STANDARD Staff user,
while Maryam is upgraded to PREMIUM. Both users request Event Hall 1 on the same
date and same time slot.

What this test is testing:
- STANDARD Staff can book EventHall.
- PREMIUM Staff can also book EventHall.
- When both users request the same slot at the same time, the Premium request is
  processed first to reflect priority access.
- After the Premium booking is confirmed, the Standard user cannot book the same
  slot because it is no longer available.

Important note:
The current BookingSystem does not replace an already-confirmed booking. Therefore,
this test simulates simultaneous requests by processing the Premium request first.
This matches the business logic that Premium users should receive priority access.

Functions tested:
- register_user()
- validate_login()
- process_upgrade_payment()
- create_booking_after_payment()
- get_available_slots()
- get_access_type()
- get_facility_catalog()
'''

def reset_priority_test_files():
    # Reset pickle files so this priority test starts clean.
    for file_name in ["smart_campus_data.dat", "smart_campus_session.dat"]:
        if os.path.exists(file_name):
            os.remove(file_name)


def priority_test_premium_gets_slot():
    reset_priority_test_files()

    system = BookingSystem()

    print("\n================================================")
    print("SMART CAMPUS: PRIORITY TEST ACTIVITY LOG")
    print("Premium User Gets the Slot When Requests Clash")
    print("================================================")

    # Create two staff users because EventHall is available to both Standard and Premium Staff.
    standard_staff = system.register_user(
        UserRole.STAFF,
        "5001",
        "Salama Standard Staff",
        "salama@staff.com",
        "1234"
    )

    premium_staff = system.register_user(
        UserRole.STAFF,
        "5002",
        "Maryam Premium Staff",
        "maryam@staff.com",
        "1234"
    )

    print("New Standard Staff Created:", standard_staff.get_name())
    print("User ID:", standard_staff.get_user_id())
    print("Access:", standard_staff.get_access_type().value)
    print("------------------------------------------------")

    print("New Premium Staff Candidate Created:", premium_staff.get_name())
    print("User ID:", premium_staff.get_user_id())
    print("Initial Access:", premium_staff.get_access_type().value)
    print("------------------------------------------------")

    # Maryam logs in and upgrades to Premium.
    system.validate_login("STF5002", "1234")
    print("Activity: Maryam logs in and upgrades to PREMIUM.")
    system.process_upgrade_payment()
    print("Maryam Access After Upgrade:", system.get_current_user().get_access_type().value)
    print("------------------------------------------------")

    booking_date = "2026-05-20"
    selected_facility = "EH-1"
    selected_slot = "09:00-10:00"

    print("Priority Clash Scenario:")
    print("Both users want the same facility, same date, and same slot.")
    print("Facility: Event Hall 1")
    print("Date:", booking_date)
    print("Slot:", selected_slot)
    print("------------------------------------------------")

    print("Priority Rule Applied:")
    print("Since Maryam is PREMIUM, her booking request is processed first.")

    premium_booking, premium_receipt = system.create_booking_after_payment(
        selected_facility,
        selected_slot,
        booking_date,
        UsageType.INTERNAL.value
    )

    print("PREMIUM BOOKING CONFIRMED")
    print("Booking ID:", premium_booking.get_booking_id())
    print("User:", premium_booking.get_user().get_name())
    print("Access:", premium_booking.get_user().get_access_type().value)
    print("Facility:", premium_booking.get_facility().get_facility_name())
    print("Slot:", premium_booking.get_slot())
    print("Cost:", premium_booking.get_total_cost(), "AED")
    print("------------------------------------------------")

    facility = system.get_facility_catalog().get_facility(selected_facility)
    print("Available slots after Premium booking:")
    print(facility.get_available_slots(booking_date))
    print("------------------------------------------------")

    # Now Salama tries to book the same facility/date/slot.
    system.logout()
    system.validate_login("STF5001", "1234")

    print("Activity: Salama STANDARD Staff tries to book the same slot.")
    print("User:", system.get_current_user().get_name())
    print("Access:", system.get_current_user().get_access_type().value)

    try:
        system.create_booking_after_payment(
            selected_facility,
            selected_slot,
            booking_date,
            UsageType.INTERNAL.value
        )

    except ValueError as error:
        print("STANDARD BOOKING DENIED")
        print("System Message:", error)
        print("Reason: The slot was already taken by the Premium user.")

    print("------------------------------------------------")
    print("Priority Test Result:")
    print("Premium user received the slot first.")
    print("Standard user was blocked after the slot became unavailable.")
    print("SMART CAMPUS: PRIORITY TEST COMPLETED")
    print("================================================")


priority_test_premium_gets_slot()


SMART CAMPUS: PRIORITY TEST ACTIVITY LOG
Premium User Gets the Slot When Requests Clash
New Standard Staff Created: Salama Standard Staff
User ID: STF5001
Access: STANDARD
------------------------------------------------
New Premium Staff Candidate Created: Maryam Premium Staff
User ID: STF5002
Initial Access: STANDARD
------------------------------------------------
Activity: Maryam logs in and upgrades to PREMIUM.
Maryam Access After Upgrade: PREMIUM
------------------------------------------------
Priority Clash Scenario:
Both users want the same facility, same date, and same slot.
Facility: Event Hall 1
Date: 2026-05-20
Slot: 09:00-10:00
------------------------------------------------
Priority Rule Applied:
Since Maryam is PREMIUM, her booking request is processed first.
PREMIUM BOOKING CONFIRMED
Booking ID: B0001
User: Maryam Premium Staff
Access: PREMIUM
Facility: Event Hall
Slot: 09:00-10:00
Cost: 0 AED
------------------------------------------------
Available slots after Pre